# 16. Closed-Chain Stance Corridor Review

Squat-specific normalization review surface for the first promoted corrected-3D-hypothesis candidate. It preserves `raw`/`norm`, builds the stable `rv_skeleton_fit` -> bounded -> endpoint-blend -> whole-video support-memory candidate stack, and reports burden/readiness before any downstream feature use.

- Input: `p01_squat_set1` MediaPipe pose CSV and manual annotation CSV
- Pipeline prerequisite: Step 04 preprocessing + Step 05 hip-torso normalization
- Output: notebook-local corrected-3D-hypothesis candidate family and burden/readiness tables
- Boundary: corrected depth has `feature_depth_gravity = 0.0` and is not used for feature extraction, biomechanical proxies, or scoring by default


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from movement.core.config import CONNECTIONS, LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.core.io import load_pose_csv
from movement.definitions.exercise_definition import load_exercise_definition
from movement.pipeline import load_pipeline_config, run_pipeline
from movement.stages.annotation import load_annotation_csv
from movement.stages.validation import run_basic_validation

print("imports OK")
print("project root:", PROJECT_ROOT)


## 01 Run Profile

The YAML file keeps the p01 squat review separate from the default pipeline while declaring the promoted corrected-3D-hypothesis candidate policy used by this notebook.


In [ ]:
RUN_PROFILE_PATH = PROJECT_ROOT / "configs/pipeline_runs/notebook_16_closed_chain_stance_corridor_review.yaml"


def load_yaml_file(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def project_path(value, fallback=None):
    if value is None:
        return fallback
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path


RUN_PROFILE = load_yaml_file(RUN_PROFILE_PATH)
INPUT_PROFILE = RUN_PROFILE.get("input", {})
EXERCISE_PROFILE = RUN_PROFILE.get("exercise", {})
PIPELINE_OVERRIDES = RUN_PROFILE.get("pipeline_overrides", {})
PREPROCESSING_REVIEW_CONFIG = PIPELINE_OVERRIDES.get("preprocessing", {})
CORRIDOR_CONFIG = RUN_PROFILE.get("visualization", {}).get("closed_chain_stance_corridor", {})
REVIEW_DISPLAY_CONFIG = CORRIDOR_CONFIG.get("review_display", {})
PLAYBACK_PROFILE = RUN_PROFILE.get("playback", {})

REVIEW_VISUAL_FOCUS = str(REVIEW_DISPLAY_CONFIG.get("visual_focus", "support_surface_height"))


def review_show_visual(key: str) -> bool:
    focus = REVIEW_VISUAL_FOCUS.lower()
    key = str(key)
    if focus in {"all", "debug_all"}:
        return True
    explicit = set(str(item) for item in REVIEW_DISPLAY_CONFIG.get("enabled_visualizations", []))
    disabled = set(str(item) for item in REVIEW_DISPLAY_CONFIG.get("disabled_visualizations", []))
    if key in disabled:
        return False
    if key in explicit:
        return True
    if key == "support_surface_playback":
        return bool(REVIEW_DISPLAY_CONFIG.get("show_support_surface_playback", focus == "support_surface_height"))
    if key == "final_playback":
        return bool(REVIEW_DISPLAY_CONFIG.get("show_final_playback", REVIEW_DISPLAY_CONFIG.get("show_support_surface_playback", focus == "support_surface_height")))
    if key in {"frame_window_audit", "multi_plane_bend_audit"}:
        return bool(REVIEW_DISPLAY_CONFIG.get("show_audit_plots", False))
    return bool(REVIEW_DISPLAY_CONFIG.get("show_legacy_visualizations", False))


POSE_CSV = project_path(INPUT_PROFILE.get("pose_csv"))
ANNOTATION_CSV = project_path(INPUT_PROFILE.get("annotation_csv"))
CONFIG_PATH = project_path(RUN_PROFILE.get("base_pipeline_config", "configs/pipeline_default.yaml"))
DEFINITIONS_DIR = project_path(EXERCISE_PROFILE.get("definitions_dir", "data/definitions/exercises"))
EXERCISE_ID = str(EXERCISE_PROFILE.get("id", "squat"))
PLAYBACK_STRIDE = int(PLAYBACK_PROFILE.get("stride", 2))

assert POSE_CSV and POSE_CSV.exists(), POSE_CSV
assert ANNOTATION_CSV and ANNOTATION_CSV.exists(), ANNOTATION_CSV
assert CONFIG_PATH and CONFIG_PATH.exists(), CONFIG_PATH
assert CORRIDOR_CONFIG.get("enabled", False), "closed_chain_stance_corridor must be enabled"

print("run profile:", RUN_PROFILE_PATH.relative_to(PROJECT_ROOT))
print("pose csv:", POSE_CSV.relative_to(PROJECT_ROOT))
print("annotation csv:", ANNOTATION_CSV.relative_to(PROJECT_ROOT))
print("corridor accepted/target:", CORRIDOR_CONFIG.get("accepted_side"), "->", CORRIDOR_CONFIG.get("target_side"))
print("promoted candidate stack:", ", ".join(CORRIDOR_CONFIG.get("correction_stack_audit", {}).get("active_review_stack", [])))


## 02 Load Pose And Run Step 04-05

This cell regularizes the pose rows, applies manual annotation through the pipeline, and stops after preprocessing and normalization. Downstream scoring stages stay off, and corrected depth remains excluded from scoring through `feature_depth_gravity = 0.0`.


In [ ]:
RAW_COORD_COLUMNS = [col for landmark in LANDMARKS for col in (f"{landmark}_x", f"{landmark}_y", f"{landmark}_z")]


def estimate_fps_from_timestamp(df: pd.DataFrame, fallback: float = 30.0) -> float:
    if "timestamp" not in df.columns:
        return fallback
    dt = df["timestamp"].dropna().diff().dropna()
    dt = dt[dt > 0]
    return float(1.0 / dt.median()) if not dt.empty else fallback


def trim_empty_pose_frames(df: pd.DataFrame) -> pd.DataFrame:
    present_cols = [col for col in RAW_COORD_COLUMNS if col in df.columns]
    nonempty = df[present_cols].notna().any(axis=1)
    if not nonempty.any():
        raise ValueError("No non-empty pose frames found.")
    return df.loc[nonempty].copy()


def regularize_frame_rows(df: pd.DataFrame, fps: float) -> pd.DataFrame:
    start_frame = int(df["frame"].min())
    end_frame = int(df["frame"].max())
    full_index = pd.DataFrame({"frame": range(start_frame, end_frame + 1)})
    payload = df.drop(columns=["timestamp"], errors="ignore")
    regularized = full_index.merge(payload, on="frame", how="left")
    regularized.insert(1, "timestamp", (regularized["frame"] - start_frame) / fps)
    return regularized


raw_df = load_pose_csv(POSE_CSV)
fps_estimate = estimate_fps_from_timestamp(raw_df)
analysis_df = regularize_frame_rows(trim_empty_pose_frames(raw_df), fps=fps_estimate)
analysis_df.attrs["fps"] = fps_estimate
ann_df = load_annotation_csv(ANNOTATION_CSV)

validation_report = run_basic_validation(
    df=analysis_df,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
structural_passed = all(validation_report[name]["passed"] for name in ["required_columns", "frame_continuity", "timestamp", "missing_values"])
assert structural_passed, "Structural import checks failed."

cfg = load_pipeline_config(CONFIG_PATH)
pre = PREPROCESSING_REVIEW_CONFIG
cfg.annotation.enabled = True
cfg.exercise_definition.enabled = True
cfg.exercise_definition.exercise_id = EXERCISE_ID
cfg.preprocessing.enabled = bool(pre.get("enabled", True))
cfg.preprocessing.interpolation.enabled = bool(pre.get("interpolation", {}).get("enabled", True))
cfg.preprocessing.smoothing.enabled = bool(pre.get("smoothing", {}).get("enabled", False))
cfg.preprocessing.smoothing.method = str(pre.get("smoothing", {}).get("method", cfg.preprocessing.smoothing.method))
cfg.preprocessing.smoothing.window_size = int(pre.get("smoothing", {}).get("window_size", cfg.preprocessing.smoothing.window_size))
far = pre.get("far_side_stabilization", {})
cfg.preprocessing.far_side_stabilization.enabled = bool(far.get("enabled", False))
if far.get("jitter_threshold_torso_per_sec") is not None:
    cfg.preprocessing.far_side_stabilization.jitter_threshold_torso_per_sec = float(far["jitter_threshold_torso_per_sec"])
if far.get("acceleration_threshold_torso_per_sec2") is not None:
    cfg.preprocessing.far_side_stabilization.acceleration_threshold_torso_per_sec2 = float(far["acceleration_threshold_torso_per_sec2"])

norm_overrides = PIPELINE_OVERRIDES.get("normalization", {})
cfg.normalization.enabled = bool(norm_overrides.get("enabled", True))
cfg.normalization.model_depth_scale = float(norm_overrides.get("model_depth_scale", cfg.normalization.model_depth_scale))
corrected_policy = norm_overrides.get("corrected_3d_hypothesis", {}) or {}
cfg.normalization.corrected_3d_hypothesis.enabled = bool(corrected_policy.get("enabled", cfg.normalization.corrected_3d_hypothesis.enabled))
cfg.normalization.corrected_3d_hypothesis.output_family = str(corrected_policy.get("output_family", cfg.normalization.corrected_3d_hypothesis.output_family))
cfg.normalization.corrected_3d_hypothesis.downstream_coordinate_mode = str(corrected_policy.get("downstream_coordinate_mode", cfg.normalization.corrected_3d_hypothesis.downstream_coordinate_mode))
cfg.normalization.corrected_3d_hypothesis.feature_depth_gravity = float(corrected_policy.get("feature_depth_gravity", cfg.normalization.corrected_3d_hypothesis.feature_depth_gravity))
cfg.normalization.corrected_3d_hypothesis.report_burden_before_feature_use = bool(corrected_policy.get("report_burden_before_feature_use", cfg.normalization.corrected_3d_hypothesis.report_burden_before_feature_use))
cfg.normalization.corrected_3d_hypothesis.require_feature_domain_declaration = bool(corrected_policy.get("require_feature_domain_declaration", cfg.normalization.corrected_3d_hypothesis.require_feature_domain_declaration))
cfg.canonicalization.enabled = False
cfg.rep_segmentation.enabled = True
cfg.phase_segmentation.enabled = False
cfg.motion_attribution.enabled = False
cfg.features.enabled = False
cfg.biomech.enabled = False
cfg.biomarker.enabled = False

result_df, report = run_pipeline(analysis_df, cfg, ann_df=ann_df, landmarks=LANDMARKS)

summary = {
    "frames": len(result_df),
    "fps_estimate": round(fps_estimate, 3),
    "validation_passed": validation_report["passed"],
    "annotation_reps": report.get("annotation", {}).get("num_reps"),
    "model_depth_scale": report.get("normalization", {}).get("model_depth_scale"),
    "corrected_output_family": report.get("normalization", {}).get("corrected_3d_hypothesis", {}).get("output_family"),
    "feature_depth_gravity": report.get("normalization", {}).get("corrected_3d_hypothesis", {}).get("feature_depth_gravity"),
    "corrected_used_for_scores": report.get("normalization", {}).get("corrected_3d_hypothesis", {}).get("used_for_features_or_scores"),
}
display(pd.DataFrame([summary]))


## 03 Shared Helpers And Promoted Candidate Setup

This section keeps only the shared helpers needed by the promoted corrected-3D-hypothesis stack. Retired stance-corridor and visual-template branches are no longer generated in this notebook.


In [ ]:
from plotly.subplots import make_subplots

COMPACT_REP_ID = int(CORRIDOR_CONFIG.get("review_rep_id", 1))
corridor_df = result_df.copy()


def fcol(landmark: str, family: str, axis: str) -> str:
    return f"{landmark}_{axis}" if family == "raw" else f"{landmark}_{family}_{axis}"


def point_array(df: pd.DataFrame, landmark: str, family: str) -> np.ndarray:
    return df[[fcol(landmark, family, axis) for axis in ["x", "y", "z"]]].to_numpy(float)


def set_point(df: pd.DataFrame, landmark: str, family: str, values: np.ndarray) -> None:
    for idx, axis in enumerate(["x", "y", "z"]):
        df[fcol(landmark, family, axis)] = values[:, idx]


def median_point(df: pd.DataFrame, landmark: str, family: str, mask: np.ndarray) -> np.ndarray:
    values = point_array(df.loc[mask], landmark, family)
    return np.nanmedian(values, axis=0)


def segment_length(df: pd.DataFrame, a: str, b: str, family: str, mask: np.ndarray | None = None) -> np.ndarray:
    values = np.linalg.norm(point_array(df, a, family) - point_array(df, b, family), axis=1)
    return values if mask is None else values[mask]


def segment_length_series(df: pd.DataFrame, a: str, b: str, family: str) -> np.ndarray:
    return np.linalg.norm(point_array(df, a, family) - point_array(df, b, family), axis=1)


def landmark_visibility(df: pd.DataFrame, landmark: str) -> np.ndarray:
    for column in [f"{landmark}_visibility", f"{landmark}_vis", f"{landmark}_presence"]:
        if column in df.columns:
            return df[column].astype(float).to_numpy()
    return np.full(len(df), np.nan, dtype=float)


def trimmed_median(values: np.ndarray, trim_q: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    trim_q = float(np.clip(trim_q, 0.0, 0.45))
    if trim_q > 0.0 and arr.size >= 4:
        low, high = np.nanquantile(arr, [trim_q, 1.0 - trim_q])
        trimmed = arr[(arr >= low) & (arr <= high)]
        if trimmed.size:
            arr = trimmed
    return float(np.nanmedian(arr))


def closest_z_for_link(anchor: np.ndarray, knee_current: np.ndarray, target_length: float) -> tuple[np.ndarray, np.ndarray]:
    current_z = knee_current[:, 2]
    if not np.isfinite(target_length) or target_length <= 0.0:
        return current_z.copy(), np.zeros(len(knee_current), dtype=bool)
    base_xy = (knee_current[:, 0] - anchor[:, 0]) ** 2 + (knee_current[:, 1] - anchor[:, 1]) ** 2
    feasible = target_length**2 >= base_xy
    dz_abs = np.sqrt(np.maximum(target_length**2 - base_xy, 0.0))
    z_plus = anchor[:, 2] + dz_abs
    z_minus = anchor[:, 2] - dz_abs
    chosen = np.where(np.abs(z_plus - current_z) <= np.abs(z_minus - current_z), z_plus, z_minus)
    chosen[~feasible] = current_z[~feasible]
    return chosen, feasible


def solve_knee_z_shift(
    hip: np.ndarray,
    knee: np.ndarray,
    ankle: np.ndarray,
    thigh_target: float,
    shank_target: float,
    mask: np.ndarray,
    strength: float,
    shift_cap: float,
    tolerance: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    thigh_z, thigh_feasible = closest_z_for_link(hip, knee, thigh_target)
    shank_z, shank_feasible = closest_z_for_link(ankle, knee, shank_target)
    current_z = knee[:, 2]
    desired_z = current_z.copy()
    both = thigh_feasible & shank_feasible
    thigh_only = thigh_feasible & ~shank_feasible
    shank_only = shank_feasible & ~thigh_feasible
    desired_z[both] = 0.5 * (thigh_z[both] + shank_z[both])
    desired_z[thigh_only] = thigh_z[thigh_only]
    desired_z[shank_only] = shank_z[shank_only]

    current_thigh = np.linalg.norm(knee - hip, axis=1)
    current_shank = np.linalg.norm(knee - ankle, axis=1)
    length_error = np.maximum(np.abs(current_thigh - thigh_target), np.abs(current_shank - shank_target))
    active = (length_error > tolerance) & mask & (thigh_feasible | shank_feasible)
    z_shift = np.zeros(len(knee), dtype=float)
    z_shift[active] = np.clip((desired_z[active] - current_z[active]) * strength, -shift_cap, shift_cap)
    infeasible = mask & ~(thigh_feasible | shank_feasible)
    return z_shift, active, infeasible


def ready_window_mask(df: pd.DataFrame, ann: pd.DataFrame, cfg: dict) -> tuple[np.ndarray, dict]:
    rep_id = int(cfg.get("review_rep_id", COMPACT_REP_ID))
    rep_rows = ann[(ann["segment_type"].eq("rep")) & (ann["rep_id"].astype("Int64").eq(rep_id))]
    if rep_rows.empty:
        raise ValueError(f"No rep annotation found for rep_id={rep_id}")
    first_rep = rep_rows.iloc[0]
    rep_start = int(first_rep["start_frame"])
    window_cfg = cfg.get("reference_window", {})
    start = rep_start - int(window_cfg.get("frames_before_first_rep", 20))
    if window_cfg.get("exclude_baseline_before_frame") is not None:
        start = max(start, int(window_cfg["exclude_baseline_before_frame"]))
    end = rep_start + int(window_cfg.get("frames_after_first_rep_start", 0))
    mask = df["frame"].between(start, end).to_numpy()
    return mask, {"rep_id": rep_id, "rep_start": rep_start, "start": int(start), "end": int(end), "frame_count": int(mask.sum())}


def nested_config(root: dict, path: str) -> dict:
    node = root
    for part in str(path).split("."):
        if not isinstance(node, dict):
            return {}
        node = node.get(part, {})
    return node if isinstance(node, dict) else {}


scene_camera_cfg = CORRIDOR_CONFIG.get("scene_camera", {})
SCENE_PROJECTION_TYPE = str(scene_camera_cfg.get("projection_type", "orthographic"))
COMPACT_SCENE_CAMERA = {
    "eye": dict(scene_camera_cfg.get("eye", {"x": 1.45, "y": -2.25, "z": 1.05})),
    "up": dict(scene_camera_cfg.get("up", {"x": 0.0, "y": 0.0, "z": 1.0})),
    "projection": {"type": SCENE_PROJECTION_TYPE},
}
COMPACT_TOP_DOWN_CAMERA = {
    "eye": dict(scene_camera_cfg.get("top_down_eye", {"x": 0.0, "y": 0.0, "z": 2.6})),
    "up": dict(scene_camera_cfg.get("top_down_up", {"x": 0.0, "y": 1.0, "z": 0.0})),
    "projection": {"type": "orthographic"},
}


def plot_xyz(row: pd.Series, landmark: str, family: str) -> tuple[float, float, float]:
    return (
        float(row[fcol(landmark, family, "x")]),
        float(row[fcol(landmark, family, "z")]),
        -float(row[fcol(landmark, family, "y")]),
    )


def skeleton_traces(row: pd.Series, family: str, label: str, color: str) -> list[go.Scatter3d]:
    px, py, pz = [], [], []
    for landmark in LANDMARKS:
        if not all(fcol(landmark, family, axis) in row.index for axis in ["x", "y", "z"]):
            continue
        x, y, z = plot_xyz(row, landmark, family)
        px.append(x)
        py.append(y)
        pz.append(z)

    lx, ly, lz = [], [], []
    for a, b in CONNECTIONS:
        if not all(fcol(landmark, family, axis) in row.index for landmark in [a, b] for axis in ["x", "y", "z"]):
            continue
        ax, ay, az = plot_xyz(row, a, family)
        bx, by, bz = plot_xyz(row, b, family)
        lx.extend([ax, bx, None])
        ly.extend([ay, by, None])
        lz.extend([az, bz, None])

    return [
        go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=5), name=f"{label} skeleton"),
        go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=3), name=f"{label} joints"),
    ]


def finite_range(values: list[float], padding: float = 0.15) -> list[float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return [-1.0, 1.0]
    low = float(arr.min())
    high = float(arr.max())
    if np.isclose(low, high):
        low -= 0.5
        high += 0.5
    span = high - low
    pad = float(padding) * span if span > 0 else float(padding)
    return [low - pad, high + pad]


def plot_ranges(df: pd.DataFrame, families: list[str], padding: float = 0.15) -> tuple[list[float], list[float], list[float]]:
    xs, ys, zs = [], [], []
    for _, row in df.iterrows():
        for family in families:
            for landmark in LANDMARKS:
                if not all(fcol(landmark, family, axis) in row.index for axis in ["x", "y", "z"]):
                    continue
                x, y, z = plot_xyz(row, landmark, family)
                xs.append(x)
                ys.append(y)
                zs.append(z)
    return finite_range(xs, padding), finite_range(ys, padding), finite_range(zs, padding)


def add_floor_axis_guide(fig: go.Figure, ranges: tuple[list[float], list[float], list[float]]) -> None:
    x_range, y_range, z_range = ranges
    origin = np.array([
        float(x_range[0] + 0.08 * (x_range[1] - x_range[0])),
        float(y_range[0] + 0.08 * (y_range[1] - y_range[0])),
        float(z_range[0] + 0.08 * (z_range[1] - z_range[0])),
    ])
    lengths = np.array([x_range[1] - x_range[0], y_range[1] - y_range[0], z_range[1] - z_range[0]]) * 0.18
    for label, vector, color in [
        ("recording x", np.array([lengths[0], 0.0, 0.0]), "rgba(0,170,170,0.9)"),
        ("model depth z", np.array([0.0, lengths[1], 0.0]), "rgba(255,150,40,0.9)"),
        ("height (-y)", np.array([0.0, 0.0, lengths[2]]), "rgba(120,120,120,0.85)"),
    ]:
        end = origin + vector
        fig.add_trace(go.Scatter3d(
            x=[origin[0], end[0]], y=[origin[1], end[1]], z=[origin[2], end[2]],
            mode="lines+markers+text", line=dict(color=color, width=6), marker=dict(color=color, size=5),
            text=["", label], textposition="top center", name=label, showlegend=False,
        ))


mask_ref, reference_info = ready_window_mask(corridor_df, ann_df, CORRIDOR_CONFIG)
compact_rep_mask = corridor_df["rep_id"].astype("Int64").eq(COMPACT_REP_ID).fillna(False).to_numpy(dtype=bool)
compact_rep_df = corridor_df.loc[compact_rep_mask].copy()

setup_summary_df = pd.DataFrame([
    {
        "review_rep_id": COMPACT_REP_ID,
        "ready_window_start": reference_info["start"],
        "ready_window_end": reference_info["end"],
        "ready_window_frames": reference_info["frame_count"],
        "rep_frames": int(compact_rep_mask.sum()),
        "promoted_candidate_surface": "rv_skeleton_fit_bounded_xy_endpoint_blend_support_memory",
        "feature_depth_gravity": report.get("normalization", {}).get("corrected_3d_hypothesis", {}).get("feature_depth_gravity", 0.0),
        "used_for_features_or_scores": report.get("normalization", {}).get("corrected_3d_hypothesis", {}).get("used_for_features_or_scores", False),
    }
])
display(Markdown("### Promoted corrected-3D-hypothesis setup"))
display(setup_summary_df)


## 08 Within-Session Stable Segment Memory

This review table estimates session-specific segment-length references from frames that are both visible and temporally stable. Accepted memory targets are not ground truth and are not per-frame caps; they can only narrow the soft target used by the following skeleton-placement candidates.


In [ ]:

RV_PARENT_CONFIG = CORRIDOR_CONFIG.get("recording_view_constrained_skeleton_fit", {})
SEGMENT_MEMORY_CONFIG = RV_PARENT_CONFIG.get("within_session_segment_memory", {})
SEGMENT_MEMORY_ENABLED = bool(SEGMENT_MEMORY_CONFIG.get("enabled", False))
segment_memory_reference_df = pd.DataFrame()
SEGMENT_MEMORY_ACCEPTED_BY_ID: dict[str, float] = {}


def memory_safe_median(values: np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanmedian(arr)) if arr.size else float("nan")


def memory_safe_percentile(values: np.ndarray, percentile: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanpercentile(arr, percentile)) if arr.size else float("nan")


if not SEGMENT_MEMORY_ENABLED:
    display(Markdown("`within_session_segment_memory` is disabled in the run profile."))
else:
    memory_family = str(SEGMENT_MEMORY_CONFIG.get("source_family", "norm"))
    memory_rep_id = int(SEGMENT_MEMORY_CONFIG.get("review_rep_id", COMPACT_REP_ID))
    memory_rep_mask = corridor_df["rep_id"].astype("Int64").eq(memory_rep_id).fillna(False).to_numpy(dtype=bool)
    memory_reference_mask = np.zeros(len(corridor_df), dtype=bool)
    if bool(SEGMENT_MEMORY_CONFIG.get("include_ready_window", True)):
        memory_reference_mask |= mask_ref
    if bool(SEGMENT_MEMORY_CONFIG.get("include_review_rep", True)):
        memory_reference_mask |= memory_rep_mask
    if "use_for_analysis" in corridor_df.columns:
        memory_reference_mask &= (corridor_df["use_for_analysis"].astype(bool).to_numpy() | mask_ref)
    if not np.any(memory_reference_mask):
        memory_reference_mask = memory_rep_mask.copy() if np.any(memory_rep_mask) else compact_rep_mask.copy()

    min_endpoint_visibility = float(SEGMENT_MEMORY_CONFIG.get("min_endpoint_visibility", 0.55))
    min_stable_frames = int(SEGMENT_MEMORY_CONFIG.get("min_stable_frames", 12))
    max_range_ratio = float(SEGMENT_MEMORY_CONFIG.get("max_p95_p05_ratio", 0.30))
    max_jump_ratio = float(SEGMENT_MEMORY_CONFIG.get("max_frame_to_frame_jump_ratio", 0.16))
    trim_q = float(np.clip(float(SEGMENT_MEMORY_CONFIG.get("trim_quantile", 0.10)), 0.0, 0.45))

    memory_rows = []
    fig_memory = go.Figure()
    memory_plot_frames = corridor_df.loc[memory_reference_mask, "frame"].to_numpy(int)

    for segment_cfg in SEGMENT_MEMORY_CONFIG.get("segments", []):
        memory_id = str(segment_cfg.get("id"))
        segment_name = str(segment_cfg.get("segment"))
        proximal = str(segment_cfg.get("proximal"))
        distal = str(segment_cfg.get("distal"))
        lengths = segment_length_series(corridor_df, proximal, distal, memory_family)
        visibility = np.nanmin(
            np.vstack([landmark_visibility(corridor_df, proximal), landmark_visibility(corridor_df, distal)]),
            axis=0,
        )
        visibility_ok = np.where(np.isfinite(visibility), visibility >= min_endpoint_visibility, True)
        frame_jump = np.abs(np.diff(lengths, prepend=np.nan))
        jump_ok = np.isnan(frame_jump) | (frame_jump <= max_jump_ratio)
        finite = np.isfinite(lengths)
        candidate_mask = memory_reference_mask & finite
        stable_mask = candidate_mask & visibility_ok & jump_ok
        stable_lengths = lengths[stable_mask]
        candidate_lengths = lengths[candidate_mask]

        stable_count = int(np.sum(stable_mask))
        candidate_count = int(np.sum(candidate_mask))
        memory_length = trimmed_median(stable_lengths, trim_q) if stable_count else float("nan")
        p05 = memory_safe_percentile(stable_lengths, 5)
        p95 = memory_safe_percentile(stable_lengths, 95)
        iqr = memory_safe_percentile(stable_lengths, 75) - memory_safe_percentile(stable_lengths, 25)
        range_ratio = (p95 - p05) / memory_length if np.isfinite(memory_length) and memory_length > 0 else float("nan")
        p95_jump = memory_safe_percentile(frame_jump[candidate_mask], 95)
        median_visibility = memory_safe_median(visibility[candidate_mask])
        stable_acceptance_ratio = stable_count / candidate_count if candidate_count else float("nan")

        rejection_reasons = []
        if stable_count < min_stable_frames:
            rejection_reasons.append("too_few_stable_frames")
        if np.isfinite(range_ratio) and range_ratio > max_range_ratio:
            rejection_reasons.append("length_range_unstable")
        if not np.isfinite(memory_length):
            rejection_reasons.append("memory_length_unavailable")
        stability_accepted = not rejection_reasons
        if stability_accepted:
            SEGMENT_MEMORY_ACCEPTED_BY_ID[memory_id] = memory_length

        memory_rows.append({
            "memory_id": memory_id,
            "segment": segment_name,
            "endpoints": f"{proximal}->{distal}",
            "source_family": memory_family,
            "candidate_frames": candidate_count,
            "stable_frames": stable_count,
            "stable_window_acceptance_ratio": stable_acceptance_ratio,
            "memory_length": memory_length,
            "memory_iqr": iqr,
            "p05_length": p05,
            "p95_length": p95,
            "p95_p05_range_ratio": range_ratio,
            "median_endpoint_visibility": median_visibility,
            "p95_frame_to_frame_jump": p95_jump,
            "stability_accepted": stability_accepted,
            "rejection_reasons": ",".join(rejection_reasons) if rejection_reasons else "",
        })

        fig_memory.add_trace(go.Scatter(
            x=memory_plot_frames,
            y=lengths[memory_reference_mask],
            mode="lines",
            name=f"{memory_id} observed",
        ))
        if np.isfinite(memory_length):
            fig_memory.add_trace(go.Scatter(
                x=memory_plot_frames,
                y=np.full(len(memory_plot_frames), memory_length),
                mode="lines",
                name=f"{memory_id} memory",
                line=dict(dash="dash"),
            ))

    segment_memory_reference_df = pd.DataFrame(memory_rows)
    display(Markdown("### within-session stable segment-memory reference table"))
    display(segment_memory_reference_df)
    fig_memory.update_layout(
        title="Reference-window segment lengths for stable segment memory",
        xaxis_title="source frame",
        yaxis_title="torso-length ratio",
        height=520,
        width=1100,
    )
    if review_show_visual("segment_memory"):
        fig_memory.show()


## 09 Recording-View-Constrained Skeleton Placement (`rv_skeleton_fit`)

This review candidate places the common-subject skeleton in candidate 3D analysis space while keeping the recording-view camera-plane evidence as the strongest constraint. It starts with `norm`, preserves x/y by construction, adjusts only candidate model-depth `z` for configured skeleton chains/widths, and reports residuals before any feature can consume it.


In [ ]:

RV_FIT_CONFIG = CORRIDOR_CONFIG.get("recording_view_constrained_skeleton_fit", {})
RV_FIT_ENABLED = bool(RV_FIT_CONFIG.get("enabled", True))
RV_FIT_FAMILY = str(RV_FIT_CONFIG.get("family", "rv_skeleton_fit"))
RV_FIT_SOURCE_FAMILY = str(RV_FIT_CONFIG.get("source_family", "norm"))
RV_FIT_REP_ID = int(RV_FIT_CONFIG.get("review_rep_id", COMPACT_REP_ID))


def rv_safe_median(values: np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanmedian(arr)) if arr.size else float("nan")


def rv_safe_percentile(values: np.ndarray, percentile: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanpercentile(arr, percentile)) if arr.size else float("nan")


def rv_normalize_sex(value: str) -> str:
    text = str(value).strip().lower()
    if text in {"male", "m", "man", "남", "남성", "남자"}:
        return "male"
    if text in {"female", "f", "woman", "여", "여성", "여자"}:
        return "female"
    raise ValueError(f"Cannot resolve participant sex for skeleton matrix: {value!r}")


def rv_matrix_path_from_profile() -> Path:
    participant = RUN_PROFILE.get("participant", {})
    target_cfg = RV_FIT_CONFIG.get("target_policy", {})
    configured = str(target_cfg.get("skeleton_matrix_path", "participant.common_subject_skeleton_matrix_path"))
    if configured == "participant.common_subject_skeleton_matrix_path":
        configured = participant.get("common_subject_skeleton_matrix_path")
    if not configured:
        configured = "data/reference/anthropometry/common_subject_skeleton_matrix.yaml"
    return project_path(configured)


def rv_target_lengths_from_matrix() -> dict[str, dict[str, float]]:
    participant = RUN_PROFILE.get("participant", {})
    sex = rv_normalize_sex(participant.get("sex", "male"))
    matrix_root = load_yaml_file(rv_matrix_path_from_profile()).get("common_subject_skeleton_matrix", {})
    axes = matrix_root.get("axes", {})
    segments = list(axes.get("segments", []))
    sexes = list(axes.get("sexes", []))
    sex_index = sexes.index(sex)
    ratio_values = matrix_root.get("ratio_matrices", {}).get("values", {})

    def ratio(segment: str, field: str) -> float:
        if segment not in segments:
            return float("nan")
        values = ratio_values.get(field, [])
        raw = values[segments.index(segment)][sex_index]
        return float("nan") if raw is None else float(raw)

    torso_ratio = ratio("torso_shoulder_hip_vertical", "target_ratio_to_stature")
    if not np.isfinite(torso_ratio) or torso_ratio <= 0:
        raise ValueError("torso_shoulder_hip_vertical target ratio is required for normalized skeleton targets")

    targets: dict[str, dict[str, float]] = {}
    for segment in segments:
        target_ratio = ratio(segment, "target_ratio_to_stature")
        if not np.isfinite(target_ratio):
            continue
        targets[segment] = {
            "target": target_ratio / torso_ratio,
            "soft_lower": ratio(segment, "soft_lower_ratio_to_stature") / torso_ratio,
            "soft_upper": ratio(segment, "soft_upper_ratio_to_stature") / torso_ratio,
            "hard_lower": ratio(segment, "hard_lower_ratio_to_stature") / torso_ratio,
            "hard_upper": ratio(segment, "hard_upper_ratio_to_stature") / torso_ratio,
        }
    return targets



segment_memory_blended_targets_df = pd.DataFrame()
SEGMENT_MEMORY_TARGET_BY_ID: dict[str, float] = {}


def rv_prepare_segment_memory_targets(targets: dict[str, dict[str, float]]) -> dict[str, float]:
    global segment_memory_blended_targets_df
    memory_cfg = globals().get("SEGMENT_MEMORY_CONFIG", {})
    memory_df = globals().get("segment_memory_reference_df", pd.DataFrame())
    accepted_lengths = globals().get("SEGMENT_MEMORY_ACCEPTED_BY_ID", {})
    if not bool(memory_cfg.get("enabled", False)) or not isinstance(memory_df, pd.DataFrame) or memory_df.empty:
        segment_memory_blended_targets_df = pd.DataFrame()
        return {}

    blend_cfg = memory_cfg.get("blend_with_anthropometric", {})
    blend_enabled = bool(blend_cfg.get("enabled", True))
    memory_weight = float(np.clip(float(blend_cfg.get("memory_weight", 0.65)), 0.0, 1.0))
    max_deviation = float(memory_cfg.get("max_memory_deviation_from_anthropometric_ratio", 0.25))
    rows = []
    target_by_id: dict[str, float] = {}
    for _, row in memory_df.iterrows():
        memory_id = str(row.get("memory_id", ""))
        segment = str(row.get("segment", ""))
        memory_length = float(accepted_lengths.get(memory_id, row.get("memory_length", np.nan)))
        anthropometric_target = float(targets.get(segment, {}).get("target", np.nan))
        deviation_ratio = abs(memory_length - anthropometric_target) / anthropometric_target if np.isfinite(memory_length) and np.isfinite(anthropometric_target) and anthropometric_target > 0 else np.nan
        accepted = bool(row.get("stability_accepted", False)) and np.isfinite(deviation_ratio) and deviation_ratio <= max_deviation
        if accepted:
            blended = (1.0 - memory_weight) * anthropometric_target + memory_weight * memory_length if blend_enabled else memory_length
            target_by_id[memory_id] = float(blended)
        else:
            blended = anthropometric_target
        rows.append({
            "memory_id": memory_id,
            "segment": segment,
            "stability_accepted": bool(row.get("stability_accepted", False)),
            "anthropometric_target": anthropometric_target,
            "memory_length": memory_length,
            "deviation_ratio": deviation_ratio,
            "max_deviation_ratio": max_deviation,
            "memory_weight": memory_weight if accepted and blend_enabled else 0.0,
            "placement_target": blended,
            "used_for_placement": accepted,
        })
    segment_memory_blended_targets_df = pd.DataFrame(rows)
    return target_by_id


def rv_default_memory_id(chain_id: str, segment_name: str) -> str:
    side = str(chain_id).split("_", 1)[0]
    return f"{side}_{segment_name}"


def rv_target_length(segment_name: str, memory_id: str | None = None) -> float:
    if memory_id and memory_id in SEGMENT_MEMORY_TARGET_BY_ID:
        return float(SEGMENT_MEMORY_TARGET_BY_ID[memory_id])
    return float(rv_targets.get(segment_name, {}).get("target", float("nan")))



def rv_family_has_coordinates(df: pd.DataFrame, family: str, landmarks: list[str] | tuple[str, ...]) -> bool:
    return all(fcol(landmark, family, axis) in df.columns for landmark in landmarks for axis in ["x", "y", "z"])

def rv_copy_coordinate_family(df: pd.DataFrame, source_family: str, target_family: str) -> pd.DataFrame:
    target_columns = [fcol(landmark, target_family, axis) for landmark in LANDMARKS for axis in ["x", "y", "z"]]
    copied = {
        fcol(landmark, target_family, axis): df[fcol(landmark, source_family, axis)].to_numpy(float)
        for landmark in LANDMARKS
        for axis in ["x", "y", "z"]
    }
    return pd.concat([df.drop(columns=target_columns, errors="ignore"), pd.DataFrame(copied, index=df.index)], axis=1)


def rv_chain_report_row(chain_id: str, segment_name: str, proximal: str, distal: str, family: str, target: float, mask: np.ndarray) -> dict:
    lengths = segment_length_series(corridor_df, proximal, distal, family)
    residual = np.abs(lengths - target)
    return {
        "chain_or_width": chain_id,
        "segment": segment_name,
        "endpoints": f"{proximal}->{distal}",
        "family": family,
        "target_length": target,
        "median_length": rv_safe_median(lengths[mask]),
        "median_abs_residual": rv_safe_median(residual[mask]),
        "p95_abs_residual": rv_safe_percentile(residual[mask], 95),
    }



def rv_support_center(df: pd.DataFrame, side: str, family: str, suffixes: list[str]) -> np.ndarray:
    arrays = []
    for suffix in suffixes:
        landmark = f"{side}_{suffix}"
        if landmark in LANDMARKS and rv_family_has_coordinates(df, family, [landmark]):
            arrays.append(point_array(df, landmark, family))
    if not arrays:
        return np.full((len(df), 3), np.nan)
    return np.nanmedian(np.stack(arrays, axis=2), axis=2)


def rv_axis_config_values(raw: dict | float | int, default: float = 0.0) -> np.ndarray:
    if isinstance(raw, dict):
        return np.array([float(raw.get(axis, default)) for axis in ["x", "y", "z"]], dtype=float)
    value = float(raw) if raw is not None else default
    return np.array([value, value, value], dtype=float)


def rv_apply_closed_chain_support_context(
    df: pd.DataFrame,
    source_family: str,
    target_family: str,
    support_cfg: dict,
    fit_mask: np.ndarray,
    reference_mask: np.ndarray,
) -> tuple[pd.DataFrame, set[str]]:
    if not bool(support_cfg.get("enabled", False)):
        return pd.DataFrame(), set()

    def trimmed_median(values: np.ndarray, trim_quantile: float) -> float:
        finite = np.asarray(values, dtype=float)
        finite = finite[np.isfinite(finite)]
        if finite.size == 0:
            return float("nan")
        trim_quantile = float(np.clip(trim_quantile, 0.0, 0.45))
        if trim_quantile > 0 and finite.size >= 3:
            lo, hi = np.nanquantile(finite, [trim_quantile, 1.0 - trim_quantile])
            trimmed = finite[(finite >= lo) & (finite <= hi)]
            if trimmed.size:
                finite = trimmed
        return float(np.nanmedian(finite))

    center_suffixes = [str(item) for item in support_cfg.get("center_landmark_suffixes", ["ankle", "heel", "foot_index"])]
    movable_suffixes = [str(item) for item in support_cfg.get("movable_landmark_suffixes", center_suffixes)]
    sides = [str(item) for item in support_cfg.get("sides", ["left", "right"])]
    strength_axes = rv_axis_config_values(support_cfg.get("strength", {"x": 0.0, "y": 0.0, "z": 0.8}), 0.0)
    cap_axes = rv_axis_config_values(support_cfg.get("max_shift_ratio", {"x": 0.0, "y": 0.0, "z": 0.18}), 0.0)
    side_strength = support_cfg.get("side_strength", {})
    apply_mask = fit_mask.copy()
    if not bool(support_cfg.get("include_ready_window", True)):
        apply_mask &= ~reference_mask
    if not bool(support_cfg.get("apply_to_review_rep", True)):
        apply_mask &= reference_mask
    if not np.any(apply_mask):
        apply_mask = fit_mask.copy()

    ref_mask = reference_mask.copy()
    if not np.any(ref_mask):
        ref_mask = fit_mask.copy()

    guard_cfg = support_cfg.get("support_width_guard", {})
    guard_enabled = bool(guard_cfg.get("enabled", False))
    guard_pair = [str(item) for item in guard_cfg.get("pair", ["left_ankle", "right_ankle"])]
    guard_reject_worse = bool(guard_cfg.get("reject_shift_when_residual_worsens", True))
    guard_target_width = float("nan")
    guard_tolerance_abs = float("nan")
    if guard_enabled and len(guard_pair) == 2:
        left_guard, right_guard = guard_pair
        if (
            left_guard in LANDMARKS
            and right_guard in LANDMARKS
            and rv_family_has_coordinates(df, source_family, guard_pair)
            and rv_family_has_coordinates(df, target_family, guard_pair)
        ):
            source_left = point_array(df, left_guard, source_family)
            source_right = point_array(df, right_guard, source_family)
            source_width = np.linalg.norm((source_right - source_left)[:, [0, 2]], axis=1)
            guard_target_width = trimmed_median(source_width[ref_mask], float(guard_cfg.get("trim_quantile", 0.10)))
            if np.isfinite(guard_target_width):
                guard_tolerance_abs = guard_target_width * float(guard_cfg.get("tolerance_ratio", 0.005))
            else:
                guard_enabled = False
        else:
            guard_enabled = False

    rows = []
    moved_landmarks: set[str] = set()
    for side in sides:
        source_center = rv_support_center(df, side, source_family, center_suffixes)
        target_center = rv_support_center(df, side, target_family, center_suffixes)
        if not np.isfinite(source_center[ref_mask]).any() or not np.isfinite(target_center[apply_mask]).any():
            rows.append({
                "support_side": side,
                "status": "unavailable",
                "candidate_frames": int(np.sum(apply_mask)),
                "reference_frames": int(np.sum(ref_mask)),
                "support_width_guard_enabled": bool(guard_enabled),
                "support_width_guard_rejected_frame_ratio": np.nan,
                "support_width_guard_target": guard_target_width,
            })
            continue

        reference_center = np.nanmedian(source_center[ref_mask], axis=0)
        raw_shift = reference_center[None, :] - target_center
        side_scale = float(side_strength.get(side, 1.0)) if isinstance(side_strength, dict) else float(side_strength or 1.0)
        shift = raw_shift * strength_axes[None, :] * side_scale
        shift = np.clip(shift, -cap_axes[None, :], cap_axes[None, :])
        shift[~apply_mask] = 0.0

        guard_rejected = np.zeros(len(df), dtype=bool)
        if guard_enabled and guard_reject_worse and len(guard_pair) == 2 and np.isfinite(guard_target_width):
            left_guard, right_guard = guard_pair
            guard_left = point_array(df, left_guard, target_family).copy()
            guard_right = point_array(df, right_guard, target_family).copy()
            guard_left_after = guard_left.copy()
            guard_right_after = guard_right.copy()
            if left_guard.startswith(f"{side}_"):
                guard_left_after += shift
            if right_guard.startswith(f"{side}_"):
                guard_right_after += shift
            before_width = np.linalg.norm((guard_right - guard_left)[:, [0, 2]], axis=1)
            after_width = np.linalg.norm((guard_right_after - guard_left_after)[:, [0, 2]], axis=1)
            before_error = np.abs(before_width - guard_target_width)
            after_error = np.abs(after_width - guard_target_width)
            guard_rejected = (
                apply_mask
                & np.isfinite(before_error)
                & np.isfinite(after_error)
                & (after_error > before_error + guard_tolerance_abs)
            )
            shift[guard_rejected] = 0.0

        before_residual = np.linalg.norm((target_center - reference_center[None, :])[:, [0, 2]], axis=1)
        after_center = target_center + shift
        after_residual = np.linalg.norm((after_center - reference_center[None, :])[:, [0, 2]], axis=1)

        for suffix in movable_suffixes:
            landmark = f"{side}_{suffix}"
            if landmark not in LANDMARKS or not rv_family_has_coordinates(df, target_family, [landmark]):
                continue
            xyz = point_array(df, landmark, target_family).copy()
            xyz[apply_mask] = xyz[apply_mask] + shift[apply_mask]
            set_point(df, landmark, target_family, xyz)
            moved_landmarks.add(landmark)

        shift_norm = np.linalg.norm(shift[:, [0, 2]], axis=1)
        rows.append({
            "support_side": side,
            "status": "applied",
            "candidate_frames": int(np.sum(apply_mask)),
            "reference_frames": int(np.sum(ref_mask)),
            "side_strength": side_scale,
            "support_width_guard_enabled": bool(guard_enabled),
            "support_width_guard_target": guard_target_width,
            "support_width_guard_tolerance": guard_tolerance_abs,
            "support_width_guard_rejected_frames": int(np.sum(guard_rejected & apply_mask)),
            "support_width_guard_rejected_frame_ratio": float(np.nanmean(guard_rejected[apply_mask])) if np.any(apply_mask) else np.nan,
            "median_center_residual_before": rv_safe_median(before_residual[apply_mask]),
            "median_center_residual_after": rv_safe_median(after_residual[apply_mask]),
            "median_abs_x_shift": rv_safe_median(np.abs(shift[:, 0])[apply_mask]),
            "median_abs_y_shift": rv_safe_median(np.abs(shift[:, 1])[apply_mask]),
            "median_abs_z_shift": rv_safe_median(np.abs(shift[:, 2])[apply_mask]),
            "p95_abs_z_shift": rv_safe_percentile(np.abs(shift[:, 2])[apply_mask], 95),
            "max_abs_z_shift": float(np.nanmax(np.abs(shift[:, 2])[apply_mask])) if np.any(apply_mask) else np.nan,
            "active_frame_ratio": float(np.nanmean(shift_norm[apply_mask] > 1e-9)) if np.any(apply_mask) else np.nan,
            "moved_landmarks": ",".join(sorted([lm for lm in moved_landmarks if lm.startswith(f"{side}_")]))
        })
    return pd.DataFrame(rows), moved_landmarks

def rv_apply_width_z_projection(
    df: pd.DataFrame,
    left: str,
    right: str,
    family: str,
    target_length: float,
    mask: np.ndarray,
    strength: float,
    shift_cap: float,
    tolerance: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    left_xyz = point_array(df, left, family).copy()
    right_xyz = point_array(df, right, family).copy()
    current_length = np.linalg.norm(right_xyz - left_xyz, axis=1)
    current_residual = np.abs(current_length - target_length)
    dxdy_sq = (right_xyz[:, 0] - left_xyz[:, 0]) ** 2 + (right_xyz[:, 1] - left_xyz[:, 1]) ** 2
    feasible = np.isfinite(target_length) & (target_length > 0) & (target_length**2 >= dxdy_sq)
    active = mask & feasible & (current_residual > tolerance)
    infeasible = mask & ~feasible
    left_shift = np.zeros_like(left_xyz)
    right_shift = np.zeros_like(right_xyz)
    if np.any(active):
        center_z = 0.5 * (left_xyz[:, 2] + right_xyz[:, 2])
        dz_current = right_xyz[:, 2] - left_xyz[:, 2]
        sign_values = dz_current[mask & np.isfinite(dz_current) & (np.abs(dz_current) > 1e-9)]
        fallback_sign = float(np.sign(np.nanmedian(sign_values))) if sign_values.size else 1.0
        if not np.isfinite(fallback_sign) or fallback_sign == 0.0:
            fallback_sign = 1.0
        dz_sign = np.sign(dz_current)
        dz_sign[np.abs(dz_current) <= 1e-9] = fallback_sign
        target_abs_dz = np.sqrt(np.maximum(target_length**2 - dxdy_sq, 0.0))
        target_left_z = center_z - 0.5 * dz_sign * target_abs_dz
        target_right_z = center_z + 0.5 * dz_sign * target_abs_dz
        left_shift[:, 2] = np.clip((target_left_z - left_xyz[:, 2]) * strength, -shift_cap, shift_cap)
        right_shift[:, 2] = np.clip((target_right_z - right_xyz[:, 2]) * strength, -shift_cap, shift_cap)
        left_shift[~active] = 0.0
        right_shift[~active] = 0.0
        set_point(df, left, family, left_xyz + left_shift)
        set_point(df, right, family, right_xyz + right_shift)
    return left_shift, right_shift, active, infeasible


def rv_bend_guard_plane_axes(plane_id: str) -> tuple[int, int]:
    planes = {
        "xz_recording_depth": (0, 2),
        "yz_height_depth": (1, 2),
        "xy_recording_plane": (0, 1),
    }
    if plane_id not in planes:
        raise ValueError(f"Unknown bend-side guard plane: {plane_id}")
    return planes[plane_id]


def rv_signed_bend_distance(proximal_xyz: np.ndarray, joint_xyz: np.ndarray, distal_xyz: np.ndarray, axes: tuple[int, int]) -> np.ndarray:
    proximal_2d = proximal_xyz[:, list(axes)]
    joint_2d = joint_xyz[:, list(axes)]
    distal_2d = distal_xyz[:, list(axes)]
    base = distal_2d - proximal_2d
    joint_vector = joint_2d - proximal_2d
    base_norm = np.linalg.norm(base, axis=1)
    signed_area = base[:, 0] * joint_vector[:, 1] - base[:, 1] * joint_vector[:, 0]
    signed_distance = np.full(len(proximal_xyz), np.nan, dtype=float)
    valid = np.isfinite(signed_area) & np.isfinite(base_norm) & (base_norm > 1e-9)
    signed_distance[valid] = signed_area[valid] / base_norm[valid]
    return signed_distance


def rv_frame_label_list(df: pd.DataFrame, mask: np.ndarray, limit: int = 24) -> str:
    if "frame" in df.columns:
        labels = df.loc[mask, "frame"].astype("Int64").astype(str).tolist()
    else:
        labels = df.index[mask].astype(str).tolist()
    if len(labels) > limit:
        return ",".join(labels[:limit]) + f",...(+{len(labels) - limit})"
    return ",".join(labels)


def rv_apply_soft_bend_side_guard(
    df: pd.DataFrame,
    source_family: str,
    target_family: str,
    proximal: str,
    joint: str,
    distal: str,
    candidate_joint_xyz: np.ndarray,
    active_mask: np.ndarray,
    guard_cfg: dict,
    fit_component: str,
    fit_mask: np.ndarray,
    candidate_proximal_xyz: np.ndarray | None = None,
    candidate_distal_xyz: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, list[dict]]:
    adjusted_joint_xyz = np.asarray(candidate_joint_xyz, dtype=float).copy()
    guarded = np.zeros(len(df), dtype=bool)
    if not bool(guard_cfg.get("enabled", False)):
        return adjusted_joint_xyz, guarded, []

    source_family = str(guard_cfg.get("source_family", source_family))
    if not rv_family_has_coordinates(df, source_family, [proximal, joint, distal]):
        return adjusted_joint_xyz, guarded, []
    if not rv_family_has_coordinates(df, target_family, [proximal, joint, distal]):
        return adjusted_joint_xyz, guarded, []

    mode = str(guard_cfg.get("mode", "soft_reject_candidate_shift"))
    report_only = bool(guard_cfg.get("report_only", False))
    near_neutral = float(guard_cfg.get("near_neutral_tolerance_ratio", 0.010))
    soft_flip = float(guard_cfg.get("soft_flip_tolerance_ratio", 0.012))
    planes = [str(item) for item in guard_cfg.get("planes", ["yz_height_depth", "xz_recording_depth"])]

    source_proximal = point_array(df, proximal, source_family)
    source_joint = point_array(df, joint, source_family)
    source_distal = point_array(df, distal, source_family)
    target_proximal = point_array(df, proximal, target_family)
    target_joint_before = point_array(df, joint, target_family).copy()
    target_distal = point_array(df, distal, target_family)
    if candidate_proximal_xyz is not None:
        target_proximal = np.asarray(candidate_proximal_xyz, dtype=float)
    if candidate_distal_xyz is not None:
        target_distal = np.asarray(candidate_distal_xyz, dtype=float)

    rows: list[dict] = []
    for plane_id in planes:
        axes = rv_bend_guard_plane_axes(plane_id)
        source_distance = rv_signed_bend_distance(source_proximal, source_joint, source_distal, axes)
        candidate_distance = rv_signed_bend_distance(target_proximal, adjusted_joint_xyz, target_distal, axes)
        source_sign = np.sign(source_distance)
        candidate_sign = np.sign(candidate_distance)
        source_sign[np.abs(source_distance) <= near_neutral] = 0.0
        candidate_sign[np.abs(candidate_distance) <= near_neutral] = 0.0
        finite = np.isfinite(source_distance) & np.isfinite(candidate_distance)
        candidate_exceeds_soft = np.abs(candidate_distance) >= soft_flip
        sign_conflict = (source_sign != 0.0) & (candidate_sign != 0.0) & (source_sign != candidate_sign)
        unsupported_from_neutral = (source_sign == 0.0) & (candidate_sign != 0.0) & candidate_exceeds_soft
        plane_guarded = (
            np.asarray(active_mask, dtype=bool)
            & finite
            & (candidate_sign != 0.0)
            & candidate_exceeds_soft
            & (sign_conflict | unsupported_from_neutral)
        )
        guarded |= plane_guarded
        rows.append({
            "fit_component": fit_component,
            "type": "soft_bend_side_guard",
            "joint": joint,
            "source_family": source_family,
            "target_family": target_family,
            "plane": plane_id,
            "mode": mode,
            "report_only": report_only,
            "near_neutral_tolerance_ratio": near_neutral,
            "soft_flip_tolerance_ratio": soft_flip,
            "guarded_frame_count": int(np.sum(plane_guarded & fit_mask)),
            "guarded_frame_ratio": float(np.nanmean(plane_guarded[fit_mask])) if np.any(fit_mask) else np.nan,
            "guarded_frames": rv_frame_label_list(df, plane_guarded & fit_mask),
        })

    if mode == "soft_reject_candidate_shift" and not report_only and np.any(guarded):
        adjusted_joint_xyz[guarded] = target_joint_before[guarded]
    return adjusted_joint_xyz, guarded, rows


EXERCISE_DEFINITION_RAW: dict | None = None


def rv_loaded_exercise_definition_raw() -> dict:
    global EXERCISE_DEFINITION_RAW
    if EXERCISE_DEFINITION_RAW is None:
        path = DEFINITIONS_DIR / f"{EXERCISE_ID}.yaml"
        EXERCISE_DEFINITION_RAW = load_yaml_file(path) if path.exists() else {}
    return EXERCISE_DEFINITION_RAW


def rv_trimmed_median(values: np.ndarray, trim_quantile: float = 0.10) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return float("nan")
    trim_quantile = float(np.clip(trim_quantile, 0.0, 0.45))
    if trim_quantile > 0.0 and finite.size >= 3:
        lo, hi = np.nanquantile(finite, [trim_quantile, 1.0 - trim_quantile])
        trimmed = finite[(finite >= lo) & (finite <= hi)]
        if trimmed.size:
            finite = trimmed
    return float(np.nanmedian(finite))


def rv_axis_index(axis: str) -> int:
    axes = {"x": 0, "y": 1, "z": 2}
    axis = str(axis).lower()
    if axis not in axes:
        raise ValueError(f"Unsupported support-surface guard axis: {axis}")
    return axes[axis]


def rv_support_surface_gate(guard_cfg: dict) -> tuple[bool, str]:
    if not bool(guard_cfg.get("enabled", False)):
        return False, "disabled"
    if not bool(guard_cfg.get("exercise_definition_gate", True)):
        return True, "config_only"
    raw = rv_loaded_exercise_definition_raw()
    classification = raw.get("classification", {})
    support = raw.get("support", {})
    required_chain = str(guard_cfg.get("required_kinetic_chain", "closed_chain"))
    required_base = str(guard_cfg.get("required_base_of_support", "bilateral_feet"))
    required_surface = str(guard_cfg.get("required_support_surface", "floor"))
    checks = [
        str(classification.get("kinetic_chain", "")) == required_chain,
        str(support.get("base_of_support", "")) == required_base,
        str(support.get("support_surface", "")) == required_surface,
    ]
    return (all(checks), "exercise_definition" if all(checks) else "exercise_definition_mismatch")


def rv_apply_support_surface_height_guard(
    df: pd.DataFrame,
    target_family: str,
    source_family: str,
    guard_cfg: dict,
    fit_mask: np.ndarray,
    reference_mask: np.ndarray,
) -> tuple[pd.DataFrame, set[str]]:
    gate_ok, gate_source = rv_support_surface_gate(guard_cfg)
    if not bool(guard_cfg.get("enabled", False)):
        return pd.DataFrame(), set()
    if not gate_ok:
        return pd.DataFrame([{
            "fit_component": "support_surface_height",
            "type": "support_surface_height_guard",
            "status": "skipped",
            "gate_source": gate_source,
            "exercise_id": EXERCISE_ID,
        }]), set()

    source_family = str(guard_cfg.get("source_family", source_family))
    axis = str(guard_cfg.get("axis", "y"))
    axis_i = rv_axis_index(axis)
    sides = [str(item) for item in guard_cfg.get("sides", ["left", "right"])]
    suffixes = [str(item) for item in guard_cfg.get("landmark_suffixes", ["ankle", "heel", "foot_index"])]
    primary_suffixes = {str(item) for item in guard_cfg.get("primary_suffixes", ["ankle"])}
    strength_default = float(np.clip(float(guard_cfg.get("strength", 0.55)), 0.0, 1.0))
    side_strength = dict(guard_cfg.get("side_strength", {}))
    max_shift = float(guard_cfg.get("max_shift_ratio", 0.04))
    tolerance = float(guard_cfg.get("residual_tolerance_ratio", 0.012))
    trim_quantile = float(guard_cfg.get("trim_quantile", 0.10))
    reject_worse = bool(guard_cfg.get("reject_shift_when_residual_worsens", True))
    report_only = bool(guard_cfg.get("report_only", False))

    ref_mask = reference_mask.copy() if str(guard_cfg.get("reference_source", "ready_window_median")) == "ready_window_median" else fit_mask.copy()
    if not np.any(ref_mask):
        ref_mask = fit_mask.copy()

    rows = []
    moved_landmarks: set[str] = set()
    for suffix in suffixes:
        landmarks = [f"{side}_{suffix}" for side in sides if f"{side}_{suffix}" in LANDMARKS]
        landmarks = [lm for lm in landmarks if rv_family_has_coordinates(df, target_family, [lm]) and rv_family_has_coordinates(df, source_family, [lm])]
        if not landmarks:
            continue

        target_values = []
        for landmark in landmarks:
            source_xyz = point_array(df, landmark, source_family)
            target_values.append(source_xyz[ref_mask, axis_i])
        target_height = rv_trimmed_median(np.concatenate(target_values), trim_quantile) if target_values else float("nan")
        if not np.isfinite(target_height):
            rows.append({
                "fit_component": "support_surface_height",
                "type": "support_surface_height_guard",
                "status": "no_reference_height",
                "suffix": suffix,
                "gate_source": gate_source,
                "exercise_id": EXERCISE_ID,
            })
            continue

        before_by_landmark: dict[str, np.ndarray] = {}
        after_by_landmark: dict[str, np.ndarray] = {}
        shift_by_landmark: dict[str, np.ndarray] = {}
        active_union = np.zeros(len(df), dtype=bool)
        for landmark in landmarks:
            xyz = point_array(df, landmark, target_family).copy()
            before_y = xyz[:, axis_i].copy()
            before_residual = np.abs(before_y - target_height)
            side = landmark.split("_", 1)[0]
            strength = float(np.clip(float(side_strength.get(side, strength_default)), 0.0, 1.0))
            active = fit_mask & np.isfinite(before_y) & (before_residual > tolerance)
            shift = np.zeros(len(df), dtype=float)
            shift[active] = np.clip((target_height - before_y[active]) * strength, -max_shift, max_shift)
            candidate_y = before_y + shift
            after_residual = np.abs(candidate_y - target_height)
            if reject_worse:
                worsened = active & (after_residual > before_residual)
                shift[worsened] = 0.0
                candidate_y = before_y + shift
                after_residual = np.abs(candidate_y - target_height)
            if not report_only:
                xyz[:, axis_i] = candidate_y
                set_point(df, landmark, target_family, xyz)
                if np.any(np.abs(shift[fit_mask]) > 1e-9):
                    moved_landmarks.add(landmark)
            before_by_landmark[landmark] = before_y
            after_by_landmark[landmark] = candidate_y
            shift_by_landmark[landmark] = shift
            active_union |= active & (np.abs(shift) > 1e-9)

        pair_gaps_before = []
        pair_gaps_after = []
        if len(sides) >= 2:
            left_lm = f"{sides[0]}_{suffix}"
            right_lm = f"{sides[1]}_{suffix}"
            if left_lm in before_by_landmark and right_lm in before_by_landmark:
                pair_gaps_before.append(np.abs(before_by_landmark[right_lm] - before_by_landmark[left_lm]))
                pair_gaps_after.append(np.abs(after_by_landmark[right_lm] - after_by_landmark[left_lm]))
        all_shifts = np.vstack([np.abs(values) for values in shift_by_landmark.values()]) if shift_by_landmark else np.zeros((1, len(df)))
        all_after_residuals = np.vstack([np.abs(values - target_height) for values in after_by_landmark.values()]) if after_by_landmark else np.zeros((1, len(df)))
        rows.append({
            "fit_component": "support_surface_height",
            "type": "support_surface_height_guard",
            "status": "applied" if np.any(active_union & fit_mask) and not report_only else "reported",
            "gate_source": gate_source,
            "exercise_id": EXERCISE_ID,
            "target_family": target_family,
            "source_family": source_family,
            "axis": axis,
            "suffix": suffix,
            "primary_proxy": suffix in primary_suffixes,
            "target_height": target_height,
            "active_frame_ratio": float(np.nanmean(active_union[fit_mask])) if np.any(fit_mask) else np.nan,
            "median_abs_y_shift": rv_safe_median(np.nanmax(all_shifts[:, fit_mask], axis=0)),
            "p95_abs_y_shift": rv_safe_percentile(np.nanmax(all_shifts[:, fit_mask], axis=0), 95),
            "max_abs_y_shift": float(np.nanmax(all_shifts[:, fit_mask])) if np.any(fit_mask) else np.nan,
            "median_pair_height_gap_before": rv_safe_median(pair_gaps_before[0][fit_mask]) if pair_gaps_before else np.nan,
            "median_pair_height_gap_after": rv_safe_median(pair_gaps_after[0][fit_mask]) if pair_gaps_after else np.nan,
            "remaining_large_residual_frame_ratio": float(np.nanmean(np.nanmax(all_after_residuals[:, fit_mask], axis=0) > tolerance)) if np.any(fit_mask) else np.nan,
            "applied_frames": rv_frame_label_list(df, active_union & fit_mask),
        })
    return pd.DataFrame(rows), moved_landmarks


rv_fit_summary_df = pd.DataFrame()
rv_fit_segment_report_df = pd.DataFrame()
rv_fit_burden_df = pd.DataFrame()
rv_fit_support_report_df = pd.DataFrame()
rv_support_context_report_df = pd.DataFrame()

if not RV_FIT_ENABLED:
    display(Markdown("`rv_skeleton_fit` is disabled in the run profile."))
else:
    corridor_df = rv_copy_coordinate_family(corridor_df, RV_FIT_SOURCE_FAMILY, RV_FIT_FAMILY)
    rv_targets = rv_target_lengths_from_matrix()
    SEGMENT_MEMORY_TARGET_BY_ID = rv_prepare_segment_memory_targets(rv_targets)
    if not segment_memory_blended_targets_df.empty:
        display(Markdown("### segment-memory placement target blend"))
        display(segment_memory_blended_targets_df)
    rv_rep_mask = corridor_df["rep_id"].astype("Int64").eq(RV_FIT_REP_ID).fillna(False).to_numpy(dtype=bool)
    rv_fit_mask = rv_rep_mask.copy()
    if bool(RV_FIT_CONFIG.get("include_ready_window", True)):
        rv_fit_mask |= mask_ref
    if "use_for_analysis" in corridor_df.columns:
        rv_fit_mask &= (corridor_df["use_for_analysis"].astype(bool).to_numpy() | mask_ref)
    if not np.any(rv_fit_mask):
        rv_fit_mask = compact_rep_mask.copy()

    solver_cfg = RV_FIT_CONFIG.get("solver", {})
    rv_strength = float(np.clip(float(solver_cfg.get("strength", 0.85)), 0.0, 1.0))
    rv_tolerance = float(solver_cfg.get("tolerance_ratio", 0.015))
    rv_joint_shift_cap = float(solver_cfg.get("max_joint_z_shift_ratio", 0.45))
    rv_width_shift_cap = float(solver_cfg.get("max_width_z_shift_ratio", 0.25))
    rv_bend_guard_cfg = solver_cfg.get("bend_side_guard", {})

    rv_segment_rows = []
    rv_chain_rows = []
    rv_active_landmarks: set[str] = set()

    rv_support_context_report_df, rv_support_context_landmarks = rv_apply_closed_chain_support_context(
        corridor_df,
        RV_FIT_SOURCE_FAMILY,
        RV_FIT_FAMILY,
        RV_FIT_CONFIG.get("closed_chain_support_context", {}),
        rv_fit_mask,
        mask_ref,
    )
    rv_active_landmarks.update(rv_support_context_landmarks)

    for chain in RV_FIT_CONFIG.get("chains", []):
        chain_id = str(chain.get("id", "chain"))
        proximal = str(chain["proximal"])
        joint = str(chain["joint"])
        distal = str(chain["distal"])
        proximal_segment = str(chain["proximal_segment"])
        distal_segment = str(chain["distal_segment"])
        proximal_memory_id = str(chain.get("proximal_memory_id", rv_default_memory_id(chain_id, proximal_segment)))
        distal_memory_id = str(chain.get("distal_memory_id", rv_default_memory_id(chain_id, distal_segment)))
        proximal_target = rv_target_length(proximal_segment, proximal_memory_id)
        distal_target = rv_target_length(distal_segment, distal_memory_id)
        rv_active_landmarks.update([proximal, joint, distal])

        rv_segment_rows.append(rv_chain_report_row(chain_id, proximal_segment, proximal, joint, RV_FIT_SOURCE_FAMILY, proximal_target, rv_fit_mask))
        rv_segment_rows.append(rv_chain_report_row(chain_id, distal_segment, joint, distal, RV_FIT_SOURCE_FAMILY, distal_target, rv_fit_mask))

        proximal_xyz = point_array(corridor_df, proximal, RV_FIT_FAMILY)
        joint_xyz = point_array(corridor_df, joint, RV_FIT_FAMILY).copy()
        distal_xyz = point_array(corridor_df, distal, RV_FIT_FAMILY)
        z_shift, active, infeasible = solve_knee_z_shift(
            proximal_xyz,
            joint_xyz,
            distal_xyz,
            proximal_target,
            distal_target,
            rv_fit_mask,
            rv_strength,
            rv_joint_shift_cap,
            rv_tolerance,
        )
        candidate_joint_xyz = joint_xyz.copy()
        candidate_joint_xyz[:, 2] = candidate_joint_xyz[:, 2] + z_shift
        guarded_joint_xyz, bend_guarded, bend_guard_rows = rv_apply_soft_bend_side_guard(
            corridor_df,
            RV_FIT_SOURCE_FAMILY,
            RV_FIT_FAMILY,
            proximal,
            joint,
            distal,
            candidate_joint_xyz,
            active,
            rv_bend_guard_cfg,
            chain_id,
            rv_fit_mask,
        )
        z_shift = guarded_joint_xyz[:, 2] - joint_xyz[:, 2]
        joint_xyz = guarded_joint_xyz
        set_point(corridor_df, joint, RV_FIT_FAMILY, joint_xyz)
        rv_chain_rows.extend(bend_guard_rows)
        rv_chain_rows.append({
            "fit_component": chain_id,
            "type": "two_link_z_projection",
            "joint": joint,
            "active_frame_ratio": float(np.nanmean(active[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "infeasible_frame_ratio": float(np.nanmean(infeasible[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "median_abs_z_shift": rv_safe_median(np.abs(z_shift[rv_fit_mask])),
            "p95_abs_z_shift": rv_safe_percentile(np.abs(z_shift[rv_fit_mask]), 95),
            "max_abs_z_shift": float(np.nanmax(np.abs(z_shift[rv_fit_mask]))) if np.any(rv_fit_mask) else np.nan,
            "bend_guarded_frame_ratio": float(np.nanmean(bend_guarded[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "bend_guarded_frames": rv_frame_label_list(corridor_df, bend_guarded & rv_fit_mask),
        })

        rv_segment_rows.append(rv_chain_report_row(chain_id, proximal_segment, proximal, joint, RV_FIT_FAMILY, proximal_target, rv_fit_mask))
        rv_segment_rows.append(rv_chain_report_row(chain_id, distal_segment, joint, distal, RV_FIT_FAMILY, distal_target, rv_fit_mask))

    for width_cfg in RV_FIT_CONFIG.get("width_segments", []):
        width_id = str(width_cfg.get("id", "width"))
        left = str(width_cfg["left"])
        right = str(width_cfg["right"])
        segment = str(width_cfg["segment"])
        memory_id = str(width_cfg.get("memory_id", width_id))
        target_length = rv_target_length(segment, memory_id)
        rv_active_landmarks.update([left, right])
        rv_segment_rows.append(rv_chain_report_row(width_id, segment, left, right, RV_FIT_SOURCE_FAMILY, target_length, rv_fit_mask))
        left_before = point_array(corridor_df, left, RV_FIT_FAMILY).copy()
        right_before = point_array(corridor_df, right, RV_FIT_FAMILY).copy()
        left_shift, right_shift, active, infeasible = rv_apply_width_z_projection(
            corridor_df,
            left,
            right,
            RV_FIT_FAMILY,
            target_length,
            rv_fit_mask,
            rv_strength,
            rv_width_shift_cap,
            rv_tolerance,
        )
        width_bend_guarded = np.zeros(len(corridor_df), dtype=bool)
        width_bend_guard_rows: list[dict] = []
        left_after = point_array(corridor_df, left, RV_FIT_FAMILY).copy()
        right_after = point_array(corridor_df, right, RV_FIT_FAMILY).copy()
        for guard_chain in RV_FIT_CONFIG.get("chains", []):
            guard_chain_id = str(guard_chain.get("id", "chain"))
            guard_proximal = str(guard_chain["proximal"])
            guard_joint = str(guard_chain["joint"])
            guard_distal = str(guard_chain["distal"])
            candidate_proximal_xyz = None
            candidate_distal_xyz = None
            if guard_proximal == left:
                candidate_proximal_xyz = left_after
            elif guard_proximal == right:
                candidate_proximal_xyz = right_after
            if guard_distal == left:
                candidate_distal_xyz = left_after
            elif guard_distal == right:
                candidate_distal_xyz = right_after
            if candidate_proximal_xyz is None and candidate_distal_xyz is None:
                continue
            _, chain_guarded, chain_guard_rows = rv_apply_soft_bend_side_guard(
                corridor_df,
                RV_FIT_SOURCE_FAMILY,
                RV_FIT_FAMILY,
                guard_proximal,
                guard_joint,
                guard_distal,
                point_array(corridor_df, guard_joint, RV_FIT_FAMILY),
                active,
                rv_bend_guard_cfg,
                f"{width_id}:{guard_chain_id}",
                rv_fit_mask,
                candidate_proximal_xyz=candidate_proximal_xyz,
                candidate_distal_xyz=candidate_distal_xyz,
            )
            width_bend_guard_rows.extend(chain_guard_rows)
            width_bend_guarded |= chain_guarded
            if candidate_proximal_xyz is left_after or candidate_distal_xyz is left_after:
                left_after[chain_guarded] = left_before[chain_guarded]
            if candidate_proximal_xyz is right_after or candidate_distal_xyz is right_after:
                right_after[chain_guarded] = right_before[chain_guarded]
        if np.any(width_bend_guarded):
            set_point(corridor_df, left, RV_FIT_FAMILY, left_after)
            set_point(corridor_df, right, RV_FIT_FAMILY, right_after)
            left_shift = left_after - left_before
            right_shift = right_after - right_before
        rv_chain_rows.extend(width_bend_guard_rows)
        rv_chain_rows.append({
            "fit_component": width_id,
            "type": "pair_width_z_projection",
            "joint": f"{left}/{right}",
            "active_frame_ratio": float(np.nanmean(active[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "infeasible_frame_ratio": float(np.nanmean(infeasible[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "median_abs_z_shift": rv_safe_median(np.maximum(np.abs(left_shift[:, 2]), np.abs(right_shift[:, 2]))[rv_fit_mask]),
            "p95_abs_z_shift": rv_safe_percentile(np.maximum(np.abs(left_shift[:, 2]), np.abs(right_shift[:, 2]))[rv_fit_mask], 95),
            "max_abs_z_shift": float(np.nanmax(np.maximum(np.abs(left_shift[:, 2]), np.abs(right_shift[:, 2]))[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "bend_guarded_frame_ratio": float(np.nanmean(width_bend_guarded[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
            "bend_guarded_frames": rv_frame_label_list(corridor_df, width_bend_guarded & rv_fit_mask),
        })
        rv_segment_rows.append(rv_chain_report_row(width_id, segment, left, right, RV_FIT_FAMILY, target_length, rv_fit_mask))

    rv_fit_segment_report_df = pd.DataFrame(rv_segment_rows)
    rv_fit_burden_df = pd.DataFrame(rv_chain_rows)

    rv_recording_deltas = []
    rv_depth_deltas = []
    for landmark in sorted(rv_active_landmarks):
        source_xyz = point_array(corridor_df, landmark, RV_FIT_SOURCE_FAMILY)
        fit_xyz = point_array(corridor_df, landmark, RV_FIT_FAMILY)
        rv_recording_deltas.append(np.linalg.norm(fit_xyz[:, :2] - source_xyz[:, :2], axis=1))
        rv_depth_deltas.append(np.abs(fit_xyz[:, 2] - source_xyz[:, 2]))
    rv_recording_delta_frame = np.nanmax(np.vstack(rv_recording_deltas), axis=0) if rv_recording_deltas else np.zeros(len(corridor_df))
    rv_depth_delta_frame = np.nanmax(np.vstack(rv_depth_deltas), axis=0) if rv_depth_deltas else np.zeros(len(corridor_df))

    support_cfg = RV_FIT_CONFIG.get("support_residual", {})
    if bool(support_cfg.get("enabled", True)):
        pair = list(support_cfg.get("pair", ["left_ankle", "right_ankle"]))
        trim_q = float(np.clip(float(support_cfg.get("trim_quantile", 0.10)), 0.0, 0.45))
        source_width = np.linalg.norm(
            point_array(corridor_df, pair[1], RV_FIT_SOURCE_FAMILY)[:, [0, 2]] - point_array(corridor_df, pair[0], RV_FIT_SOURCE_FAMILY)[:, [0, 2]],
            axis=1,
        )
        fit_width = np.linalg.norm(
            point_array(corridor_df, pair[1], RV_FIT_FAMILY)[:, [0, 2]] - point_array(corridor_df, pair[0], RV_FIT_FAMILY)[:, [0, 2]],
            axis=1,
        )
        support_target = trimmed_median(source_width[mask_ref], trim_q) if np.any(mask_ref) else trimmed_median(source_width[rv_fit_mask], trim_q)
        rv_fit_support_report_df = pd.DataFrame([{
            "support_pair": "->".join(pair),
            "target_source": support_cfg.get("target_source", "ready_window_median"),
            "target_width": support_target,
            "median_width_norm": rv_safe_median(source_width[rv_fit_mask]),
            "median_width_rv_skeleton_fit": rv_safe_median(fit_width[rv_fit_mask]),
            "median_abs_residual_norm": rv_safe_median(np.abs(source_width[rv_fit_mask] - support_target)),
            "median_abs_residual_rv_skeleton_fit": rv_safe_median(np.abs(fit_width[rv_fit_mask] - support_target)),
        }])

    before_rows = rv_fit_segment_report_df[rv_fit_segment_report_df["family"].eq(RV_FIT_SOURCE_FAMILY)]
    after_rows = rv_fit_segment_report_df[rv_fit_segment_report_df["family"].eq(RV_FIT_FAMILY)]
    rv_fit_summary_df = pd.DataFrame([{
        "candidate_family": RV_FIT_FAMILY,
        "source_family": RV_FIT_SOURCE_FAMILY,
        "review_rep_id": RV_FIT_REP_ID,
        "fit_frames": int(rv_fit_mask.sum()),
        "active_landmark_count": len(rv_active_landmarks),
        "median_recording_xy_residual": rv_safe_median(rv_recording_delta_frame[rv_fit_mask]),
        "max_recording_xy_residual": float(np.nanmax(rv_recording_delta_frame[rv_fit_mask])) if np.any(rv_fit_mask) else np.nan,
        "median_depth_shift": rv_safe_median(rv_depth_delta_frame[rv_fit_mask]),
        "p95_depth_shift": rv_safe_percentile(rv_depth_delta_frame[rv_fit_mask], 95),
        "median_segment_residual_before": rv_safe_median(before_rows["median_abs_residual"].to_numpy(float)),
        "median_segment_residual_after": rv_safe_median(after_rows["median_abs_residual"].to_numpy(float)),
        "used_for_features_or_scores": bool(RV_FIT_CONFIG.get("reporting", {}).get("used_for_features_or_scores", False)),
        "confidence_when_applied": RV_FIT_CONFIG.get("reporting", {}).get("confidence_when_applied", "low"),
    }])

    display(Markdown("### rv_skeleton_fit burden ledger"))
    display(rv_fit_summary_df)
    display(rv_fit_burden_df)
    if not rv_support_context_report_df.empty:
        display(Markdown("### closed-chain support-context burden"))
        display(rv_support_context_report_df)
    display(rv_fit_support_report_df)

    rv_length_plot_segments = [
        ("left thigh", "left_hip", "left_knee", "thigh"),
        ("right thigh", "right_hip", "right_knee", "thigh"),
        ("left shank", "left_knee", "left_ankle", "shank"),
        ("right shank", "right_knee", "right_ankle", "shank"),
    ]
    rv_plot_df = corridor_df.loc[rv_rep_mask].copy() if np.any(rv_rep_mask) else corridor_df.loc[rv_fit_mask].copy()
    fig_rv_lengths = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("thigh length", "shank length"))
    for label, a, b, segment_name in rv_length_plot_segments:
        row_idx = 1 if segment_name == "thigh" else 2
        target_length = rv_targets.get(segment_name, {}).get("target", float("nan"))
        frames = rv_plot_df["frame"].to_numpy(int)
        fig_rv_lengths.add_trace(go.Scatter(x=frames, y=segment_length_series(rv_plot_df, a, b, RV_FIT_SOURCE_FAMILY), mode="lines", name=f"norm {label}", line=dict(color="rgba(0,0,0,0.55)", dash="dot")), row=row_idx, col=1)
        fig_rv_lengths.add_trace(go.Scatter(x=frames, y=segment_length_series(rv_plot_df, a, b, RV_FIT_FAMILY), mode="lines", name=f"rv {label}", line=dict(width=3)), row=row_idx, col=1)
        if np.isfinite(target_length):
            fig_rv_lengths.add_trace(go.Scatter(x=frames, y=np.full(len(frames), target_length), mode="lines", name=f"target {segment_name}", line=dict(color="rgba(160,160,160,0.55)", dash="dash"), showlegend=(label.startswith("left"))), row=row_idx, col=1)
    fig_rv_lengths.update_layout(title="Rep 1 segment lengths: norm vs rv_skeleton_fit", height=640, width=1100)
    fig_rv_lengths.update_yaxes(title_text="torso-length ratio")
    fig_rv_lengths.update_xaxes(title_text="source frame", row=2, col=1)
    if review_show_visual("rv_fit_lengths"):
        fig_rv_lengths.show()

    def rv_fit_traces(row: pd.Series) -> list[go.Scatter3d]:
        traces = skeleton_traces(row, RV_FIT_SOURCE_FAMILY, "norm", "rgba(0,0,0,0.95)")
        traces += skeleton_traces(row, RV_FIT_FAMILY, RV_FIT_FAMILY, "rgba(220,40,80,0.92)")
        return traces

    rv_anim_stride = max(1, int(PLAYBACK_STRIDE))
    rv_anim_df = rv_plot_df.iloc[::rv_anim_stride].copy()
    rv_ranges = plot_ranges(rv_plot_df, [RV_FIT_SOURCE_FAMILY, RV_FIT_FAMILY])
    rv_frame_duration_ms = max(10, int(1000 / max(float(globals().get("fps_estimate", PLAYBACK_PROFILE.get("fps", 30.0))), 1.0)))
    fig_rv_fit_play = go.Figure(data=rv_fit_traces(rv_anim_df.iloc[0]))
    fig_rv_fit_play.frames = [
        go.Frame(data=rv_fit_traces(row), name=str(int(row["frame"])))
        for _, row in rv_anim_df.iterrows()
    ]
    fig_rv_fit_play.update_layout(
        title=f"Rep {RV_FIT_REP_ID}: norm vs recording-view-constrained skeleton fit ({RV_FIT_FAMILY})",
        scene=dict(
            xaxis=dict(title="recording x", range=rv_ranges[0]),
            yaxis=dict(title="model depth z", range=rv_ranges[1]),
            zaxis=dict(title="recording height (-y)", range=rv_ranges[2]),
            aspectmode="data",
            camera=COMPACT_SCENE_CAMERA,
        ),
        height=720,
        width=1050,
        showlegend=True,
        sliders=[{
            "active": 0,
            "currentvalue": {"prefix": "Source frame: "},
            "pad": {"t": 50},
            "steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}}], "label": frame.name, "method": "animate"} for frame in fig_rv_fit_play.frames],
        }],
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 0.0,
            "y": 1.0,
            "xanchor": "left",
            "yanchor": "top",
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": rv_frame_duration_ms, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True, "mode": "immediate"}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
            ],
        }],
    )
    add_floor_axis_guide(fig_rv_fit_play, rv_ranges)
    if review_show_visual("rv_fit_playback"):
        fig_rv_fit_play.show()


## 10 Bounded Recording-View Residual Skeleton Placement (`rv_skeleton_fit_bounded_xy`)

This review candidate extends the strict `rv_skeleton_fit` baseline by allowing a small, explicitly bounded camera-plane residual. It remains a low-confidence 3D-hypothesis output: the residual is reported as burden and must not be used as recording-view-only feature evidence.


In [ ]:

BOUNDED_RV_CONFIG = RV_FIT_CONFIG.get("bounded_recording_view_residual", {})
BOUNDED_RV_ENABLED = bool(BOUNDED_RV_CONFIG.get("enabled", False))
BOUNDED_RV_FAMILY = str(BOUNDED_RV_CONFIG.get("family", "rv_skeleton_fit_bounded_xy"))
BOUNDED_RV_SOURCE_FAMILY = str(BOUNDED_RV_CONFIG.get("source_family", RV_FIT_SOURCE_FAMILY))
BOUNDED_RV_BASELINE_FAMILY = str(BOUNDED_RV_CONFIG.get("baseline_family", RV_FIT_FAMILY))
BOUNDED_RV_REP_ID = int(BOUNDED_RV_CONFIG.get("review_rep_id", RV_FIT_REP_ID))


def rv_family_has_coordinates(df: pd.DataFrame, family: str, landmarks: list[str] | tuple[str, ...]) -> bool:
    return all(fcol(landmark, family, axis) in df.columns for landmark in landmarks for axis in ["x", "y", "z"])


def rv_clip_xyz_shift(shift: np.ndarray, xy_cap: float, z_cap: float) -> np.ndarray:
    clipped = np.asarray(shift, dtype=float).copy()
    clipped[:, 0] = np.clip(clipped[:, 0], -xy_cap, xy_cap)
    clipped[:, 1] = np.clip(clipped[:, 1], -xy_cap, xy_cap)
    clipped[:, 2] = np.clip(clipped[:, 2], -z_cap, z_cap)
    return clipped


def rv_perpendicular_unit(base_unit: np.ndarray, current_perp: np.ndarray) -> np.ndarray:
    perp_norm = np.linalg.norm(current_perp, axis=1)
    unit = np.zeros_like(current_perp)
    has_perp = perp_norm > 1e-9
    unit[has_perp] = current_perp[has_perp] / perp_norm[has_perp, None]

    missing = ~has_perp
    if np.any(missing):
        fallback_a = np.cross(base_unit[missing], np.array([0.0, 1.0, 0.0]))
        fallback_norm = np.linalg.norm(fallback_a, axis=1)
        fallback_b = np.cross(base_unit[missing], np.array([0.0, 0.0, 1.0]))
        use_b = fallback_norm <= 1e-9
        fallback_a[use_b] = fallback_b[use_b]
        fallback_norm = np.linalg.norm(fallback_a, axis=1)
        fallback_norm[fallback_norm <= 1e-9] = 1.0
        unit[missing] = fallback_a / fallback_norm[:, None]
    return unit


def rv_apply_two_link_bounded_projection(
    df: pd.DataFrame,
    proximal: str,
    joint: str,
    distal: str,
    family: str,
    proximal_target: float,
    distal_target: float,
    mask: np.ndarray,
    strength: float,
    xy_cap: float,
    z_cap: float,
    tolerance: float,
    guard_cfg: dict | None = None,
    source_family: str | None = None,
    fit_component: str = "",
    fit_mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[dict]]:
    proximal_xyz = point_array(df, proximal, family)
    joint_xyz = point_array(df, joint, family).copy()
    distal_xyz = point_array(df, distal, family)

    before_error = np.maximum(
        np.abs(np.linalg.norm(joint_xyz - proximal_xyz, axis=1) - proximal_target),
        np.abs(np.linalg.norm(distal_xyz - joint_xyz, axis=1) - distal_target),
    )
    base = distal_xyz - proximal_xyz
    base_length = np.linalg.norm(base, axis=1)
    finite_targets = np.isfinite(proximal_target) and np.isfinite(distal_target) and proximal_target > 0 and distal_target > 0
    feasible = (
        finite_targets
        & np.isfinite(base_length)
        & (base_length > 1e-9)
        & (base_length <= proximal_target + distal_target)
        & (base_length >= abs(proximal_target - distal_target))
    )
    active = mask & feasible & (before_error > tolerance)
    infeasible = mask & ~feasible
    shift = np.zeros_like(joint_xyz)
    guard_rows: list[dict] = []

    if np.any(active):
        safe_length = np.where(base_length > 1e-9, base_length, 1.0)
        base_unit = base / safe_length[:, None]
        along_current = np.sum((joint_xyz - proximal_xyz) * base_unit, axis=1)
        current_line_point = proximal_xyz + along_current[:, None] * base_unit
        current_perp = joint_xyz - current_line_point
        perp_unit = rv_perpendicular_unit(base_unit, current_perp)

        target_along = (proximal_target**2 - distal_target**2 + base_length**2) / (2.0 * safe_length)
        target_perp = np.sqrt(np.maximum(proximal_target**2 - target_along**2, 0.0))
        target_joint = proximal_xyz + target_along[:, None] * base_unit + target_perp[:, None] * perp_unit
        raw_shift = (target_joint - joint_xyz) * strength
        clipped_shift = rv_clip_xyz_shift(raw_shift, xy_cap, z_cap)
        shift[active] = clipped_shift[active]
        candidate_joint_xyz = joint_xyz + shift
        if guard_cfg is not None and source_family is not None:
            candidate_joint_xyz, _, guard_rows = rv_apply_soft_bend_side_guard(
                df,
                source_family,
                family,
                proximal,
                joint,
                distal,
                candidate_joint_xyz,
                active,
                guard_cfg,
                fit_component or joint,
                mask if fit_mask is None else fit_mask,
            )
            shift = candidate_joint_xyz - joint_xyz
        joint_xyz = candidate_joint_xyz
        set_point(df, joint, family, joint_xyz)
    return shift, active, infeasible, guard_rows


def rv_apply_pair_bounded_projection(
    df: pd.DataFrame,
    left: str,
    right: str,
    family: str,
    target_length: float,
    mask: np.ndarray,
    strength: float,
    xy_cap: float,
    z_cap: float,
    tolerance: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    left_xyz = point_array(df, left, family).copy()
    right_xyz = point_array(df, right, family).copy()
    vector = right_xyz - left_xyz
    current_length = np.linalg.norm(vector, axis=1)
    finite_target = np.isfinite(target_length) and target_length > 0
    feasible = finite_target & np.isfinite(current_length) & (current_length > 1e-9)
    residual = np.abs(current_length - target_length)
    active = mask & feasible & (residual > tolerance)
    infeasible = mask & ~feasible
    left_shift = np.zeros_like(left_xyz)
    right_shift = np.zeros_like(right_xyz)

    if np.any(active):
        safe_length = np.where(current_length > 1e-9, current_length, 1.0)
        unit = vector / safe_length[:, None]
        center = 0.5 * (left_xyz + right_xyz)
        target_left = center - 0.5 * target_length * unit
        target_right = center + 0.5 * target_length * unit
        left_shift = rv_clip_xyz_shift((target_left - left_xyz) * strength, xy_cap, z_cap)
        right_shift = rv_clip_xyz_shift((target_right - right_xyz) * strength, xy_cap, z_cap)
        left_shift[~active] = 0.0
        right_shift[~active] = 0.0
        set_point(df, left, family, left_xyz + left_shift)
        set_point(df, right, family, right_xyz + right_shift)
    return left_shift, right_shift, active, infeasible


def rv_framewise_max_delta(df: pd.DataFrame, landmarks: list[str], source_family: str, target_family: str) -> tuple[np.ndarray, np.ndarray]:
    xy_deltas = []
    z_deltas = []
    for landmark in landmarks:
        source_xyz = point_array(df, landmark, source_family)
        target_xyz = point_array(df, landmark, target_family)
        xy_deltas.append(np.linalg.norm(target_xyz[:, :2] - source_xyz[:, :2], axis=1))
        z_deltas.append(np.abs(target_xyz[:, 2] - source_xyz[:, 2]))
    if not xy_deltas:
        return np.zeros(len(df)), np.zeros(len(df))
    return np.nanmax(np.vstack(xy_deltas), axis=0), np.nanmax(np.vstack(z_deltas), axis=0)


def rv_trimmed_median_values(values: np.ndarray, trim_quantile: float = 0.10) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return float("nan")
    trim_q = float(np.clip(trim_quantile, 0.0, 0.45))
    if trim_q > 0.0 and len(finite) >= 4:
        lo, hi = np.quantile(finite, [trim_q, 1.0 - trim_q])
        trimmed = finite[(finite >= lo) & (finite <= hi)]
        if len(trimmed) > 0:
            finite = trimmed
    return rv_safe_median(finite)


def rv_support_side_score(
    df: pd.DataFrame,
    side: str,
    suffixes: list[str],
    source_family: str,
    reference_mask: np.ndarray,
) -> dict:
    landmarks = [f"{side}_{suffix}" for suffix in suffixes]
    visibility_values = []
    jump_values = []
    for landmark in landmarks:
        if not rv_family_has_coordinates(df, source_family, [landmark]):
            continue
        visibility = landmark_visibility(df, landmark)
        if len(visibility) == len(df) and np.any(reference_mask):
            visibility_values.append(visibility[reference_mask])
        ref_xyz = point_array(df, landmark, source_family)[reference_mask]
        if len(ref_xyz) >= 2:
            jumps = np.linalg.norm(np.diff(ref_xyz[:, [0, 2]], axis=0), axis=1)
            jump_values.append(jumps[np.isfinite(jumps)])
    visibility_array = np.concatenate(visibility_values) if visibility_values else np.array([], dtype=float)
    jump_array = np.concatenate(jump_values) if jump_values else np.array([], dtype=float)
    return {
        "side": side,
        "median_visibility": rv_safe_median(visibility_array),
        "p95_reference_xz_jump": rv_safe_percentile(jump_array, 95),
        "available_landmarks": ",".join(
            landmark for landmark in landmarks if rv_family_has_coordinates(df, source_family, [landmark])
        ),
    }


def rv_select_trusted_support_side(
    df: pd.DataFrame,
    source_family: str,
    guard_cfg: dict,
    reference_mask: np.ndarray,
) -> tuple[str, pd.DataFrame]:
    suffixes = [str(item) for item in guard_cfg.get("support_landmark_suffixes", ["ankle", "heel", "foot_index"])]
    candidate_sides = [str(item) for item in guard_cfg.get("candidate_sides", ["left", "right"])]
    fallback_side = str(guard_cfg.get("fallback_trusted_side", candidate_sides[0] if candidate_sides else "left"))
    score_rows = [rv_support_side_score(df, side, suffixes, source_family, reference_mask) for side in candidate_sides]
    best_side = fallback_side
    best_key = (-np.inf, np.inf)
    for row in score_rows:
        visibility = row["median_visibility"]
        jump = row["p95_reference_xz_jump"]
        visibility_key = visibility if np.isfinite(visibility) else -np.inf
        jump_key = jump if np.isfinite(jump) else np.inf
        key = (visibility_key, -jump_key)
        if key > best_key:
            best_key = key
            best_side = str(row["side"])
    score_df = pd.DataFrame(score_rows)
    if not score_df.empty:
        score_df["selected_trusted_side"] = score_df["side"].eq(best_side)
    return best_side, score_df


def rv_apply_visible_support_mirrored_anchor_prior(
    df: pd.DataFrame,
    target_family: str,
    source_family: str,
    guard_cfg: dict,
    fit_mask: np.ndarray,
    reference_mask: np.ndarray,
) -> tuple[pd.DataFrame, set[str]]:
    if not bool(guard_cfg.get("enabled", False)):
        return pd.DataFrame(), set()
    gate_ok, gate_source = rv_support_surface_gate(guard_cfg)
    if not gate_ok:
        return pd.DataFrame([{
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "skipped",
            "gate_source": gate_source,
            "exercise_id": EXERCISE_ID,
        }]), set()

    source_family = str(guard_cfg.get("source_family", source_family))
    ref_mask = reference_mask.copy() if str(guard_cfg.get("reference_source", "ready_window_median")) == "ready_window_median" else fit_mask.copy()
    if not np.any(ref_mask):
        ref_mask = fit_mask.copy()
    min_reference_frames = int(guard_cfg.get("min_reference_frames", 8))
    if int(np.sum(ref_mask)) < min_reference_frames:
        return pd.DataFrame([{
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "skipped_insufficient_reference_frames",
            "reference_frames": int(np.sum(ref_mask)),
            "min_reference_frames": min_reference_frames,
        }]), set()

    trusted_side, side_score_df = rv_select_trusted_support_side(df, source_family, guard_cfg, ref_mask)
    target_side_cfg = str(guard_cfg.get("target_side", "auto_opposite"))
    if target_side_cfg in {"auto", "auto_opposite", "opposite"}:
        target_side = "right" if trusted_side == "left" else "left"
    else:
        target_side = target_side_cfg
    suffixes = [str(item) for item in guard_cfg.get("support_landmark_suffixes", ["ankle", "heel", "foot_index"])]
    primary_suffixes = {str(item) for item in guard_cfg.get("primary_suffixes", ["ankle"])}
    body_landmarks = [str(item) for item in guard_cfg.get("body_center_landmarks", ["left_hip", "right_hip"])]
    axes = [str(item) for item in guard_cfg.get("axes", ["z"])]
    axis_indices = [rv_axis_index(axis) for axis in axes]
    strength = float(np.clip(float(guard_cfg.get("strength", 1.0)), 0.0, 1.0))
    max_z_shift = float(guard_cfg.get("max_z_shift_ratio", 0.12))
    tolerance = float(guard_cfg.get("tolerance_ratio", 0.005))
    report_only = bool(guard_cfg.get("report_only", False))
    width_guard_cfg = guard_cfg.get("support_width_no_worsen", {}) if isinstance(guard_cfg, dict) else {}
    width_guard_enabled = bool(width_guard_cfg.get("enabled", False))
    width_trim_quantile = float(width_guard_cfg.get("trim_quantile", guard_cfg.get("trim_quantile", 0.10)))
    width_tolerance = float(width_guard_cfg.get("tolerance_ratio", tolerance))

    if not body_landmarks or not rv_family_has_coordinates(df, source_family, body_landmarks):
        return pd.DataFrame([{
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "missing_body_frame",
            "source_family": source_family,
        }]), set()

    body_stack = np.stack([point_array(df, landmark, source_family) for landmark in body_landmarks], axis=1)
    body_center = np.nanmean(body_stack, axis=1)
    body_center_ref = np.nanmedian(body_center[ref_mask], axis=0)
    if not np.all(np.isfinite(body_center_ref)):
        return pd.DataFrame([{
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "invalid_body_frame",
            "source_family": source_family,
        }]), set()

    score_lookup = side_score_df.set_index("side").to_dict("index") if not side_score_df.empty else {}
    trusted_score = score_lookup.get(trusted_side, {})
    opposite_score = score_lookup.get(target_side, {})
    moved_landmarks: set[str] = set()
    rows = []
    for suffix in suffixes:
        trusted_landmark = f"{trusted_side}_{suffix}"
        target_landmark = f"{target_side}_{suffix}"
        if not rv_family_has_coordinates(df, source_family, [trusted_landmark]) or not rv_family_has_coordinates(df, target_family, [target_landmark]):
            continue
        trusted_ref = np.nanmedian(point_array(df, trusted_landmark, source_family)[ref_mask], axis=0)
        if not np.all(np.isfinite(trusted_ref)):
            continue
        mirrored_anchor = body_center_ref.copy()
        trusted_delta = trusted_ref - body_center_ref
        mirrored_anchor[0] = body_center_ref[0] - trusted_delta[0]
        mirrored_anchor[1] = body_center_ref[1] + trusted_delta[1]
        mirrored_anchor[2] = body_center_ref[2] + trusted_delta[2]

        target_xyz_before = point_array(df, target_landmark, target_family)
        shift = np.zeros_like(target_xyz_before)
        for axis_i in axis_indices:
            cap = max_z_shift if axis_i == 2 else float(guard_cfg.get("max_xy_shift_ratio", 0.0))
            residual = mirrored_anchor[axis_i] - target_xyz_before[:, axis_i]
            active = fit_mask & np.isfinite(residual) & (np.abs(residual) > tolerance)
            axis_shift = np.clip(residual * strength, -cap, cap)
            axis_shift[~active] = 0.0
            shift[:, axis_i] = axis_shift
        target_xyz_after = target_xyz_before + shift
        support_width_target = np.nan
        width_residual_before = np.full(len(df), np.nan)
        width_residual_after = np.full(len(df), np.nan)
        width_rejected = np.zeros(len(df), dtype=bool)
        if width_guard_enabled and suffix in primary_suffixes:
            left_landmark = f"left_{suffix}"
            right_landmark = f"right_{suffix}"
            pair_landmarks = [left_landmark, right_landmark]
            if rv_family_has_coordinates(df, source_family, pair_landmarks) and rv_family_has_coordinates(df, target_family, pair_landmarks):
                source_left = point_array(df, left_landmark, source_family)
                source_right = point_array(df, right_landmark, source_family)
                source_width = np.linalg.norm(source_right[:, [0, 2]] - source_left[:, [0, 2]], axis=1)
                support_width_target = rv_trimmed_median_values(source_width[ref_mask], width_trim_quantile)
                if np.isfinite(support_width_target):
                    left_before = point_array(df, left_landmark, target_family)
                    right_before = point_array(df, right_landmark, target_family)
                    left_after = left_before.copy()
                    right_after = right_before.copy()
                    if target_landmark == left_landmark:
                        left_after = target_xyz_after
                    elif target_landmark == right_landmark:
                        right_after = target_xyz_after
                    width_before = np.linalg.norm(right_before[:, [0, 2]] - left_before[:, [0, 2]], axis=1)
                    width_after = np.linalg.norm(right_after[:, [0, 2]] - left_after[:, [0, 2]], axis=1)
                    width_residual_before = np.abs(width_before - support_width_target)
                    width_residual_after = np.abs(width_after - support_width_target)
                    width_rejected = fit_mask & (width_residual_after > width_residual_before + width_tolerance)
                    shift[width_rejected] = 0.0
                    target_xyz_after = target_xyz_before + shift
                    if target_landmark == left_landmark:
                        left_after = target_xyz_after
                    elif target_landmark == right_landmark:
                        right_after = target_xyz_after
                    width_after = np.linalg.norm(right_after[:, [0, 2]] - left_after[:, [0, 2]], axis=1)
                    width_residual_after = np.abs(width_after - support_width_target)
        if not report_only:
            set_point(df, target_landmark, target_family, target_xyz_after)
        active_shift = np.linalg.norm(shift[:, axis_indices], axis=1) > 1e-9 if axis_indices else np.zeros(len(df), dtype=bool)
        if np.any(active_shift & fit_mask) and not report_only:
            moved_landmarks.add(target_landmark)

        z_before = mirrored_anchor[2] - target_xyz_before[:, 2]
        z_after = mirrored_anchor[2] - target_xyz_after[:, 2]
        rows.append({
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "report_only" if report_only else "applied",
            "gate_source": gate_source,
            "exercise_id": EXERCISE_ID,
            "target_family": target_family,
            "source_family": source_family,
            "trusted_side": trusted_side,
            "target_side": target_side,
            "trusted_landmark": trusted_landmark,
            "target_landmark": target_landmark,
            "primary_landmark": suffix in primary_suffixes,
            "trusted_median_visibility": trusted_score.get("median_visibility", np.nan),
            "target_median_visibility": opposite_score.get("median_visibility", np.nan),
            "trusted_p95_reference_xz_jump": trusted_score.get("p95_reference_xz_jump", np.nan),
            "target_p95_reference_xz_jump": opposite_score.get("p95_reference_xz_jump", np.nan),
            "anchor_x": mirrored_anchor[0],
            "anchor_y": mirrored_anchor[1],
            "anchor_z": mirrored_anchor[2],
            "median_abs_z_residual_before": rv_safe_median(np.abs(z_before[fit_mask])),
            "median_abs_z_residual_after": rv_safe_median(np.abs(z_after[fit_mask])),
            "support_width_target": support_width_target,
            "median_support_width_residual_before": rv_safe_median(width_residual_before[fit_mask]),
            "median_support_width_residual_after": rv_safe_median(width_residual_after[fit_mask]),
            "support_width_rejected_frame_ratio": float(np.nanmean(width_rejected[fit_mask])) if np.any(fit_mask) else np.nan,
            "support_width_rejected_frames": rv_frame_label_list(df, width_rejected & fit_mask),
            "active_frame_ratio": float(np.nanmean(active_shift[fit_mask])) if np.any(fit_mask) else np.nan,
            "median_abs_z_shift": rv_safe_median(np.abs(shift[fit_mask, 2])),
            "p95_abs_z_shift": rv_safe_percentile(np.abs(shift[fit_mask, 2]), 95),
            "max_abs_z_shift": float(np.nanmax(np.abs(shift[fit_mask, 2]))) if np.any(fit_mask) else np.nan,
            "applied_frames": rv_frame_label_list(df, active_shift & fit_mask),
            "used_for_features_or_scores": bool(guard_cfg.get("reporting", {}).get("used_for_features_or_scores", False)),
            "confidence_when_applied": guard_cfg.get("reporting", {}).get("confidence_when_applied", "very_low"),
        })

    if not rows:
        return pd.DataFrame([{
            "fit_component": "visible_support_anchor",
            "type": "visible_support_mirrored_anchor_prior",
            "status": "no_target_landmarks",
            "trusted_side": trusted_side,
            "target_side": target_side,
        }]), moved_landmarks
    return pd.DataFrame(rows), moved_landmarks

bounded_rv_summary_df = pd.DataFrame()
bounded_rv_segment_report_df = pd.DataFrame()
bounded_rv_burden_df = pd.DataFrame()
bounded_rv_support_report_df = pd.DataFrame()
bounded_rv_support_surface_report_df = pd.DataFrame()
bounded_rv_visible_support_anchor_report_df = pd.DataFrame()

if not BOUNDED_RV_ENABLED:
    display(Markdown("`rv_skeleton_fit_bounded_xy` is disabled in the run profile."))
else:
    bounded_required_landmarks = sorted({
        item
        for chain in RV_FIT_CONFIG.get("chains", [])
        for item in [str(chain["proximal"]), str(chain["joint"]), str(chain["distal"])]
    } | {
        item
        for width_cfg in RV_FIT_CONFIG.get("width_segments", [])
        for item in [str(width_cfg["left"]), str(width_cfg["right"])]
    })
    bounded_start_family = BOUNDED_RV_BASELINE_FAMILY if rv_family_has_coordinates(corridor_df, BOUNDED_RV_BASELINE_FAMILY, bounded_required_landmarks) else BOUNDED_RV_SOURCE_FAMILY
    corridor_df = rv_copy_coordinate_family(corridor_df, bounded_start_family, BOUNDED_RV_FAMILY)

    bounded_targets = rv_targets if "rv_targets" in globals() else rv_target_lengths_from_matrix()
    bounded_rep_mask = corridor_df["rep_id"].astype("Int64").eq(BOUNDED_RV_REP_ID).fillna(False).to_numpy(dtype=bool)
    bounded_fit_mask = bounded_rep_mask.copy()
    if bool(BOUNDED_RV_CONFIG.get("include_ready_window", True)):
        bounded_fit_mask |= mask_ref
    if "use_for_analysis" in corridor_df.columns:
        bounded_fit_mask &= (corridor_df["use_for_analysis"].astype(bool).to_numpy() | mask_ref)
    if not np.any(bounded_fit_mask):
        bounded_fit_mask = compact_rep_mask.copy()

    bounded_solver_cfg = BOUNDED_RV_CONFIG.get("solver", {})
    bounded_strength = float(np.clip(float(bounded_solver_cfg.get("strength", 0.70)), 0.0, 1.0))
    bounded_tolerance = float(bounded_solver_cfg.get("tolerance_ratio", 0.015))
    bounded_joint_xy_cap = float(bounded_solver_cfg.get("max_joint_xy_shift_ratio", 0.025))
    bounded_joint_z_cap = float(bounded_solver_cfg.get("max_joint_z_shift_ratio", 0.45))
    bounded_width_xy_cap = float(bounded_solver_cfg.get("max_width_xy_shift_ratio", 0.015))
    bounded_width_z_cap = float(bounded_solver_cfg.get("max_width_z_shift_ratio", 0.25))
    bounded_bend_guard_cfg = bounded_solver_cfg.get("bend_side_guard", {})

    bounded_segment_rows = []
    bounded_burden_rows = []
    bounded_active_landmarks: set[str] = set(bounded_required_landmarks)

    for width_cfg in RV_FIT_CONFIG.get("width_segments", []):
        width_id = str(width_cfg.get("id", "width"))
        left = str(width_cfg["left"])
        right = str(width_cfg["right"])
        segment = str(width_cfg["segment"])
        memory_id = str(width_cfg.get("memory_id", width_id))
        target_length = rv_target_length(segment, memory_id)
        for family in [BOUNDED_RV_SOURCE_FAMILY, bounded_start_family]:
            if rv_family_has_coordinates(corridor_df, family, [left, right]):
                bounded_segment_rows.append(rv_chain_report_row(width_id, segment, left, right, family, target_length, bounded_fit_mask))
        left_before = point_array(corridor_df, left, BOUNDED_RV_FAMILY).copy()
        right_before = point_array(corridor_df, right, BOUNDED_RV_FAMILY).copy()
        left_shift, right_shift, active, infeasible = rv_apply_pair_bounded_projection(
            corridor_df,
            left,
            right,
            BOUNDED_RV_FAMILY,
            target_length,
            bounded_fit_mask,
            bounded_strength,
            bounded_width_xy_cap,
            bounded_width_z_cap,
            bounded_tolerance,
        )
        width_bend_guarded = np.zeros(len(corridor_df), dtype=bool)
        width_bend_guard_rows: list[dict] = []
        left_after = point_array(corridor_df, left, BOUNDED_RV_FAMILY).copy()
        right_after = point_array(corridor_df, right, BOUNDED_RV_FAMILY).copy()
        for guard_chain in RV_FIT_CONFIG.get("chains", []):
            guard_chain_id = str(guard_chain.get("id", "chain"))
            guard_proximal = str(guard_chain["proximal"])
            guard_joint = str(guard_chain["joint"])
            guard_distal = str(guard_chain["distal"])
            candidate_proximal_xyz = None
            candidate_distal_xyz = None
            if guard_proximal == left:
                candidate_proximal_xyz = left_after
            elif guard_proximal == right:
                candidate_proximal_xyz = right_after
            if guard_distal == left:
                candidate_distal_xyz = left_after
            elif guard_distal == right:
                candidate_distal_xyz = right_after
            if candidate_proximal_xyz is None and candidate_distal_xyz is None:
                continue
            _, chain_guarded, chain_guard_rows = rv_apply_soft_bend_side_guard(
                corridor_df,
                BOUNDED_RV_SOURCE_FAMILY,
                BOUNDED_RV_FAMILY,
                guard_proximal,
                guard_joint,
                guard_distal,
                point_array(corridor_df, guard_joint, BOUNDED_RV_FAMILY),
                active,
                bounded_bend_guard_cfg,
                f"{width_id}:{guard_chain_id}",
                bounded_fit_mask,
                candidate_proximal_xyz=candidate_proximal_xyz,
                candidate_distal_xyz=candidate_distal_xyz,
            )
            width_bend_guard_rows.extend(chain_guard_rows)
            width_bend_guarded |= chain_guarded
            if candidate_proximal_xyz is left_after or candidate_distal_xyz is left_after:
                left_after[chain_guarded] = left_before[chain_guarded]
            if candidate_proximal_xyz is right_after or candidate_distal_xyz is right_after:
                right_after[chain_guarded] = right_before[chain_guarded]
        if np.any(width_bend_guarded):
            set_point(corridor_df, left, BOUNDED_RV_FAMILY, left_after)
            set_point(corridor_df, right, BOUNDED_RV_FAMILY, right_after)
            left_shift = left_after - left_before
            right_shift = right_after - right_before
        bounded_burden_rows.extend(width_bend_guard_rows)
        max_xy_shift = np.maximum(np.linalg.norm(left_shift[:, :2], axis=1), np.linalg.norm(right_shift[:, :2], axis=1))
        max_z_shift = np.maximum(np.abs(left_shift[:, 2]), np.abs(right_shift[:, 2]))
        bounded_burden_rows.append({
            "fit_component": width_id,
            "type": "pair_width_bounded_xyz_projection",
            "joint": f"{left}/{right}",
            "active_frame_ratio": float(np.nanmean(active[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "infeasible_frame_ratio": float(np.nanmean(infeasible[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "median_xy_shift": rv_safe_median(max_xy_shift[bounded_fit_mask]),
            "p95_xy_shift": rv_safe_percentile(max_xy_shift[bounded_fit_mask], 95),
            "max_xy_shift": float(np.nanmax(max_xy_shift[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "median_z_shift": rv_safe_median(max_z_shift[bounded_fit_mask]),
            "p95_z_shift": rv_safe_percentile(max_z_shift[bounded_fit_mask], 95),
            "max_z_shift": float(np.nanmax(max_z_shift[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "bend_guarded_frame_ratio": float(np.nanmean(width_bend_guarded[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "bend_guarded_frames": rv_frame_label_list(corridor_df, width_bend_guarded & bounded_fit_mask),
        })
        bounded_segment_rows.append(rv_chain_report_row(width_id, segment, left, right, BOUNDED_RV_FAMILY, target_length, bounded_fit_mask))

    bounded_rv_support_surface_report_df, bounded_support_surface_landmarks = rv_apply_support_surface_height_guard(
        corridor_df,
        BOUNDED_RV_FAMILY,
        BOUNDED_RV_SOURCE_FAMILY,
        bounded_solver_cfg.get("support_surface_height_guard", {}),
        bounded_fit_mask,
        mask_ref,
    )
    bounded_active_landmarks.update(bounded_support_surface_landmarks)

    bounded_rv_visible_support_anchor_report_df, bounded_visible_support_anchor_landmarks = rv_apply_visible_support_mirrored_anchor_prior(
        corridor_df,
        BOUNDED_RV_FAMILY,
        BOUNDED_RV_SOURCE_FAMILY,
        bounded_solver_cfg.get("visible_support_mirrored_anchor_prior", {}),
        bounded_fit_mask,
        mask_ref,
    )
    bounded_active_landmarks.update(bounded_visible_support_anchor_landmarks)

    for chain in RV_FIT_CONFIG.get("chains", []):
        chain_id = str(chain.get("id", "chain"))
        proximal = str(chain["proximal"])
        joint = str(chain["joint"])
        distal = str(chain["distal"])
        proximal_segment = str(chain["proximal_segment"])
        distal_segment = str(chain["distal_segment"])
        proximal_memory_id = str(chain.get("proximal_memory_id", rv_default_memory_id(chain_id, proximal_segment)))
        distal_memory_id = str(chain.get("distal_memory_id", rv_default_memory_id(chain_id, distal_segment)))
        proximal_target = rv_target_length(proximal_segment, proximal_memory_id)
        distal_target = rv_target_length(distal_segment, distal_memory_id)
        for family in [BOUNDED_RV_SOURCE_FAMILY, bounded_start_family]:
            if rv_family_has_coordinates(corridor_df, family, [proximal, joint, distal]):
                bounded_segment_rows.append(rv_chain_report_row(chain_id, proximal_segment, proximal, joint, family, proximal_target, bounded_fit_mask))
                bounded_segment_rows.append(rv_chain_report_row(chain_id, distal_segment, joint, distal, family, distal_target, bounded_fit_mask))
        joint_shift, active, infeasible, bend_guard_rows = rv_apply_two_link_bounded_projection(
            corridor_df,
            proximal,
            joint,
            distal,
            BOUNDED_RV_FAMILY,
            proximal_target,
            distal_target,
            bounded_fit_mask,
            bounded_strength,
            bounded_joint_xy_cap,
            bounded_joint_z_cap,
            bounded_tolerance,
            bounded_bend_guard_cfg,
            BOUNDED_RV_SOURCE_FAMILY,
            chain_id,
            bounded_fit_mask,
        )
        bounded_burden_rows.extend(bend_guard_rows)
        xy_shift = np.linalg.norm(joint_shift[:, :2], axis=1)
        z_shift = np.abs(joint_shift[:, 2])
        bounded_burden_rows.append({
            "fit_component": chain_id,
            "type": "two_link_bounded_xyz_projection",
            "joint": joint,
            "active_frame_ratio": float(np.nanmean(active[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "infeasible_frame_ratio": float(np.nanmean(infeasible[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "median_xy_shift": rv_safe_median(xy_shift[bounded_fit_mask]),
            "p95_xy_shift": rv_safe_percentile(xy_shift[bounded_fit_mask], 95),
            "max_xy_shift": float(np.nanmax(xy_shift[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
            "median_z_shift": rv_safe_median(z_shift[bounded_fit_mask]),
            "p95_z_shift": rv_safe_percentile(z_shift[bounded_fit_mask], 95),
            "max_z_shift": float(np.nanmax(z_shift[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
        })
        bounded_segment_rows.append(rv_chain_report_row(chain_id, proximal_segment, proximal, joint, BOUNDED_RV_FAMILY, proximal_target, bounded_fit_mask))
        bounded_segment_rows.append(rv_chain_report_row(chain_id, distal_segment, joint, distal, BOUNDED_RV_FAMILY, distal_target, bounded_fit_mask))

    bounded_rv_segment_report_df = pd.DataFrame(bounded_segment_rows)
    bounded_rv_burden_df = pd.DataFrame(bounded_burden_rows)
    bounded_xy_delta, bounded_z_delta = rv_framewise_max_delta(corridor_df, sorted(bounded_active_landmarks), BOUNDED_RV_SOURCE_FAMILY, BOUNDED_RV_FAMILY)
    strict_xy_delta, strict_z_delta = rv_framewise_max_delta(corridor_df, sorted(bounded_active_landmarks), BOUNDED_RV_SOURCE_FAMILY, BOUNDED_RV_BASELINE_FAMILY) if rv_family_has_coordinates(corridor_df, BOUNDED_RV_BASELINE_FAMILY, bounded_required_landmarks) else (np.zeros(len(corridor_df)), np.zeros(len(corridor_df)))

    support_cfg = RV_FIT_CONFIG.get("support_residual", {})
    if bool(support_cfg.get("enabled", True)):
        pair = list(support_cfg.get("pair", ["left_ankle", "right_ankle"]))
        trim_q = float(np.clip(float(support_cfg.get("trim_quantile", 0.10)), 0.0, 0.45))
        source_width = np.linalg.norm(
            point_array(corridor_df, pair[1], BOUNDED_RV_SOURCE_FAMILY)[:, [0, 2]] - point_array(corridor_df, pair[0], BOUNDED_RV_SOURCE_FAMILY)[:, [0, 2]],
            axis=1,
        )
        bounded_width = np.linalg.norm(
            point_array(corridor_df, pair[1], BOUNDED_RV_FAMILY)[:, [0, 2]] - point_array(corridor_df, pair[0], BOUNDED_RV_FAMILY)[:, [0, 2]],
            axis=1,
        )
        support_target = trimmed_median(source_width[mask_ref], trim_q) if np.any(mask_ref) else trimmed_median(source_width[bounded_fit_mask], trim_q)
        bounded_rv_support_report_df = pd.DataFrame([{
            "support_pair": "->".join(pair),
            "target_source": support_cfg.get("target_source", "ready_window_median"),
            "target_width": support_target,
            "median_width_norm": rv_safe_median(source_width[bounded_fit_mask]),
            f"median_width_{BOUNDED_RV_FAMILY}": rv_safe_median(bounded_width[bounded_fit_mask]),
            "median_abs_residual_norm": rv_safe_median(np.abs(source_width[bounded_fit_mask] - support_target)),
            f"median_abs_residual_{BOUNDED_RV_FAMILY}": rv_safe_median(np.abs(bounded_width[bounded_fit_mask] - support_target)),
        }])

    source_rows = bounded_rv_segment_report_df[bounded_rv_segment_report_df["family"].eq(BOUNDED_RV_SOURCE_FAMILY)]
    baseline_rows = bounded_rv_segment_report_df[bounded_rv_segment_report_df["family"].eq(bounded_start_family)]
    bounded_rows = bounded_rv_segment_report_df[bounded_rv_segment_report_df["family"].eq(BOUNDED_RV_FAMILY)]
    bounded_rv_summary_df = pd.DataFrame([{
        "candidate_family": BOUNDED_RV_FAMILY,
        "source_family": BOUNDED_RV_SOURCE_FAMILY,
        "baseline_family": bounded_start_family,
        "review_rep_id": BOUNDED_RV_REP_ID,
        "fit_frames": int(bounded_fit_mask.sum()),
        "active_landmark_count": len(bounded_active_landmarks),
        "median_recording_xy_residual": rv_safe_median(bounded_xy_delta[bounded_fit_mask]),
        "p95_recording_xy_residual": rv_safe_percentile(bounded_xy_delta[bounded_fit_mask], 95),
        "max_recording_xy_residual": float(np.nanmax(bounded_xy_delta[bounded_fit_mask])) if np.any(bounded_fit_mask) else np.nan,
        "median_depth_shift": rv_safe_median(bounded_z_delta[bounded_fit_mask]),
        "p95_depth_shift": rv_safe_percentile(bounded_z_delta[bounded_fit_mask], 95),
        "median_segment_residual_source": rv_safe_median(source_rows["median_abs_residual"].to_numpy(float)),
        "median_segment_residual_baseline": rv_safe_median(baseline_rows["median_abs_residual"].to_numpy(float)),
        "median_segment_residual_bounded": rv_safe_median(bounded_rows["median_abs_residual"].to_numpy(float)),
        "used_for_features_or_scores": bool(BOUNDED_RV_CONFIG.get("reporting", {}).get("used_for_features_or_scores", False)),
        "confidence_when_applied": BOUNDED_RV_CONFIG.get("reporting", {}).get("confidence_when_applied", "very_low"),
        "exclude_from_recording_view_only_features": bool(BOUNDED_RV_CONFIG.get("reporting", {}).get("exclude_from_recording_view_only_features", True)),
    }])

    display(Markdown("### bounded recording-view residual burden ledger"))
    display(bounded_rv_summary_df)
    display(bounded_rv_burden_df)
    if not bounded_rv_support_surface_report_df.empty:
        display(Markdown("### support-surface height guard burden"))
        display(bounded_rv_support_surface_report_df)
    if not bounded_rv_visible_support_anchor_report_df.empty:
        display(Markdown("### visible-support mirrored anchor burden"))
        display(bounded_rv_visible_support_anchor_report_df)
    display(bounded_rv_support_report_df)

    bounded_plot_df = corridor_df.loc[bounded_rep_mask].copy() if np.any(bounded_rep_mask) else corridor_df.loc[bounded_fit_mask].copy()
    bounded_frames = bounded_plot_df["frame"].to_numpy(int)
    fig_bounded_residual = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("max recording-view x/y residual", "max model-depth z shift"),
    )
    plot_index = bounded_plot_df.index.to_numpy()
    fig_bounded_residual.add_trace(go.Scatter(x=bounded_frames, y=strict_xy_delta[plot_index], mode="lines", name=f"strict {BOUNDED_RV_BASELINE_FAMILY} x/y", line=dict(color="rgba(220,40,80,0.60)", dash="dot")), row=1, col=1)
    fig_bounded_residual.add_trace(go.Scatter(x=bounded_frames, y=bounded_xy_delta[plot_index], mode="lines", name=f"bounded {BOUNDED_RV_FAMILY} x/y", line=dict(color="rgba(0,170,170,0.95)", width=3)), row=1, col=1)
    fig_bounded_residual.add_trace(go.Scatter(x=bounded_frames, y=strict_z_delta[plot_index], mode="lines", name=f"strict {BOUNDED_RV_BASELINE_FAMILY} z", line=dict(color="rgba(220,40,80,0.60)", dash="dot")), row=2, col=1)
    fig_bounded_residual.add_trace(go.Scatter(x=bounded_frames, y=bounded_z_delta[plot_index], mode="lines", name=f"bounded {BOUNDED_RV_FAMILY} z", line=dict(color="rgba(0,170,170,0.95)", width=3)), row=2, col=1)
    fig_bounded_residual.update_layout(title="Rep 1 residual burden: strict vs bounded recording-view skeleton placement", height=620, width=1100)
    fig_bounded_residual.update_yaxes(title_text="torso-length ratio")
    fig_bounded_residual.update_xaxes(title_text="source frame", row=2, col=1)
    if review_show_visual("bounded_residual"):
        fig_bounded_residual.show()

    fig_bounded_lengths = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("thigh length", "shank length"))
    for label, a, b, segment_name in rv_length_plot_segments:
        row_idx = 1 if segment_name == "thigh" else 2
        target_length = bounded_targets.get(segment_name, {}).get("target", float("nan"))
        fig_bounded_lengths.add_trace(go.Scatter(x=bounded_frames, y=segment_length_series(bounded_plot_df, a, b, BOUNDED_RV_SOURCE_FAMILY), mode="lines", name=f"norm {label}", line=dict(color="rgba(0,0,0,0.55)", dash="dot")), row=row_idx, col=1)
        if rv_family_has_coordinates(bounded_plot_df, BOUNDED_RV_BASELINE_FAMILY, [a, b]):
            fig_bounded_lengths.add_trace(go.Scatter(x=bounded_frames, y=segment_length_series(bounded_plot_df, a, b, BOUNDED_RV_BASELINE_FAMILY), mode="lines", name=f"strict {label}", line=dict(color="rgba(220,40,80,0.65)", width=2)), row=row_idx, col=1)
        fig_bounded_lengths.add_trace(go.Scatter(x=bounded_frames, y=segment_length_series(bounded_plot_df, a, b, BOUNDED_RV_FAMILY), mode="lines", name=f"bounded {label}", line=dict(color="rgba(0,170,170,0.95)", width=3)), row=row_idx, col=1)
        if np.isfinite(target_length):
            fig_bounded_lengths.add_trace(go.Scatter(x=bounded_frames, y=np.full(len(bounded_frames), target_length), mode="lines", name=f"target {segment_name}", line=dict(color="rgba(160,160,160,0.55)", dash="dash"), showlegend=(label.startswith("left"))), row=row_idx, col=1)
    fig_bounded_lengths.update_layout(title="Rep 1 segment lengths: norm vs strict vs bounded recording-view skeleton placement", height=680, width=1100)
    fig_bounded_lengths.update_yaxes(title_text="torso-length ratio")
    fig_bounded_lengths.update_xaxes(title_text="source frame", row=2, col=1)
    if review_show_visual("bounded_lengths"):
        fig_bounded_lengths.show()

    bounded_playback_anchor_cfg = dict(REVIEW_DISPLAY_CONFIG.get("playback_anchor", {}))


    def bounded_playback_anchor_family() -> str:
        family = str(bounded_playback_anchor_cfg.get("family", "bounded_candidate"))
        aliases = {
            "bounded_candidate": BOUNDED_RV_FAMILY,
            "source": BOUNDED_RV_SOURCE_FAMILY,
            "baseline": BOUNDED_RV_BASELINE_FAMILY,
        }
        return aliases.get(family, family)


    def bounded_display_anchor(row: pd.Series) -> np.ndarray:
        if not bool(bounded_playback_anchor_cfg.get("enabled", True)):
            return np.zeros(3, dtype=float)
        family = bounded_playback_anchor_family()
        landmarks = [str(item) for item in bounded_playback_anchor_cfg.get("landmarks", ["left_ankle", "right_ankle"])]
        points = []
        for landmark in landmarks:
            columns = [fcol(landmark, family, axis) for axis in ["x", "y", "z"]]
            if all(column in row.index and np.isfinite(float(row[column])) for column in columns):
                points.append(np.asarray(plot_xyz(row, landmark, family), dtype=float))
        if not points:
            return np.zeros(3, dtype=float)
        anchor = np.nanmedian(np.vstack(points), axis=0)
        if not bool(bounded_playback_anchor_cfg.get("include_height", True)):
            anchor[2] = 0.0
        anchor[~np.isfinite(anchor)] = 0.0
        return anchor


    def bounded_display_xyz(row: pd.Series, landmark: str, family: str) -> tuple[float, float, float]:
        xyz = np.asarray(plot_xyz(row, landmark, family), dtype=float)
        return tuple((xyz - bounded_display_anchor(row)).tolist())


    def bounded_skeleton_traces(row: pd.Series, family: str, label: str, color: str) -> list[go.Scatter3d]:
        px, py, pz = [], [], []
        for landmark in LANDMARKS:
            x, y, z = bounded_display_xyz(row, landmark, family)
            px.append(x)
            py.append(y)
            pz.append(z)

        lx, ly, lz = [], [], []
        for a, b in CONNECTIONS:
            ax, ay, az = bounded_display_xyz(row, a, family)
            bx, by, bz = bounded_display_xyz(row, b, family)
            lx.extend([ax, bx, None])
            ly.extend([ay, by, None])
            lz.extend([az, bz, None])

        return [
            go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=5), name=f"{label} skeleton"),
            go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=3), name=f"{label} joints"),
        ]


    def support_surface_patch_geometry(df: pd.DataFrame, family: str, report_df: pd.DataFrame, guard_cfg: dict) -> dict[str, float] | None:
        if report_df.empty or not bool(guard_cfg.get("show_reference_patch", True)):
            return None
        target_rows = report_df[report_df.get("type", pd.Series(dtype=str)).astype(str).eq("support_surface_height_guard")].copy()
        target_rows = target_rows[target_rows.get("status", pd.Series(dtype=str)).astype(str).isin(["applied", "reported"])]
        if target_rows.empty or "target_height" not in target_rows.columns:
            return None
        contact_suffixes = [str(item) for item in guard_cfg.get("patch_contact_suffixes", ["heel", "foot_index"])]
        contact_rows = target_rows[target_rows["suffix"].astype(str).isin(contact_suffixes)] if "suffix" in target_rows.columns else target_rows
        if contact_rows.empty:
            contact_rows = target_rows
        target_heights = contact_rows["target_height"].to_numpy(float)
        if not np.isfinite(target_heights).any():
            return None

        sides = [str(item) for item in guard_cfg.get("sides", ["left", "right"])]
        suffixes = sorted(set(contact_suffixes + ["ankle"]))
        xs, ys, patch_z_values = [], [], []
        for _, row in df.iterrows():
            anchor = bounded_display_anchor(row)
            patch_z_values.extend((-target_heights - anchor[2]).tolist())
            for side in sides:
                for suffix in suffixes:
                    landmark = f"{side}_{suffix}"
                    if landmark in LANDMARKS and all(fcol(landmark, family, axis) in row.index for axis in ["x", "y", "z"]):
                        x, y, _ = bounded_display_xyz(row, landmark, family)
                        xs.append(x)
                        ys.append(y)
        x_arr = np.asarray(xs, dtype=float)
        y_arr = np.asarray(ys, dtype=float)
        finite = np.isfinite(x_arr) & np.isfinite(y_arr)
        if not np.any(finite):
            return None
        x_min, x_max = float(np.nanmin(x_arr[finite])), float(np.nanmax(x_arr[finite]))
        y_min, y_max = float(np.nanmin(y_arr[finite])), float(np.nanmax(y_arr[finite]))
        pad_x = max(0.06, 0.12 * max(x_max - x_min, 1e-6))
        pad_y = max(0.06, 0.12 * max(y_max - y_min, 1e-6))
        patch_z = rv_safe_median(np.asarray(patch_z_values, dtype=float))
        return {"x0": x_min - pad_x, "x1": x_max + pad_x, "y0": y_min - pad_y, "y1": y_max + pad_y, "z": patch_z}


    def support_surface_patch_traces(geometry: dict[str, float] | None) -> list:
        if not geometry:
            return []
        x0, x1 = geometry["x0"], geometry["x1"]
        y0, y1 = geometry["y0"], geometry["y1"]
        z = geometry["z"]
        return [
            go.Mesh3d(
                x=[x0, x1, x1, x0],
                y=[y0, y0, y1, y1],
                z=[z, z, z, z],
                i=[0, 0],
                j=[1, 2],
                k=[2, 3],
                color="rgb(80,180,120)",
                opacity=0.18,
                name="support-surface reference patch",
                showlegend=True,
                hoverinfo="name",
            ),
            go.Scatter3d(
                x=[x0, x1, x1, x0, x0],
                y=[y0, y0, y1, y1, y0],
                z=[z, z, z, z, z],
                mode="lines",
                line=dict(color="rgba(40,130,80,0.75)", width=4),
                name="support patch boundary",
                showlegend=False,
                hoverinfo="skip",
            ),
        ]


    def expand_ranges_with_patch(ranges: tuple[list[float], list[float], list[float]], geometry: dict[str, float] | None) -> tuple[list[float], list[float], list[float]]:
        if not geometry:
            return ranges
        x_range, y_range, z_range = [list(item) for item in ranges]
        x_range = [min(x_range[0], geometry["x0"]), max(x_range[1], geometry["x1"])]
        y_range = [min(y_range[0], geometry["y0"]), max(y_range[1], geometry["y1"])]
        z_range = [min(z_range[0], geometry["z"]), max(z_range[1], geometry["z"])]
        return x_range, y_range, z_range


    def bounded_display_ranges(df: pd.DataFrame, families: list[str], geometry: dict[str, float] | None) -> tuple[list[float], list[float], list[float]]:
        xs, ys, zs = [], [], []
        for _, row in df.iterrows():
            for family in families:
                for landmark in LANDMARKS:
                    x, y, z = bounded_display_xyz(row, landmark, family)
                    xs.append(x)
                    ys.append(y)
                    zs.append(z)
        return expand_ranges_with_patch((finite_range(xs, 0.15), finite_range(ys, 0.15), finite_range(zs, 0.15)), geometry)


    def bounded_axis_layout(title: str, axis_range: list[float]) -> dict:
        return {
            "title": title,
            "range": [float(axis_range[0]), float(axis_range[1])],
            "autorange": False,
            "showspikes": False,
        }


    def bounded_manual_aspect_ratio(ranges: tuple[list[float], list[float], list[float]]) -> dict[str, float]:
        spans = np.array([max(float(axis[1]) - float(axis[0]), 1e-6) for axis in ranges], dtype=float)
        max_span = float(np.nanmax(spans)) if np.isfinite(spans).any() else 1.0
        if not np.isfinite(max_span) or max_span <= 0.0:
            max_span = 1.0
        ratio = spans / max_span
        return {"x": float(ratio[0]), "y": float(ratio[1]), "z": float(ratio[2])}


    support_patch_geometry = support_surface_patch_geometry(
        bounded_plot_df,
        BOUNDED_RV_FAMILY,
        bounded_rv_support_surface_report_df,
        bounded_bend_guard_cfg.get("support_surface_height_guard", {}),
    )

    def bounded_rv_traces(row: pd.Series) -> list[go.Scatter3d]:
        traces = support_surface_patch_traces(support_patch_geometry)
        traces += bounded_skeleton_traces(row, BOUNDED_RV_SOURCE_FAMILY, "norm", "rgba(0,0,0,0.95)")
        if rv_family_has_coordinates(pd.DataFrame([row]), BOUNDED_RV_BASELINE_FAMILY, bounded_required_landmarks):
            traces += bounded_skeleton_traces(row, BOUNDED_RV_BASELINE_FAMILY, f"strict {BOUNDED_RV_BASELINE_FAMILY}", "rgba(220,40,80,0.65)")
        traces += bounded_skeleton_traces(row, BOUNDED_RV_FAMILY, f"bounded {BOUNDED_RV_FAMILY}", "rgba(0,170,170,0.95)")
        return traces

    bounded_anim_stride = max(1, int(PLAYBACK_STRIDE))
    bounded_anim_df = bounded_plot_df.iloc[::bounded_anim_stride].copy()
    bounded_plot_families = [BOUNDED_RV_SOURCE_FAMILY, BOUNDED_RV_FAMILY]
    if rv_family_has_coordinates(bounded_plot_df, BOUNDED_RV_BASELINE_FAMILY, bounded_required_landmarks):
        bounded_plot_families.insert(1, BOUNDED_RV_BASELINE_FAMILY)
    bounded_ranges = bounded_display_ranges(bounded_plot_df, bounded_plot_families, support_patch_geometry)
    bounded_aspect_ratio = bounded_manual_aspect_ratio(bounded_ranges)
    bounded_frame_duration_ms = max(10, int(1000 / max(float(globals().get("fps_estimate", PLAYBACK_PROFILE.get("fps", 30.0))), 1.0)))
    fig_bounded_play = go.Figure(data=bounded_rv_traces(bounded_anim_df.iloc[0]))
    fig_bounded_play.frames = [
        go.Frame(data=bounded_rv_traces(row), name=str(int(row["frame"])))
        for _, row in bounded_anim_df.iterrows()
    ]
    fig_bounded_play.update_layout(
        title=f"Rep {BOUNDED_RV_REP_ID}: support-anchored playback - norm vs strict vs bounded recording-view skeleton placement",
        scene=dict(
            xaxis=bounded_axis_layout("recording x relative to support anchor", bounded_ranges[0]),
            yaxis=bounded_axis_layout("model depth z relative to support anchor", bounded_ranges[1]),
            zaxis=bounded_axis_layout("height relative to support anchor", bounded_ranges[2]),
            aspectmode="manual",
            aspectratio=bounded_aspect_ratio,
            camera=COMPACT_SCENE_CAMERA,
        ),
        height=720,
        width=1050,
        showlegend=True,
        sliders=[{
            "active": 0,
            "currentvalue": {"prefix": "Source frame: "},
            "pad": {"t": 50},
            "steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}}], "label": frame.name, "method": "animate"} for frame in fig_bounded_play.frames],
        }],
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "x": 0.0,
            "y": 1.0,
            "xanchor": "left",
            "yanchor": "top",
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": bounded_frame_duration_ms, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True, "mode": "immediate"}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
            ],
        }],
    )
    add_floor_axis_guide(fig_bounded_play, bounded_ranges)


## 11 Promoted Stack Audit And Support/Body Hypothesis Diagnostic

This report-only surface confirms the promoted correction stack and compares support-anchored versus body-anchored artifact explanations using burden/residual evidence from the existing candidates. It does not create a new coordinate family and it is not scoring evidence.


In [ ]:
CORRECTION_STACK_AUDIT_CONFIG = CORRIDOR_CONFIG.get("correction_stack_audit", {})
SUPPORT_BODY_HYPOTHESIS_CONFIG = CORRIDOR_CONFIG.get("support_body_hypothesis_diagnostic", {})


def audit_config_reporting_flag(config_path: str) -> bool:
    node = nested_config(CORRIDOR_CONFIG, config_path) if config_path else {}
    if not isinstance(node, dict):
        return False
    reporting = node.get("reporting", {})
    return bool(reporting.get("used_for_features_or_scores", False)) if isinstance(reporting, dict) else False


def audit_effective_enabled(config_path: str, configured_enabled: bool) -> bool:
    node = nested_config(CORRIDOR_CONFIG, config_path) if config_path else {}
    if isinstance(node, dict) and "enabled" in node:
        return bool(node.get("enabled", False))
    return bool(configured_enabled)


def add_stack_row(rows: list[dict], policy_id: str, tier: str, config_path: str, enabled: bool) -> None:
    effective_enabled = audit_effective_enabled(config_path, bool(enabled))
    rows.append({
        "policy_id": policy_id,
        "tier": tier,
        "config_path": config_path,
        "configured_enabled": bool(enabled),
        "enabled": bool(effective_enabled),
        "audit_action": "promoted_active_stack" if effective_enabled else "expected_active_but_disabled",
        "used_for_features_or_scores": audit_config_reporting_flag(config_path),
    })


correction_stack_audit_df = pd.DataFrame()
if not bool(CORRECTION_STACK_AUDIT_CONFIG.get("enabled", False)):
    display(Markdown("`correction_stack_audit` is disabled in the run profile."))
else:
    rv_solver_cfg = RV_FIT_CONFIG.get("solver", {}) if isinstance(RV_FIT_CONFIG, dict) else {}
    rv_support_context_cfg = RV_FIT_CONFIG.get("closed_chain_support_context", {}) if isinstance(RV_FIT_CONFIG, dict) else {}
    bounded_solver_cfg = BOUNDED_RV_CONFIG.get("solver", {}) if isinstance(BOUNDED_RV_CONFIG, dict) else {}
    stack_rows: list[dict] = []
    promoted_policies = [
        ("within_session_segment_memory", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.within_session_segment_memory", bool(SEGMENT_MEMORY_CONFIG.get("enabled", False))),
        ("closed_chain_support_context", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.closed_chain_support_context", bool(rv_support_context_cfg.get("enabled", False))),
        ("support_width_guard", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.closed_chain_support_context.support_width_guard", bool(rv_support_context_cfg.get("support_width_guard", {}).get("enabled", False))),
        ("rv_skeleton_fit", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit", bool(RV_FIT_ENABLED)),
        ("rv_skeleton_fit_bounded_xy", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.bounded_recording_view_residual", bool(BOUNDED_RV_ENABLED)),
        ("soft_bend_side_guard", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.bounded_recording_view_residual.solver.bend_side_guard", bool(bounded_solver_cfg.get("bend_side_guard", {}).get("enabled", False))),
        ("support_surface_height_guard", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.bounded_recording_view_residual.solver.support_surface_height_guard", bool(bounded_solver_cfg.get("support_surface_height_guard", {}).get("enabled", False))),
        ("visible_support_mirrored_anchor_prior", "promoted_candidate_stack", "recording_view_constrained_skeleton_fit.bounded_recording_view_residual.solver.visible_support_mirrored_anchor_prior", bool(bounded_solver_cfg.get("visible_support_mirrored_anchor_prior", {}).get("enabled", False))),
        ("bounded_pre_post_standing_anchor_blend", "promoted_candidate_stack", "bounded_pre_post_standing_anchor_blend", bool(CORRIDOR_CONFIG.get("bounded_pre_post_standing_anchor_blend", {}).get("enabled", False))),
        ("planted_support_temporal_memory", "promoted_candidate_stack", "planted_support_temporal_memory", bool(CORRIDOR_CONFIG.get("planted_support_temporal_memory", {}).get("enabled", False))),
        ("score_readiness_review", "promoted_candidate_gate", "score_readiness_review", bool(CORRIDOR_CONFIG.get("score_readiness_review", {}).get("enabled", False))),
        ("bend_flip_provenance_review", "promoted_candidate_gate", "bend_flip_provenance_review", bool(CORRIDOR_CONFIG.get("bend_flip_provenance_review", {}).get("enabled", False))),
    ]
    for policy_id, tier, config_path, enabled in promoted_policies:
        add_stack_row(stack_rows, policy_id, tier, config_path, enabled)

    correction_stack_audit_df = pd.DataFrame(stack_rows)
    if not correction_stack_audit_df.empty:
        correction_stack_audit_df = correction_stack_audit_df.sort_values(
            by=["tier", "policy_id"],
            ascending=[True, True],
        ).reset_index(drop=True)
    display(Markdown("### Promoted correction stack audit ledger"))
    display(correction_stack_audit_df)


def diagnostic_mask_from_config(cfg: dict) -> np.ndarray:
    rep_id = int(cfg.get("review_rep_id", COMPACT_REP_ID))
    mask = corridor_df["rep_id"].astype("Int64").eq(rep_id).fillna(False).to_numpy(dtype=bool)
    if bool(cfg.get("include_ready_window", True)):
        mask |= mask_ref
    if "use_for_analysis" in corridor_df.columns:
        mask &= (corridor_df["use_for_analysis"].astype(bool).to_numpy() | mask_ref)
    if not np.any(mask):
        mask = compact_rep_mask.copy()
    return mask


def diagnostic_family_available(family: str, landmarks: list[str]) -> bool:
    if not family or not landmarks:
        return False
    return all(fcol(landmark, family, axis) in corridor_df.columns for landmark in landmarks for axis in ["x", "y", "z"])


def diagnostic_safe_median(values: np.ndarray) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanmedian(arr)) if arr.size else float("nan")


def diagnostic_safe_percentile(values: np.ndarray, percentile: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanpercentile(arr, percentile)) if arr.size else float("nan")


def diagnostic_landmark_shift(source_family: str, candidate_family: str, landmarks: list[str], mask: np.ndarray) -> dict:
    frame_mags = []
    depth_abs = []
    xy_mags = []
    for landmark in landmarks:
        if not diagnostic_family_available(source_family, [landmark]) or not diagnostic_family_available(candidate_family, [landmark]):
            continue
        src = point_array(corridor_df, landmark, source_family)
        cand = point_array(corridor_df, landmark, candidate_family)
        delta = cand - src
        frame_mags.append(np.linalg.norm(delta, axis=1))
        depth_abs.append(np.abs(delta[:, 2]))
        xy_mags.append(np.linalg.norm(delta[:, :2], axis=1))
    if not frame_mags:
        return {
            "median_shift": float("nan"),
            "p95_shift": float("nan"),
            "max_shift": float("nan"),
            "median_depth_shift": float("nan"),
            "median_recording_xy_shift": float("nan"),
        }
    frame_max = np.nanmax(np.vstack(frame_mags), axis=0)
    depth_max = np.nanmax(np.vstack(depth_abs), axis=0)
    xy_max = np.nanmax(np.vstack(xy_mags), axis=0)
    return {
        "median_shift": diagnostic_safe_median(frame_max[mask]),
        "p95_shift": diagnostic_safe_percentile(frame_max[mask], 95),
        "max_shift": float(np.nanmax(frame_max[mask])) if np.any(mask) and np.isfinite(frame_max[mask]).any() else float("nan"),
        "median_depth_shift": diagnostic_safe_median(depth_max[mask]),
        "median_recording_xy_shift": diagnostic_safe_median(xy_max[mask]),
    }


def diagnostic_support_metrics(family: str, support_landmarks: list[str], primary_landmarks: list[str], mask: np.ndarray) -> dict:
    height_values = []
    for landmark in support_landmarks:
        if diagnostic_family_available(family, [landmark]):
            height_values.append(point_array(corridor_df, landmark, family)[:, 1])
    if height_values:
        height_range = np.nanmax(np.vstack(height_values), axis=0) - np.nanmin(np.vstack(height_values), axis=0)
    else:
        height_range = np.full(len(corridor_df), np.nan)

    width = np.full(len(corridor_df), np.nan)
    if len(primary_landmarks) >= 2 and diagnostic_family_available(family, primary_landmarks[:2]):
        left = point_array(corridor_df, primary_landmarks[0], family)
        right = point_array(corridor_df, primary_landmarks[1], family)
        width = np.linalg.norm(left[:, [0, 2]] - right[:, [0, 2]], axis=1)
    ref_mask = mask_ref if np.any(mask_ref) else mask
    target_width = diagnostic_safe_median(width[ref_mask])
    width_residual = np.abs(width - target_width) if np.isfinite(target_width) else np.full(len(corridor_df), np.nan)
    return {
        "median_support_height_range": diagnostic_safe_median(height_range[mask]),
        "p95_support_height_range": diagnostic_safe_percentile(height_range[mask], 95),
        "support_width_target": target_width,
        "median_support_width_residual": diagnostic_safe_median(width_residual[mask]),
        "p95_support_width_residual": diagnostic_safe_percentile(width_residual[mask], 95),
    }


def diagnostic_segment_residual(family: str, mask: np.ndarray) -> dict:
    target_table = globals().get("rv_targets", {})
    if not target_table and "rv_target_lengths_from_matrix" in globals():
        target_table = rv_target_lengths_from_matrix()
    residuals = []
    for chain in RV_FIT_CONFIG.get("chains", []):
        for segment_key, a, b in [
            (str(chain.get("proximal_segment", "")), str(chain.get("proximal", "")), str(chain.get("joint", ""))),
            (str(chain.get("distal_segment", "")), str(chain.get("joint", "")), str(chain.get("distal", ""))),
        ]:
            if not segment_key or not diagnostic_family_available(family, [a, b]):
                continue
            target = float(target_table.get(segment_key, {}).get("target", np.nan)) if isinstance(target_table, dict) else float("nan")
            if not np.isfinite(target):
                continue
            residuals.append(np.abs(segment_length_series(corridor_df, a, b, family) - target))
    if not residuals:
        return {"median_segment_residual": float("nan"), "p95_segment_residual": float("nan")}
    residual = np.concatenate([values[mask] for values in residuals])
    return {
        "median_segment_residual": diagnostic_safe_median(residual),
        "p95_segment_residual": diagnostic_safe_percentile(residual, 95),
    }


support_body_metric_df = pd.DataFrame()
support_body_hypothesis_df = pd.DataFrame()
if not bool(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("enabled", False)):
    display(Markdown("`support_body_hypothesis_diagnostic` is disabled in the run profile."))
else:
    diag_source_family = str(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("source_family", RV_FIT_SOURCE_FAMILY))
    diag_candidate_family = str(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("bounded_candidate_family", BOUNDED_RV_FAMILY))
    if not diagnostic_family_available(diag_candidate_family, ["left_ankle", "right_ankle"]):
        diag_candidate_family = str(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("strict_candidate_family", RV_FIT_FAMILY))
    diag_mask = diagnostic_mask_from_config(SUPPORT_BODY_HYPOTHESIS_CONFIG)
    support_landmarks = [str(item) for item in SUPPORT_BODY_HYPOTHESIS_CONFIG.get("support_landmarks", ["left_ankle", "right_ankle"])]
    primary_support_landmarks = [str(item) for item in SUPPORT_BODY_HYPOTHESIS_CONFIG.get("primary_support_landmarks", ["left_ankle", "right_ankle"])]
    body_landmarks = [str(item) for item in SUPPORT_BODY_HYPOTHESIS_CONFIG.get("body_landmarks", ["left_hip", "right_hip"])]
    lower_limb_landmarks = [str(item) for item in SUPPORT_BODY_HYPOTHESIS_CONFIG.get("lower_limb_landmarks", [])]

    support_shift = diagnostic_landmark_shift(diag_source_family, diag_candidate_family, support_landmarks, diag_mask)
    body_shift = diagnostic_landmark_shift(diag_source_family, diag_candidate_family, body_landmarks, diag_mask)
    lower_limb_shift = diagnostic_landmark_shift(diag_source_family, diag_candidate_family, lower_limb_landmarks, diag_mask)
    source_support = diagnostic_support_metrics(diag_source_family, support_landmarks, primary_support_landmarks, diag_mask)
    candidate_support = diagnostic_support_metrics(diag_candidate_family, support_landmarks, primary_support_landmarks, diag_mask)
    source_segment = diagnostic_segment_residual(diag_source_family, diag_mask)
    candidate_segment = diagnostic_segment_residual(diag_candidate_family, diag_mask)

    support_median = support_shift["median_shift"]
    body_median = body_shift["median_shift"]
    denom = support_median + body_median
    support_concentration = float(support_median / denom) if np.isfinite(denom) and denom > 1e-9 else float("nan")
    segment_gain = source_segment["median_segment_residual"] - candidate_segment["median_segment_residual"]
    height_gain = source_support["median_support_height_range"] - candidate_support["median_support_height_range"]
    width_gain = source_support["median_support_width_residual"] - candidate_support["median_support_width_residual"]

    support_body_metric_df = pd.DataFrame([{
        "source_family": diag_source_family,
        "candidate_family": diag_candidate_family,
        "review_frames": int(np.sum(diag_mask)),
        "median_support_shift": support_shift["median_shift"],
        "p95_support_shift": support_shift["p95_shift"],
        "median_body_shift": body_shift["median_shift"],
        "p95_body_shift": body_shift["p95_shift"],
        "median_lower_limb_shift": lower_limb_shift["median_shift"],
        "support_shift_concentration": support_concentration,
        "source_median_support_height_range": source_support["median_support_height_range"],
        "candidate_median_support_height_range": candidate_support["median_support_height_range"],
        "support_height_range_gain": height_gain,
        "source_median_support_width_residual": source_support["median_support_width_residual"],
        "candidate_median_support_width_residual": candidate_support["median_support_width_residual"],
        "support_width_residual_gain": width_gain,
        "source_median_segment_residual": source_segment["median_segment_residual"],
        "candidate_median_segment_residual": candidate_segment["median_segment_residual"],
        "segment_residual_gain": segment_gain,
    }])
    for col in support_body_metric_df.columns:
        if col not in {"source_family", "candidate_family"}:
            support_body_metric_df[col] = pd.to_numeric(support_body_metric_df[col], errors="coerce")
    numeric_cols = support_body_metric_df.select_dtypes(include=[np.number]).columns
    support_body_metric_df[numeric_cols] = support_body_metric_df[numeric_cols].round(6)

    concentration_warn = float(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("support_burden_concentration_warn_ratio", 0.60))
    large_shift_warn = float(SUPPORT_BODY_HYPOTHESIS_CONFIG.get("large_shift_warn_ratio", 0.08))
    support_anchor_signal = "review_body_depth_candidate" if (
        np.isfinite(support_concentration) and support_concentration >= concentration_warn
    ) or (
        np.isfinite(support_shift["p95_shift"]) and support_shift["p95_shift"] >= large_shift_warn
    ) else "support_shift_burden_moderate"
    body_anchor_signal = "current_support_candidate_plausible_only_as_low_confidence" if (
        np.isfinite(segment_gain) and segment_gain > 0.0
    ) else "no_segment_residual_gain"

    support_body_hypothesis_df = pd.DataFrame([
        {
            "hypothesis_id": "support_anchored_body_depth_hypothesis",
            "anchor_assumption": "feet/support landmarks are the task anchor",
            "main_burden_to_watch": "body/pelvis or lower-limb depth pressure; avoid moving support heavily",
            "current_candidate_warning": support_anchor_signal,
            "confidence_rule": "If support shift burden is high, test body-depth counterfactual before stronger foot correction.",
            "used_for_features_or_scores": False,
        },
        {
            "hypothesis_id": "body_anchored_support_depth_hypothesis",
            "anchor_assumption": "pelvis/body placement is the observation anchor",
            "main_burden_to_watch": "support landmark shift and support width/height residual change",
            "current_candidate_warning": body_anchor_signal,
            "confidence_rule": "Even when segment residual improves, support movement remains low-confidence correction burden.",
            "used_for_features_or_scores": False,
        },
    ])

    display(Markdown("### Support/body hypothesis diagnostic metrics"))
    display(support_body_metric_df)
    display(Markdown("### Support/body hypothesis interpretation"))
    display(support_body_hypothesis_df)


## 11.5 Visible-Support Mirrored Anchor Focused Comparison

This post-candidate view is intentionally narrow: it compares the support anchor effect after `rv_skeleton_fit_bounded_xy` has been created.


In [ ]:
display(Markdown("### Visible-Support Mirrored Anchor Focus"))

if "bounded_rv_visible_support_anchor_report_df" in globals() and not bounded_rv_visible_support_anchor_report_df.empty:
    focus_cols = [
        col
        for col in [
            "trusted_side",
            "target_side",
            "trusted_landmark",
            "target_landmark",
            "anchor_z",
            "median_abs_z_residual_before",
            "median_abs_z_residual_after",
            "support_width_target",
            "median_support_width_residual_before",
            "median_support_width_residual_after",
            "support_width_rejected_frame_ratio",
            "active_frame_ratio",
            "median_abs_z_shift",
            "p95_abs_z_shift",
            "max_abs_z_shift",
            "used_for_features_or_scores",
            "confidence_when_applied",
        ]
        if col in bounded_rv_visible_support_anchor_report_df.columns
    ]
    display(bounded_rv_visible_support_anchor_report_df[focus_cols])
else:
    display(Markdown("No visible-support mirrored anchor report is available for this run."))

if "fig_bounded_play" in globals() and review_show_visual("bounded_playback"):
    fig_bounded_play.show()

if BOUNDED_RV_ENABLED and "bounded_rep_mask" in globals() and np.any(bounded_rep_mask):
    anchor_focus_df = corridor_df.loc[bounded_rep_mask].copy()
    anchor_focus_frames = anchor_focus_df["frame"].astype(int).to_numpy()
    anchor_cfg = bounded_solver_cfg.get("visible_support_mirrored_anchor_prior", {}) if "bounded_solver_cfg" in globals() else {}
    support_pair = list(RV_FIT_CONFIG.get("support_residual", {}).get("pair", ["left_ankle", "right_ankle"]))
    trim_q = float(np.clip(float(anchor_cfg.get("support_width_no_worsen", {}).get("trim_quantile", 0.10)), 0.0, 0.45))
    source_width = np.linalg.norm(
        point_array(corridor_df, support_pair[1], BOUNDED_RV_SOURCE_FAMILY)[:, [0, 2]]
        - point_array(corridor_df, support_pair[0], BOUNDED_RV_SOURCE_FAMILY)[:, [0, 2]],
        axis=1,
    )
    anchor_support_width_target = rv_trimmed_median_values(source_width[mask_ref], trim_q) if np.any(mask_ref) else rv_trimmed_median_values(source_width[bounded_rep_mask], trim_q)

    anchor_report = bounded_rv_visible_support_anchor_report_df if "bounded_rv_visible_support_anchor_report_df" in globals() else pd.DataFrame()
    target_landmark = "right_ankle"
    anchor_z = np.nan
    if not anchor_report.empty and "primary_landmark" in anchor_report.columns:
        primary_rows = anchor_report[anchor_report["primary_landmark"].astype(bool)]
        if not primary_rows.empty:
            target_landmark = str(primary_rows.iloc[0].get("target_landmark", target_landmark))
            anchor_z = float(primary_rows.iloc[0].get("anchor_z", np.nan))

    fig_visible_anchor_focus = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            f"{support_pair[0]}->{support_pair[1]} support-width residual",
            f"{target_landmark} model-depth z vs mirrored anchor",
        ),
        vertical_spacing=0.12,
    )
    family_styles = [
        (BOUNDED_RV_SOURCE_FAMILY, "norm", "rgba(0,0,0,0.80)", "dot"),
        (BOUNDED_RV_BASELINE_FAMILY, "strict", "rgba(220,40,80,0.70)", "dash"),
        (BOUNDED_RV_FAMILY, "bounded + mirrored anchor", "rgba(0,170,170,0.95)", "solid"),
    ]
    for family, label, color, dash in family_styles:
        if not rv_family_has_coordinates(corridor_df, family, support_pair + [target_landmark]):
            continue
        left = point_array(anchor_focus_df, support_pair[0], family)
        right = point_array(anchor_focus_df, support_pair[1], family)
        width = np.linalg.norm(right[:, [0, 2]] - left[:, [0, 2]], axis=1)
        residual = np.abs(width - anchor_support_width_target)
        target_z = point_array(anchor_focus_df, target_landmark, family)[:, 2]
        fig_visible_anchor_focus.add_trace(
            go.Scatter(x=anchor_focus_frames, y=residual, mode="lines", name=f"{label} width residual", line=dict(color=color, dash=dash, width=3 if family == BOUNDED_RV_FAMILY else 2)),
            row=1,
            col=1,
        )
        fig_visible_anchor_focus.add_trace(
            go.Scatter(x=anchor_focus_frames, y=target_z, mode="lines", name=f"{label} {target_landmark} z", line=dict(color=color, dash=dash, width=3 if family == BOUNDED_RV_FAMILY else 2)),
            row=2,
            col=1,
        )
    if np.isfinite(anchor_z):
        fig_visible_anchor_focus.add_trace(
            go.Scatter(x=anchor_focus_frames, y=np.full(len(anchor_focus_frames), anchor_z), mode="lines", name="mirrored anchor z", line=dict(color="rgba(80,80,80,0.75)", dash="dashdot", width=2)),
            row=2,
            col=1,
        )
    fig_visible_anchor_focus.update_layout(
        title="Rep 1 visible-support mirrored anchor: post-candidate focused comparison",
        height=720,
        width=1100,
        legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0.0),
    )
    fig_visible_anchor_focus.update_xaxes(title_text="source frame", row=2, col=1)
    fig_visible_anchor_focus.update_yaxes(title_text="torso-length ratio", row=1, col=1)
    fig_visible_anchor_focus.update_yaxes(title_text="model depth z", row=2, col=1)
    if review_show_visual("support_surface_playback") or review_show_visual("visible_support_anchor_focus"):
        fig_visible_anchor_focus.show()


## 11.6 Standing Endpoint Support Anchor Compare

Report-only comparison of the first and last standing endpoints in the reviewed repetition.


In [ ]:
STANDING_ENDPOINT_CONFIG = CORRIDOR_CONFIG.get("standing_endpoint_support_anchor_compare", {})


def endpoint_trimmed_median(values: np.ndarray, trim_quantile: float = 0.0) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return float("nan")
    trim_q = float(np.clip(trim_quantile, 0.0, 0.45))
    if trim_q > 0.0 and finite.size >= 4:
        lo, hi = np.quantile(finite, [trim_q, 1.0 - trim_q])
        trimmed = finite[(finite >= lo) & (finite <= hi)]
        if trimmed.size:
            finite = trimmed
    return float(np.nanmedian(finite))


def endpoint_frame_mask(df: pd.DataFrame, start_frame: int, end_frame: int) -> np.ndarray:
    frames = df["frame"].astype(int)
    return frames.between(int(start_frame), int(end_frame)).to_numpy(dtype=bool)


def endpoint_support_metric_row(
    family: str,
    endpoint: str,
    frame_start: int,
    frame_end: int,
    mask: np.ndarray,
    support_pair: list[str],
    support_suffixes: list[str],
    body_landmarks: list[str],
) -> dict:
    row = {
        "family": family,
        "endpoint": endpoint,
        "frame_start": int(frame_start),
        "frame_end": int(frame_end),
        "frame_count": int(np.sum(mask)),
    }
    if not np.any(mask) or not rv_family_has_coordinates(corridor_df, family, support_pair):
        row["status"] = "missing_coordinates"
        return row

    left_support = point_array(corridor_df, support_pair[0], family)
    right_support = point_array(corridor_df, support_pair[1], family)
    support_width = np.linalg.norm((right_support - left_support)[:, [0, 2]], axis=1)
    support_center = 0.5 * (left_support + right_support)
    row.update({
        "status": "ok",
        "support_width": endpoint_trimmed_median(support_width[mask]),
        "support_center_x": endpoint_trimmed_median(support_center[mask, 0]),
        "support_center_y": endpoint_trimmed_median(support_center[mask, 1]),
        "support_center_z": endpoint_trimmed_median(support_center[mask, 2]),
    })

    support_y_values = []
    for side in ["left", "right"]:
        for suffix in support_suffixes:
            landmark = f"{side}_{suffix}"
            if rv_family_has_coordinates(corridor_df, family, [landmark]):
                xyz = point_array(corridor_df, landmark, family)
                support_y_values.append(xyz[:, 1])
                row[f"{landmark}_z"] = endpoint_trimmed_median(xyz[mask, 2])
    if support_y_values:
        support_y_stack = np.vstack(support_y_values)
        support_height_range = np.nanmax(support_y_stack, axis=0) - np.nanmin(support_y_stack, axis=0)
        row["support_height_range"] = endpoint_trimmed_median(support_height_range[mask])
    else:
        row["support_height_range"] = np.nan

    if body_landmarks and rv_family_has_coordinates(corridor_df, family, body_landmarks):
        body_stack = np.stack([point_array(corridor_df, landmark, family) for landmark in body_landmarks], axis=1)
        body_center = np.nanmean(body_stack, axis=1)
        body_support_offset = body_center - support_center
        row.update({
            "body_center_x": endpoint_trimmed_median(body_center[mask, 0]),
            "body_center_y": endpoint_trimmed_median(body_center[mask, 1]),
            "body_center_z": endpoint_trimmed_median(body_center[mask, 2]),
            "body_support_offset_x": endpoint_trimmed_median(body_support_offset[mask, 0]),
            "body_support_offset_y": endpoint_trimmed_median(body_support_offset[mask, 1]),
            "body_support_offset_z": endpoint_trimmed_median(body_support_offset[mask, 2]),
        })

    segment_residuals = []
    for chain in RV_FIT_CONFIG.get("chains", []):
        proximal = str(chain["proximal"])
        joint = str(chain["joint"])
        distal = str(chain["distal"])
        if rv_family_has_coordinates(corridor_df, family, [proximal, joint, distal]):
            proximal_segment = str(chain["proximal_segment"])
            distal_segment = str(chain["distal_segment"])
            proximal_memory_id = str(chain.get("proximal_memory_id", rv_default_memory_id(str(chain.get("id", "chain")), proximal_segment)))
            distal_memory_id = str(chain.get("distal_memory_id", rv_default_memory_id(str(chain.get("id", "chain")), distal_segment)))
            proximal_target = rv_target_length(proximal_segment, proximal_memory_id)
            distal_target = rv_target_length(distal_segment, distal_memory_id)
            proximal_len = np.linalg.norm(point_array(corridor_df, proximal, family) - point_array(corridor_df, joint, family), axis=1)
            distal_len = np.linalg.norm(point_array(corridor_df, joint, family) - point_array(corridor_df, distal, family), axis=1)
            if np.isfinite(proximal_target):
                segment_residuals.append(np.abs(proximal_len[mask] - proximal_target))
            if np.isfinite(distal_target):
                segment_residuals.append(np.abs(distal_len[mask] - distal_target))
    row["median_lower_limb_segment_residual"] = endpoint_trimmed_median(np.concatenate(segment_residuals)) if segment_residuals else np.nan
    return row


def endpoint_series_for_family(family: str, rep_mask: np.ndarray, support_pair: list[str], body_landmarks: list[str]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if not rv_family_has_coordinates(corridor_df, family, support_pair + body_landmarks):
        nan = np.full(int(np.sum(rep_mask)), np.nan)
        return nan, nan, nan, nan
    left_support = point_array(corridor_df, support_pair[0], family)
    right_support = point_array(corridor_df, support_pair[1], family)
    support_width = np.linalg.norm((right_support - left_support)[:, [0, 2]], axis=1)
    support_center = 0.5 * (left_support + right_support)
    body_stack = np.stack([point_array(corridor_df, landmark, family) for landmark in body_landmarks], axis=1)
    body_center = np.nanmean(body_stack, axis=1)
    body_support_offset_z = body_center[:, 2] - support_center[:, 2]
    right_ankle_z = point_array(corridor_df, support_pair[1], family)[:, 2]
    left_ankle_z = point_array(corridor_df, support_pair[0], family)[:, 2]
    return support_width[rep_mask], body_support_offset_z[rep_mask], left_ankle_z[rep_mask], right_ankle_z[rep_mask]


if not bool(STANDING_ENDPOINT_CONFIG.get("enabled", False)):
    display(Markdown("`standing_endpoint_support_anchor_compare` is disabled in the run profile."))
else:
    endpoint_rep_id = int(STANDING_ENDPOINT_CONFIG.get("review_rep_id", BOUNDED_RV_REP_ID if "BOUNDED_RV_REP_ID" in globals() else COMPACT_REP_ID))
    endpoint_rep_rows = ann_df[(ann_df["segment_type"].eq("rep")) & (ann_df["rep_id"].astype("Int64").eq(endpoint_rep_id))]
    if endpoint_rep_rows.empty:
        display(Markdown(f"No rep annotation found for endpoint comparison rep_id={endpoint_rep_id}."))
    else:
        endpoint_rep = endpoint_rep_rows.iloc[0]
        endpoint_start_frame = int(endpoint_rep["start_frame"])
        endpoint_end_frame = int(endpoint_rep["end_frame"])
        start_before = int(STANDING_ENDPOINT_CONFIG.get("start_window_before_frames", 2))
        end_before = int(STANDING_ENDPOINT_CONFIG.get("end_window_before_frames", 2))
        start_window_start = endpoint_start_frame - start_before
        start_window_end = endpoint_start_frame
        end_window_start = endpoint_end_frame - end_before
        end_window_end = endpoint_end_frame
        support_pair = [str(item) for item in STANDING_ENDPOINT_CONFIG.get("support_pair", ["left_ankle", "right_ankle"])]
        support_suffixes = [str(item) for item in STANDING_ENDPOINT_CONFIG.get("support_landmark_suffixes", ["ankle", "heel", "foot_index"])]
        body_landmarks = [str(item) for item in STANDING_ENDPOINT_CONFIG.get("body_center_landmarks", ["left_hip", "right_hip"])]
        requested_families = [str(item) for item in STANDING_ENDPOINT_CONFIG.get("families", ["norm", RV_FIT_FAMILY, BOUNDED_RV_FAMILY])]
        endpoint_families = []
        for family in requested_families:
            if family not in endpoint_families and rv_family_has_coordinates(corridor_df, family, support_pair):
                endpoint_families.append(family)

        start_mask = endpoint_frame_mask(corridor_df, start_window_start, start_window_end)
        end_mask = endpoint_frame_mask(corridor_df, end_window_start, end_window_end)
        exact_start_mask = endpoint_frame_mask(corridor_df, endpoint_start_frame, endpoint_start_frame)
        exact_end_mask = endpoint_frame_mask(corridor_df, endpoint_end_frame, endpoint_end_frame)
        endpoint_rows = []
        exact_endpoint_rows = []
        for family in endpoint_families:
            endpoint_rows.append(endpoint_support_metric_row(family, "start_window", start_window_start, start_window_end, start_mask, support_pair, support_suffixes, body_landmarks))
            endpoint_rows.append(endpoint_support_metric_row(family, "end_window", end_window_start, end_window_end, end_mask, support_pair, support_suffixes, body_landmarks))
            exact_endpoint_rows.append(endpoint_support_metric_row(family, "start_frame", endpoint_start_frame, endpoint_start_frame, exact_start_mask, support_pair, support_suffixes, body_landmarks))
            exact_endpoint_rows.append(endpoint_support_metric_row(family, "end_frame", endpoint_end_frame, endpoint_end_frame, exact_end_mask, support_pair, support_suffixes, body_landmarks))

        standing_endpoint_df = pd.DataFrame(endpoint_rows)
        standing_endpoint_exact_df = pd.DataFrame(exact_endpoint_rows)
        threshold_cfg = STANDING_ENDPOINT_CONFIG.get("thresholds", {})
        support_width_thr = float(threshold_cfg.get("support_width_delta_ratio", 0.03))
        support_height_thr = float(threshold_cfg.get("support_height_range_delta_ratio", 0.03))
        pelvis_offset_thr = float(threshold_cfg.get("pelvis_support_offset_delta_ratio", 0.04))
        anchor_z_thr = float(threshold_cfg.get("support_anchor_z_delta_ratio", 0.05))

        delta_rows = []
        for family in endpoint_families:
            family_rows = standing_endpoint_df[standing_endpoint_df["family"].eq(family)].set_index("endpoint")
            if {"start_window", "end_window"}.issubset(set(family_rows.index)):
                start = family_rows.loc["start_window"]
                end = family_rows.loc["end_window"]
                row = {
                    "family": family,
                    "start_window": f"{start_window_start}-{start_window_end}",
                    "end_window": f"{end_window_start}-{end_window_end}",
                    "support_width_delta": abs(float(end.get("support_width", np.nan)) - float(start.get("support_width", np.nan))),
                    "support_height_range_delta": abs(float(end.get("support_height_range", np.nan)) - float(start.get("support_height_range", np.nan))),
                    "body_support_offset_z_delta": abs(float(end.get("body_support_offset_z", np.nan)) - float(start.get("body_support_offset_z", np.nan))),
                    "right_ankle_z_delta": abs(float(end.get("right_ankle_z", np.nan)) - float(start.get("right_ankle_z", np.nan))),
                    "left_ankle_z_delta": abs(float(end.get("left_ankle_z", np.nan)) - float(start.get("left_ankle_z", np.nan))),
                    "segment_residual_delta": abs(float(end.get("median_lower_limb_segment_residual", np.nan)) - float(start.get("median_lower_limb_segment_residual", np.nan))),
                }
                warnings = []
                if np.isfinite(row["support_width_delta"]) and row["support_width_delta"] > support_width_thr:
                    warnings.append("support_width_endpoint_delta")
                if np.isfinite(row["support_height_range_delta"]) and row["support_height_range_delta"] > support_height_thr:
                    warnings.append("support_height_endpoint_delta")
                if np.isfinite(row["body_support_offset_z_delta"]) and row["body_support_offset_z_delta"] > pelvis_offset_thr:
                    warnings.append("body_support_depth_endpoint_delta")
                if max(row["right_ankle_z_delta"], row["left_ankle_z_delta"]) > anchor_z_thr:
                    warnings.append("support_anchor_depth_endpoint_delta")
                row["warning_count"] = len(warnings)
                row["warning_flags"] = ",".join(warnings)
                row["used_for_features_or_scores"] = bool(STANDING_ENDPOINT_CONFIG.get("reporting", {}).get("used_for_features_or_scores", False))
                row["confidence_when_applied"] = STANDING_ENDPOINT_CONFIG.get("reporting", {}).get("confidence_when_applied", "very_low")
                row["suggested_next_step"] = "review_bounded_pre_post_standing_anchor_blend" if warnings and family == BOUNDED_RV_FAMILY else "no_coordinate_change_from_this_diagnostic"
                delta_rows.append(row)
        standing_endpoint_delta_df = pd.DataFrame(delta_rows)

        display(Markdown("### Standing Endpoint Support Anchor Compare"))
        display(standing_endpoint_delta_df)
        display(Markdown("### Endpoint Window Metrics"))
        display(standing_endpoint_df)
        display(Markdown("### Exact Endpoint Frame Metrics"))
        display(standing_endpoint_exact_df)

        rep_mask = endpoint_frame_mask(corridor_df, endpoint_start_frame, endpoint_end_frame)
        rep_frames = corridor_df.loc[rep_mask, "frame"].astype(int).to_numpy()
        fig_endpoint_series = make_subplots(
            rows=3,
            cols=1,
            shared_xaxes=True,
            subplot_titles=("support width", "body/support depth offset", "ankle model-depth z"),
            vertical_spacing=0.10,
        )
        endpoint_styles = [
            ("norm", "norm", "rgba(0,0,0,0.75)", "dot"),
            (RV_FIT_FAMILY, "strict", "rgba(220,40,80,0.70)", "dash"),
            (BOUNDED_RV_FAMILY, "bounded", "rgba(0,170,170,0.95)", "solid"),
        ]
        for family, label, color, dash in endpoint_styles:
            if family not in endpoint_families:
                continue
            support_width, body_offset_z, left_ankle_z, right_ankle_z = endpoint_series_for_family(family, rep_mask, support_pair, body_landmarks)
            fig_endpoint_series.add_trace(go.Scatter(x=rep_frames, y=support_width, mode="lines", name=f"{label} support width", line=dict(color=color, dash=dash, width=3 if family == BOUNDED_RV_FAMILY else 2)), row=1, col=1)
            fig_endpoint_series.add_trace(go.Scatter(x=rep_frames, y=body_offset_z, mode="lines", name=f"{label} body-support z", line=dict(color=color, dash=dash, width=3 if family == BOUNDED_RV_FAMILY else 2)), row=2, col=1)
            fig_endpoint_series.add_trace(go.Scatter(x=rep_frames, y=right_ankle_z, mode="lines", name=f"{label} right ankle z", line=dict(color=color, dash=dash, width=3 if family == BOUNDED_RV_FAMILY else 2)), row=3, col=1)
            fig_endpoint_series.add_trace(go.Scatter(x=rep_frames, y=left_ankle_z, mode="lines", name=f"{label} left ankle z", line=dict(color=color, dash="dashdot", width=1)), row=3, col=1)
        for marker_frame in [endpoint_start_frame, endpoint_end_frame]:
            fig_endpoint_series.add_vline(x=marker_frame, line=dict(color="rgba(90,90,90,0.55)", dash="dot"))
        fig_endpoint_series.update_layout(
            title=f"Rep {endpoint_rep_id} standing endpoint support-anchor comparison",
            height=820,
            width=1100,
            legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0.0),
        )
        fig_endpoint_series.update_xaxes(title_text="source frame", row=3, col=1)
        fig_endpoint_series.update_yaxes(title_text="torso-length ratio")
        if review_show_visual("support_surface_playback") or review_show_visual("standing_endpoint_compare"):
            fig_endpoint_series.show()

        overlay_family = BOUNDED_RV_FAMILY if BOUNDED_RV_FAMILY in endpoint_families else endpoint_families[-1]
        overlay_landmarks = ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle", "left_heel", "right_heel", "left_foot_index", "right_foot_index"]
        overlay_connections = [
            ("left_hip", "left_knee"), ("left_knee", "left_ankle"), ("right_hip", "right_knee"), ("right_knee", "right_ankle"),
            ("left_hip", "right_hip"), ("left_ankle", "right_ankle"), ("left_heel", "left_foot_index"), ("right_heel", "right_foot_index"),
        ]
        if rv_family_has_coordinates(corridor_df, overlay_family, overlay_landmarks):
            start_row = corridor_df.loc[corridor_df["frame"].astype(int).eq(endpoint_start_frame)].iloc[0]
            end_row = corridor_df.loc[corridor_df["frame"].astype(int).eq(endpoint_end_frame)].iloc[0]

            def endpoint_overlay_trace(row: pd.Series, label: str, color: str) -> list[go.Scatter3d]:
                lx, ly, lz = [], [], []
                for a, b in overlay_connections:
                    ax, ay, az = plot_xyz(row, a, overlay_family)
                    bx, by, bz = plot_xyz(row, b, overlay_family)
                    lx.extend([ax, bx, None])
                    ly.extend([ay, by, None])
                    lz.extend([az, bz, None])
                px, py, pz = [], [], []
                for landmark in overlay_landmarks:
                    x, y, z = plot_xyz(row, landmark, overlay_family)
                    px.append(x)
                    py.append(y)
                    pz.append(z)
                return [
                    go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=6), name=f"{label} lower body"),
                    go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=4), name=f"{label} support joints"),
                ]

            fig_endpoint_overlay = go.Figure()
            for trace in endpoint_overlay_trace(start_row, f"start frame {endpoint_start_frame}", "rgba(230,130,40,0.95)"):
                fig_endpoint_overlay.add_trace(trace)
            for trace in endpoint_overlay_trace(end_row, f"end frame {endpoint_end_frame}", "rgba(0,170,170,0.95)"):
                fig_endpoint_overlay.add_trace(trace)
            fig_endpoint_overlay.update_layout(
                title=f"Rep {endpoint_rep_id} exact standing endpoints overlay ({overlay_family})",
                scene=dict(
                    xaxis=dict(title="recording x"),
                    yaxis=dict(title="model depth z"),
                    zaxis=dict(title="recording height (-y)"),
                    aspectmode="data",
                    camera=COMPACT_SCENE_CAMERA,
                ),
                height=720,
                width=1050,
                showlegend=True,
            )
            if review_show_visual("support_surface_playback") or review_show_visual("standing_endpoint_compare"):
                fig_endpoint_overlay.show()


## 11.7 Bounded Pre/Post Standing Anchor Blend

Corrected-3D-hypothesis review candidate that applies only to the start standing endpoint window and a short fade-out ramp.


In [ ]:
ENDPOINT_BLEND_CONFIG = CORRIDOR_CONFIG.get("bounded_pre_post_standing_anchor_blend", {})
ENDPOINT_BLEND_FAMILY = str(ENDPOINT_BLEND_CONFIG.get("family", "rv_skeleton_fit_bounded_xy_endpoint_blend"))
endpoint_blend_burden_df = pd.DataFrame()
endpoint_blend_delta_df = pd.DataFrame()
endpoint_blend_window_df = pd.DataFrame()


def endpoint_blend_weights(df: pd.DataFrame, start_frame: int, cfg: dict) -> tuple[np.ndarray, dict[str, int]]:
    before = int(cfg.get("start_window_before_frames", 2))
    after = int(cfg.get("start_window_after_frames", 0))
    ramp_after = int(cfg.get("ramp_after_start_frames", 0))
    start_window_start = int(start_frame) - before
    start_window_end = int(start_frame) + after
    ramp_start = start_window_end + 1
    ramp_end = start_window_end + ramp_after
    frames = df["frame"].astype(int).to_numpy()
    weights = np.zeros(len(df), dtype=float)
    weights[(frames >= start_window_start) & (frames <= start_window_end)] = 1.0
    if ramp_after > 0:
        ramp_mask = (frames >= ramp_start) & (frames <= ramp_end)
        weights[ramp_mask] = 1.0 - ((frames[ramp_mask] - start_window_end) / float(ramp_after + 1))
    return weights, {
        "start_window_start": start_window_start,
        "start_window_end": start_window_end,
        "ramp_start": ramp_start,
        "ramp_end": ramp_end,
    }


def endpoint_delta_from_window_table(df: pd.DataFrame, family: str) -> dict:
    rows = df[df["family"].eq(family)].set_index("endpoint")
    if not {"start_window", "end_window"}.issubset(set(rows.index)):
        return {"family": family, "status": "missing_endpoint_rows"}
    start = rows.loc["start_window"]
    end = rows.loc["end_window"]
    return {
        "family": family,
        "status": "ok",
        "support_width_delta": abs(float(end.get("support_width", np.nan)) - float(start.get("support_width", np.nan))),
        "support_height_range_delta": abs(float(end.get("support_height_range", np.nan)) - float(start.get("support_height_range", np.nan))),
        "body_support_offset_z_delta": abs(float(end.get("body_support_offset_z", np.nan)) - float(start.get("body_support_offset_z", np.nan))),
        "left_ankle_z_delta": abs(float(end.get("left_ankle_z", np.nan)) - float(start.get("left_ankle_z", np.nan))),
        "right_ankle_z_delta": abs(float(end.get("right_ankle_z", np.nan)) - float(start.get("right_ankle_z", np.nan))),
        "segment_residual_delta": abs(float(end.get("median_lower_limb_segment_residual", np.nan)) - float(start.get("median_lower_limb_segment_residual", np.nan))),
    }


def endpoint_blend_improvement_row(before: dict, after: dict) -> dict:
    row = {"before_family": before.get("family"), "after_family": after.get("family")}
    for metric in [
        "support_width_delta",
        "support_height_range_delta",
        "body_support_offset_z_delta",
        "left_ankle_z_delta",
        "right_ankle_z_delta",
        "segment_residual_delta",
    ]:
        before_value = float(before.get(metric, np.nan))
        after_value = float(after.get(metric, np.nan))
        row[f"{metric}_before"] = before_value
        row[f"{metric}_after"] = after_value
        row[f"{metric}_gain"] = before_value - after_value if np.isfinite(before_value) and np.isfinite(after_value) else np.nan
    return row


if not bool(ENDPOINT_BLEND_CONFIG.get("enabled", False)):
    display(Markdown("`bounded_pre_post_standing_anchor_blend` is disabled in the run profile."))
else:
    blend_source_family = str(ENDPOINT_BLEND_CONFIG.get("source_family", BOUNDED_RV_FAMILY if "BOUNDED_RV_FAMILY" in globals() else "rv_skeleton_fit_bounded_xy"))
    endpoint_rep_id = int(ENDPOINT_BLEND_CONFIG.get("review_rep_id", BOUNDED_RV_REP_ID if "BOUNDED_RV_REP_ID" in globals() else COMPACT_REP_ID))
    endpoint_rep_rows = ann_df[(ann_df["segment_type"].eq("rep")) & (ann_df["rep_id"].astype("Int64").eq(endpoint_rep_id))]
    support_suffixes = [str(item) for item in ENDPOINT_BLEND_CONFIG.get("support_landmark_suffixes", ["ankle", "heel", "foot_index"])]
    sides = [str(item) for item in ENDPOINT_BLEND_CONFIG.get("sides", ["left", "right"])]
    support_landmarks = [f"{side}_{suffix}" for side in sides for suffix in support_suffixes]
    support_pair = [str(item) for item in ENDPOINT_BLEND_CONFIG.get("support_pair", ["left_ankle", "right_ankle"])] if "support_pair" in ENDPOINT_BLEND_CONFIG else ["left_ankle", "right_ankle"]
    body_landmarks = [str(item) for item in ENDPOINT_BLEND_CONFIG.get("body_center_landmarks", ["left_hip", "right_hip"])]
    if endpoint_rep_rows.empty:
        display(Markdown(f"No rep annotation found for endpoint blend rep_id={endpoint_rep_id}."))
    elif not rv_family_has_coordinates(corridor_df, blend_source_family, support_landmarks + body_landmarks):
        display(Markdown(f"Endpoint blend source family `{blend_source_family}` is missing required support/body coordinates."))
    else:
        endpoint_rep = endpoint_rep_rows.iloc[0]
        endpoint_start_frame = int(endpoint_rep["start_frame"])
        endpoint_end_frame = int(endpoint_rep["end_frame"])
        end_before = int(ENDPOINT_BLEND_CONFIG.get("end_window_before_frames", 2))
        end_window_start = endpoint_end_frame - end_before
        end_window_end = endpoint_end_frame
        end_mask = endpoint_frame_mask(corridor_df, end_window_start, end_window_end)
        blend_weights, blend_window = endpoint_blend_weights(corridor_df, endpoint_start_frame, ENDPOINT_BLEND_CONFIG)
        apply_mask = blend_weights > 0.0
        corridor_df = rv_copy_coordinate_family(corridor_df, blend_source_family, ENDPOINT_BLEND_FAMILY)

        strength = float(np.clip(float(ENDPOINT_BLEND_CONFIG.get("strength", 0.85)), 0.0, 1.0))
        max_z_shift = float(ENDPOINT_BLEND_CONFIG.get("max_z_shift_ratio", 0.12))
        report_only = bool(ENDPOINT_BLEND_CONFIG.get("report_only", False))
        width_guard_cfg = ENDPOINT_BLEND_CONFIG.get("support_width_no_worsen", {})
        width_guard_enabled = bool(width_guard_cfg.get("enabled", True))
        width_tolerance = float(width_guard_cfg.get("tolerance_ratio", 0.005))

        shift_by_landmark: dict[str, np.ndarray] = {}
        target_z_by_landmark: dict[str, float] = {}
        for landmark in support_landmarks:
            source_xyz = point_array(corridor_df, landmark, blend_source_family)
            target_z = endpoint_trimmed_median(source_xyz[end_mask, 2]) if np.any(end_mask) else np.nan
            target_z_by_landmark[landmark] = target_z
            current_xyz = point_array(corridor_df, landmark, ENDPOINT_BLEND_FAMILY)
            shift = np.zeros(len(corridor_df), dtype=float)
            if np.isfinite(target_z):
                raw_shift = (target_z - current_xyz[:, 2]) * strength * blend_weights
                shift = np.clip(raw_shift, -max_z_shift, max_z_shift)
                shift[~apply_mask] = 0.0
            shift_by_landmark[landmark] = shift

        width_rejected = np.zeros(len(corridor_df), dtype=bool)
        support_width_target = np.nan
        width_residual_before = np.full(len(corridor_df), np.nan)
        width_residual_after = np.full(len(corridor_df), np.nan)
        if width_guard_enabled and rv_family_has_coordinates(corridor_df, ENDPOINT_BLEND_FAMILY, support_pair):
            left_before = point_array(corridor_df, support_pair[0], ENDPOINT_BLEND_FAMILY)
            right_before = point_array(corridor_df, support_pair[1], ENDPOINT_BLEND_FAMILY)
            left_after = left_before.copy()
            right_after = right_before.copy()
            if support_pair[0] in shift_by_landmark:
                left_after[:, 2] = left_after[:, 2] + shift_by_landmark[support_pair[0]]
            if support_pair[1] in shift_by_landmark:
                right_after[:, 2] = right_after[:, 2] + shift_by_landmark[support_pair[1]]
            source_left = point_array(corridor_df, support_pair[0], blend_source_family)
            source_right = point_array(corridor_df, support_pair[1], blend_source_family)
            end_width = np.linalg.norm((source_right - source_left)[:, [0, 2]], axis=1)
            support_width_target = endpoint_trimmed_median(end_width[end_mask]) if np.any(end_mask) else np.nan
            if np.isfinite(support_width_target):
                width_before = np.linalg.norm((right_before - left_before)[:, [0, 2]], axis=1)
                width_after = np.linalg.norm((right_after - left_after)[:, [0, 2]], axis=1)
                width_residual_before = np.abs(width_before - support_width_target)
                width_residual_after = np.abs(width_after - support_width_target)
                width_rejected = apply_mask & (width_residual_after > width_residual_before + width_tolerance)
                for landmark in shift_by_landmark:
                    shift_by_landmark[landmark][width_rejected] = 0.0

        burden_rows = []
        for landmark in support_landmarks:
            before_xyz = point_array(corridor_df, landmark, ENDPOINT_BLEND_FAMILY)
            shift = shift_by_landmark[landmark]
            after_xyz = before_xyz.copy()
            after_xyz[:, 2] = after_xyz[:, 2] + shift
            if not report_only:
                set_point(corridor_df, landmark, ENDPOINT_BLEND_FAMILY, after_xyz)
            target_z = target_z_by_landmark.get(landmark, np.nan)
            z_residual_before = np.abs(target_z - before_xyz[:, 2]) if np.isfinite(target_z) else np.full(len(corridor_df), np.nan)
            z_residual_after = np.abs(target_z - after_xyz[:, 2]) if np.isfinite(target_z) else np.full(len(corridor_df), np.nan)
            burden_rows.append({
                "fit_component": "standing_endpoint_anchor_blend",
                "type": "bounded_pre_post_standing_anchor_blend",
                "status": "report_only" if report_only else "applied",
                "source_family": blend_source_family,
                "target_family": ENDPOINT_BLEND_FAMILY,
                "landmark": landmark,
                "target_z": target_z,
                "start_window": f"{blend_window['start_window_start']}-{blend_window['start_window_end']}",
                "ramp_window": f"{blend_window['ramp_start']}-{blend_window['ramp_end']}",
                "end_window": f"{end_window_start}-{end_window_end}",
                "support_width_target": support_width_target if landmark in support_pair else np.nan,
                "support_width_rejected_frame_ratio": float(np.nanmean(width_rejected[apply_mask])) if np.any(apply_mask) and landmark in support_pair else np.nan,
                "support_width_rejected_frames": rv_frame_label_list(corridor_df, width_rejected & apply_mask) if landmark in support_pair else "",
                "active_frame_ratio": float(np.nanmean(np.abs(shift[apply_mask]) > 1e-9)) if np.any(apply_mask) else np.nan,
                "median_abs_z_residual_before": rv_safe_median(z_residual_before[apply_mask]),
                "median_abs_z_residual_after": rv_safe_median(z_residual_after[apply_mask]),
                "median_abs_z_shift": rv_safe_median(np.abs(shift[apply_mask])),
                "p95_abs_z_shift": rv_safe_percentile(np.abs(shift[apply_mask]), 95),
                "max_abs_z_shift": float(np.nanmax(np.abs(shift[apply_mask]))) if np.any(apply_mask) else np.nan,
                "applied_frames": rv_frame_label_list(corridor_df, (np.abs(shift) > 1e-9) & apply_mask),
                "used_for_features_or_scores": bool(ENDPOINT_BLEND_CONFIG.get("reporting", {}).get("used_for_features_or_scores", False)),
                "confidence_when_applied": ENDPOINT_BLEND_CONFIG.get("reporting", {}).get("confidence_when_applied", "very_low"),
            })
        endpoint_blend_burden_df = pd.DataFrame(burden_rows)

        start_mask = endpoint_frame_mask(corridor_df, blend_window["start_window_start"], blend_window["start_window_end"])
        endpoint_blend_rows = []
        for family in [blend_source_family, ENDPOINT_BLEND_FAMILY]:
            endpoint_blend_rows.append(endpoint_support_metric_row(family, "start_window", blend_window["start_window_start"], blend_window["start_window_end"], start_mask, support_pair, support_suffixes, body_landmarks))
            endpoint_blend_rows.append(endpoint_support_metric_row(family, "end_window", end_window_start, end_window_end, end_mask, support_pair, support_suffixes, body_landmarks))
        endpoint_blend_window_df = pd.DataFrame(endpoint_blend_rows)
        before_delta = endpoint_delta_from_window_table(endpoint_blend_window_df, blend_source_family)
        after_delta = endpoint_delta_from_window_table(endpoint_blend_window_df, ENDPOINT_BLEND_FAMILY)
        endpoint_blend_delta_df = pd.DataFrame([endpoint_blend_improvement_row(before_delta, after_delta)])

        display(Markdown("### Bounded Pre/Post Standing Anchor Blend Burden"))
        display(endpoint_blend_burden_df)
        display(Markdown("### Endpoint Blend Delta Improvement"))
        display(endpoint_blend_delta_df)
        display(Markdown("### Endpoint Blend Window Metrics"))
        display(endpoint_blend_window_df)

        rep_mask = endpoint_frame_mask(corridor_df, endpoint_start_frame, endpoint_end_frame)
        rep_frames = corridor_df.loc[rep_mask, "frame"].astype(int).to_numpy()
        fig_endpoint_blend = make_subplots(
            rows=3,
            cols=1,
            shared_xaxes=True,
            subplot_titles=("support width", "body/support depth offset", "ankle model-depth z"),
            vertical_spacing=0.10,
        )
        for family, label, color, dash in [
            (blend_source_family, "bounded", "rgba(220,40,80,0.70)", "dash"),
            (ENDPOINT_BLEND_FAMILY, "endpoint blend", "rgba(0,170,170,0.95)", "solid"),
        ]:
            support_width, body_offset_z, left_ankle_z, right_ankle_z = endpoint_series_for_family(family, rep_mask, support_pair, body_landmarks)
            fig_endpoint_blend.add_trace(go.Scatter(x=rep_frames, y=support_width, mode="lines", name=f"{label} support width", line=dict(color=color, dash=dash, width=3)), row=1, col=1)
            fig_endpoint_blend.add_trace(go.Scatter(x=rep_frames, y=body_offset_z, mode="lines", name=f"{label} body-support z", line=dict(color=color, dash=dash, width=3)), row=2, col=1)
            fig_endpoint_blend.add_trace(go.Scatter(x=rep_frames, y=right_ankle_z, mode="lines", name=f"{label} right ankle z", line=dict(color=color, dash=dash, width=3)), row=3, col=1)
            fig_endpoint_blend.add_trace(go.Scatter(x=rep_frames, y=left_ankle_z, mode="lines", name=f"{label} left ankle z", line=dict(color=color, dash="dashdot", width=2)), row=3, col=1)
        for marker_frame in [endpoint_start_frame, endpoint_end_frame, blend_window["ramp_end"]]:
            fig_endpoint_blend.add_vline(x=marker_frame, line=dict(color="rgba(90,90,90,0.55)", dash="dot"))
        fig_endpoint_blend.update_layout(
            title=f"Rep {endpoint_rep_id} endpoint-limited blend vs bounded candidate",
            height=820,
            width=1100,
            legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0.0),
        )
        fig_endpoint_blend.update_xaxes(title_text="source frame", row=3, col=1)
        fig_endpoint_blend.update_yaxes(title_text="torso-length ratio")
        if review_show_visual("support_surface_playback") or review_show_visual("endpoint_blend_compare"):
            fig_endpoint_blend.show()

        if rv_family_has_coordinates(corridor_df, ENDPOINT_BLEND_FAMILY, ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle", "left_heel", "right_heel", "left_foot_index", "right_foot_index"]):
            start_row = corridor_df.loc[corridor_df["frame"].astype(int).eq(endpoint_start_frame)].iloc[0]
            end_row = corridor_df.loc[corridor_df["frame"].astype(int).eq(endpoint_end_frame)].iloc[0]
            overlay_landmarks = ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle", "left_heel", "right_heel", "left_foot_index", "right_foot_index"]
            overlay_connections = [
                ("left_hip", "left_knee"), ("left_knee", "left_ankle"), ("right_hip", "right_knee"), ("right_knee", "right_ankle"),
                ("left_hip", "right_hip"), ("left_ankle", "right_ankle"), ("left_heel", "left_foot_index"), ("right_heel", "right_foot_index"),
            ]

            def endpoint_blend_overlay_trace(row: pd.Series, label: str, color: str) -> list[go.Scatter3d]:
                lx, ly, lz = [], [], []
                for a, b in overlay_connections:
                    ax, ay, az = plot_xyz(row, a, ENDPOINT_BLEND_FAMILY)
                    bx, by, bz = plot_xyz(row, b, ENDPOINT_BLEND_FAMILY)
                    lx.extend([ax, bx, None])
                    ly.extend([ay, by, None])
                    lz.extend([az, bz, None])
                px, py, pz = [], [], []
                for landmark in overlay_landmarks:
                    x, y, z = plot_xyz(row, landmark, ENDPOINT_BLEND_FAMILY)
                    px.append(x)
                    py.append(y)
                    pz.append(z)
                return [
                    go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=6), name=f"{label} lower body"),
                    go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=4), name=f"{label} support joints"),
                ]

            fig_endpoint_blend_overlay = go.Figure()
            for trace in endpoint_blend_overlay_trace(start_row, f"start frame {endpoint_start_frame}", "rgba(230,130,40,0.95)"):
                fig_endpoint_blend_overlay.add_trace(trace)
            for trace in endpoint_blend_overlay_trace(end_row, f"end frame {endpoint_end_frame}", "rgba(0,170,170,0.95)"):
                fig_endpoint_blend_overlay.add_trace(trace)
            fig_endpoint_blend_overlay.update_layout(
                title=f"Rep {endpoint_rep_id} exact standing endpoints overlay ({ENDPOINT_BLEND_FAMILY})",
                scene=dict(
                    xaxis=dict(title="recording x"),
                    yaxis=dict(title="model depth z"),
                    zaxis=dict(title="recording height (-y)"),
                    aspectmode="data",
                    camera=COMPACT_SCENE_CAMERA,
                ),
                height=720,
                width=1050,
                showlegend=True,
            )
            if review_show_visual("support_surface_playback") or review_show_visual("endpoint_blend_compare"):
                fig_endpoint_blend_overlay.show()

        endpoint_play_start = int(blend_window["start_window_start"])
        endpoint_play_end = int(endpoint_end_frame)
        endpoint_play_mask = endpoint_frame_mask(corridor_df, endpoint_play_start, endpoint_play_end)
        endpoint_play_df = corridor_df.loc[endpoint_play_mask].copy()
        endpoint_play_norm_family = str(ENDPOINT_BLEND_CONFIG.get("playback_norm_family", BOUNDED_RV_SOURCE_FAMILY if "BOUNDED_RV_SOURCE_FAMILY" in globals() else "norm"))
        endpoint_play_output_family = ENDPOINT_BLEND_FAMILY
        endpoint_play_families = [endpoint_play_norm_family, endpoint_play_output_family]
        endpoint_play_required = list(LANDMARKS)

        if endpoint_play_df.empty or not all(rv_family_has_coordinates(endpoint_play_df, family, endpoint_play_required) for family in endpoint_play_families):
            display(Markdown("Endpoint blend playback is unavailable because required coordinate columns are missing."))
        else:
            def endpoint_play_has_point(row: pd.Series, landmark: str, family: str) -> bool:
                columns = [fcol(landmark, family, axis) for axis in ["x", "y", "z"]]
                return all(column in row.index and np.isfinite(float(row[column])) for column in columns)


            def endpoint_play_anchor(row: pd.Series) -> np.ndarray:
                points = []
                for landmark in support_pair:
                    if endpoint_play_has_point(row, landmark, ENDPOINT_BLEND_FAMILY):
                        points.append(np.asarray(plot_xyz(row, landmark, ENDPOINT_BLEND_FAMILY), dtype=float))
                if not points:
                    return np.zeros(3, dtype=float)
                anchor = np.nanmedian(np.vstack(points), axis=0)
                anchor[~np.isfinite(anchor)] = 0.0
                return anchor


            def endpoint_play_xyz(row: pd.Series, landmark: str, family: str) -> tuple[float, float, float]:
                xyz = np.asarray(plot_xyz(row, landmark, family), dtype=float)
                return tuple((xyz - endpoint_play_anchor(row)).tolist())


            def endpoint_play_skeleton_traces(row: pd.Series, family: str, label: str, color: str, width: int) -> list[go.Scatter3d]:
                px, py, pz = [], [], []
                for landmark in LANDMARKS:
                    x, y, z = endpoint_play_xyz(row, landmark, family)
                    px.append(x)
                    py.append(y)
                    pz.append(z)

                lx, ly, lz = [], [], []
                for a, b in CONNECTIONS:
                    ax, ay, az = endpoint_play_xyz(row, a, family)
                    bx, by, bz = endpoint_play_xyz(row, b, family)
                    lx.extend([ax, bx, None])
                    ly.extend([ay, by, None])
                    lz.extend([az, bz, None])

                return [
                    go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=width), name=f"{label} skeleton"),
                    go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=3), name=f"{label} joints"),
                ]


            def endpoint_play_traces(row: pd.Series) -> list[go.Scatter3d]:
                traces = []
                traces += endpoint_play_skeleton_traces(row, endpoint_play_norm_family, "norm input", "rgba(30,30,30,0.72)", 4)
                traces += endpoint_play_skeleton_traces(row, endpoint_play_output_family, "final endpoint-blend output", "rgba(0,170,170,0.96)", 6)
                traces.append(go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="rgba(30,30,30,0.85)", size=5), name="display anchor"))
                return traces


            def endpoint_play_ranges(df: pd.DataFrame, families: list[str]) -> tuple[list[float], list[float], list[float]]:
                xs, ys, zs = [], [], []
                for _, row in df.iterrows():
                    for family in families:
                        for landmark in LANDMARKS:
                            x, y, z = endpoint_play_xyz(row, landmark, family)
                            xs.append(x)
                            ys.append(y)
                            zs.append(z)
                return finite_range(xs, 0.15), finite_range(ys, 0.15), finite_range(zs, 0.15)


            def endpoint_play_axis_layout(title: str, axis_range: list[float]) -> dict:
                return {
                    "title": title,
                    "range": [float(axis_range[0]), float(axis_range[1])],
                    "autorange": False,
                    "showspikes": False,
                }


            def endpoint_play_aspect_ratio(ranges: tuple[list[float], list[float], list[float]]) -> dict[str, float]:
                spans = np.array([max(float(axis[1]) - float(axis[0]), 1e-6) for axis in ranges], dtype=float)
                max_span = float(np.nanmax(spans)) if np.isfinite(spans).any() else 1.0
                if not np.isfinite(max_span) or max_span <= 0.0:
                    max_span = 1.0
                ratio = spans / max_span
                return {"x": float(ratio[0]), "y": float(ratio[1]), "z": float(ratio[2])}


            endpoint_play_anim_df = endpoint_play_df.copy()
            endpoint_play_ranges_value = endpoint_play_ranges(endpoint_play_df, endpoint_play_families)
            endpoint_play_frame_duration_ms = max(10, int(1000 / max(float(globals().get("fps_estimate", PLAYBACK_PROFILE.get("fps", 30.0))) * max(float(PLAYBACK_PROFILE.get("speed", 1.0)), 1e-6), 1.0)))
            fig_endpoint_blend_play = go.Figure(data=endpoint_play_traces(endpoint_play_anim_df.iloc[0]))
            fig_endpoint_blend_play.frames = [
                go.Frame(data=endpoint_play_traces(row), name=str(int(row["frame"])))
                for _, row in endpoint_play_anim_df.iterrows()
            ]
            fig_endpoint_blend_play.update_layout(
                title=f"Rep {endpoint_rep_id}: norm input vs final endpoint-blend output",
                scene=dict(
                    xaxis=endpoint_play_axis_layout("recording x relative to endpoint-blend support anchor", endpoint_play_ranges_value[0]),
                    yaxis=endpoint_play_axis_layout("model depth z relative to endpoint-blend support anchor", endpoint_play_ranges_value[1]),
                    zaxis=endpoint_play_axis_layout("height relative to endpoint-blend support anchor", endpoint_play_ranges_value[2]),
                    aspectmode="manual",
                    aspectratio=endpoint_play_aspect_ratio(endpoint_play_ranges_value),
                    camera=COMPACT_SCENE_CAMERA,
                ),
                height=720,
                width=1050,
                showlegend=True,
                sliders=[{
                    "active": 0,
                    "currentvalue": {"prefix": "Source frame: "},
                    "pad": {"t": 50},
                    "steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}}], "label": frame.name, "method": "animate"} for frame in fig_endpoint_blend_play.frames],
                }],
                updatemenus=[
                    {
                        "type": "buttons",
                        "showactive": False,
                        "x": 0.0,
                        "y": 1.0,
                        "xanchor": "left",
                        "yanchor": "top",
                        "buttons": [
                            {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": endpoint_play_frame_duration_ms, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True, "mode": "immediate"}]},
                            {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
                        ],
                    },
                    {
                        "type": "buttons",
                        "showactive": True,
                        "x": 0.0,
                        "y": 0.88,
                        "xanchor": "left",
                        "yanchor": "top",
                        "buttons": [
                            {"label": "3D", "method": "relayout", "args": [{"scene.camera": COMPACT_SCENE_CAMERA}]},
                            {"label": "Top-down x-z", "method": "relayout", "args": [{"scene.camera": COMPACT_TOP_DOWN_CAMERA}]},
                        ],
                    },
                ],
            )
            if "add_floor_axis_guide" in globals():
                add_floor_axis_guide(fig_endpoint_blend_play, endpoint_play_ranges_value)
            if review_show_visual("endpoint_blend_playback_inline"):
                fig_endpoint_blend_play.show()


### 11.8 Whole-video planted support temporal memory

This cell copies the endpoint-blend candidate into the final corrected-3D-hypothesis family, applies capped low-strength support-landmark `z` shifts across the whole recording, and rebuilds the final `norm` versus final-output playback for the full video. The candidate remains score-excluded.


In [ ]:
SUPPORT_MEMORY_CONFIG = CORRIDOR_CONFIG.get("planted_support_temporal_memory", {})
SUPPORT_MEMORY_FAMILY = str(SUPPORT_MEMORY_CONFIG.get("family", f"{ENDPOINT_BLEND_FAMILY}_support_memory"))
SUPPORT_MEMORY_AUDIT_LABEL = "support_memory_final"
FINAL_REVIEW_OUTPUT_FAMILY = ENDPOINT_BLEND_FAMILY
FINAL_REVIEW_AUDIT_LABEL = "endpoint_blend"
support_memory_burden_df = pd.DataFrame()
support_memory_frame_df = pd.DataFrame()


def support_memory_trimmed_median(values: np.ndarray, trim_quantile: float = 0.10) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    trim = int(np.floor(arr.size * float(np.clip(trim_quantile, 0.0, 0.45))))
    if trim > 0 and arr.size > trim * 2:
        arr = np.sort(arr)[trim:-trim]
    return float(np.nanmedian(arr)) if arr.size else np.nan


def support_memory_visibility_mask(df: pd.DataFrame, landmark: str, min_visibility: float) -> tuple[np.ndarray, str]:
    for col in [f"{landmark}_visibility", f"{landmark}_vis", f"{landmark}_presence"]:
        if col in df.columns:
            values = df[col].astype(float).to_numpy()
            return np.isfinite(values) & (values >= float(min_visibility)), col
    return np.ones(len(df), dtype=bool), "visibility_not_available"


def support_memory_reference_mask(
    df: pd.DataFrame,
    landmark: str,
    family: str,
    min_visibility: float,
    max_jump_ratio: float,
    min_reference_frames: int,
) -> tuple[np.ndarray, str, str]:
    xyz = point_array(df, landmark, family)
    finite = np.isfinite(xyz[:, 2])
    visible, visibility_source = support_memory_visibility_mask(df, landmark, min_visibility)
    jump = np.full(len(df), np.nan)
    if len(df) > 1:
        jump[1:] = np.linalg.norm(np.diff(xyz[:, [0, 2]], axis=0), axis=1)
    stable = np.isnan(jump) | (jump <= float(max_jump_ratio))
    mask = finite & visible & stable
    status = "stable_visible_reference"
    if int(mask.sum()) < int(min_reference_frames):
        mask = finite & visible
        status = "visible_reference_fallback"
    if int(mask.sum()) < int(min_reference_frames):
        mask = finite
        status = "finite_reference_fallback"
    return mask, status, visibility_source


def support_memory_frame_preview(df: pd.DataFrame, mask: np.ndarray, limit: int = 18) -> str:
    frames = df.loc[mask, "frame"].astype(int).tolist()
    if len(frames) <= limit:
        return ",".join(str(frame) for frame in frames)
    head = ",".join(str(frame) for frame in frames[: limit // 2])
    tail = ",".join(str(frame) for frame in frames[-(limit // 2) :])
    return f"{head},...,{tail}"


SUPPORT_MEMORY_BEND_PLANES = {
    "xz_recording_depth": (0, 2),
    "yz_height_depth": (1, 2),
    "xy_recording_plane": (0, 1),
}


def support_memory_plane_axes(plane_id: str) -> tuple[int, int]:
    return SUPPORT_MEMORY_BEND_PLANES.get(str(plane_id), (0, 2))


def support_memory_signed_bend_plane(
    hip: np.ndarray,
    knee: np.ndarray,
    ankle: np.ndarray,
    axes: tuple[int, int],
    epsilon: float,
) -> tuple[np.ndarray, np.ndarray]:
    h = hip[:, list(axes)]
    k = knee[:, list(axes)]
    a = ankle[:, list(axes)]
    base = a - h
    bend = k - h
    signed_area = bend[:, 0] * base[:, 1] - bend[:, 1] * base[:, 0]
    base_len = np.linalg.norm(base, axis=1)
    signed_distance = np.full(len(base), np.nan)
    valid = base_len > 1e-9
    signed_distance[valid] = signed_area[valid] / base_len[valid]
    sign = np.sign(signed_distance)
    sign[np.abs(signed_distance) <= float(epsilon)] = 0.0
    return signed_distance, sign


if not bool(SUPPORT_MEMORY_CONFIG.get("enabled", False)):
    display(Markdown("`planted_support_temporal_memory` is disabled in the run profile; final playback keeps the endpoint-blend candidate."))
else:
    memory_source_family = str(SUPPORT_MEMORY_CONFIG.get("source_family", ENDPOINT_BLEND_FAMILY))
    memory_sides = [str(item) for item in SUPPORT_MEMORY_CONFIG.get("sides", ["left", "right"])]
    memory_suffixes = [str(item) for item in SUPPORT_MEMORY_CONFIG.get("support_landmark_suffixes", ["ankle", "heel", "foot_index"])]
    memory_support_landmarks = [f"{side}_{suffix}" for side in memory_sides for suffix in memory_suffixes]
    memory_support_pair = [str(item) for item in SUPPORT_MEMORY_CONFIG.get("support_pair", ["left_ankle", "right_ankle"])]
    required_memory_landmarks = sorted(set(memory_support_landmarks + memory_support_pair))
    if not rv_family_has_coordinates(corridor_df, memory_source_family, required_memory_landmarks):
        display(Markdown(f"Planted support temporal memory is unavailable because `{memory_source_family}` lacks required support coordinates."))
    else:
        corridor_df = rv_copy_coordinate_family(corridor_df, memory_source_family, SUPPORT_MEMORY_FAMILY)
        reference_cfg = SUPPORT_MEMORY_CONFIG.get("reference", {})
        reporting_cfg = SUPPORT_MEMORY_CONFIG.get("reporting", {})
        apply_mask = np.ones(len(corridor_df), dtype=bool)
        strength = float(np.clip(float(SUPPORT_MEMORY_CONFIG.get("strength", 0.35)), 0.0, 1.0))
        max_z_shift = float(SUPPORT_MEMORY_CONFIG.get("max_z_shift_ratio", 0.04))
        trim_q = float(reference_cfg.get("trim_quantile", 0.10))
        min_visibility = float(reference_cfg.get("min_visibility", 0.55))
        max_jump = float(reference_cfg.get("max_frame_to_frame_jump_ratio", 0.03))
        min_ref = int(reference_cfg.get("min_reference_frames", 12))

        shift_by_landmark: dict[str, np.ndarray] = {}
        target_z_by_landmark: dict[str, float] = {}
        reference_info: dict[str, dict] = {}
        for landmark in memory_support_landmarks:
            ref_mask, ref_status, vis_source = support_memory_reference_mask(corridor_df, landmark, memory_source_family, min_visibility, max_jump, min_ref)
            source_xyz = point_array(corridor_df, landmark, memory_source_family)
            current_xyz = point_array(corridor_df, landmark, SUPPORT_MEMORY_FAMILY)
            target_z = support_memory_trimmed_median(source_xyz[ref_mask, 2], trim_q)
            target_z_by_landmark[landmark] = target_z
            active = apply_mask & np.isfinite(current_xyz[:, 2]) & np.isfinite(target_z)
            shift = np.zeros(len(corridor_df), dtype=float)
            if np.any(active):
                raw_shift = (float(target_z) - current_xyz[:, 2]) * strength
                shift = np.clip(raw_shift, -max_z_shift, max_z_shift)
                shift[~active] = 0.0
            shift_by_landmark[landmark] = shift
            reference_info[landmark] = {
                "reference_status": ref_status,
                "visibility_source": vis_source,
                "reference_frame_count": int(ref_mask.sum()),
                "reference_frames_preview": support_memory_frame_preview(corridor_df, ref_mask),
            }

        width_guard_cfg = SUPPORT_MEMORY_CONFIG.get("support_width_no_worsen", {})
        width_tolerance = float(width_guard_cfg.get("tolerance_ratio", 0.005))
        width_rejected = np.zeros(len(corridor_df), dtype=bool)
        support_width_target = np.nan
        width_before = np.full(len(corridor_df), np.nan)
        width_after = np.full(len(corridor_df), np.nan)
        if bool(width_guard_cfg.get("enabled", True)) and rv_family_has_coordinates(corridor_df, SUPPORT_MEMORY_FAMILY, memory_support_pair):
            left_before = point_array(corridor_df, memory_support_pair[0], SUPPORT_MEMORY_FAMILY)
            right_before = point_array(corridor_df, memory_support_pair[1], SUPPORT_MEMORY_FAMILY)
            left_after = left_before.copy()
            right_after = right_before.copy()
            if memory_support_pair[0] in shift_by_landmark:
                left_after[:, 2] = left_after[:, 2] + shift_by_landmark[memory_support_pair[0]]
            if memory_support_pair[1] in shift_by_landmark:
                right_after[:, 2] = right_after[:, 2] + shift_by_landmark[memory_support_pair[1]]
            source_left = point_array(corridor_df, memory_support_pair[0], memory_source_family)
            source_right = point_array(corridor_df, memory_support_pair[1], memory_source_family)
            source_width = np.linalg.norm((source_right - source_left)[:, [0, 2]], axis=1)
            support_width_target = support_memory_trimmed_median(source_width, trim_q)
            if np.isfinite(support_width_target):
                width_before = np.linalg.norm((right_before - left_before)[:, [0, 2]], axis=1)
                width_after = np.linalg.norm((right_after - left_after)[:, [0, 2]], axis=1)
                before_residual = np.abs(width_before - support_width_target)
                after_residual = np.abs(width_after - support_width_target)
                width_rejected = apply_mask & (after_residual > before_residual + width_tolerance)
                for landmark in shift_by_landmark:
                    shift_by_landmark[landmark][width_rejected] = 0.0


        bend_guard_cfg = SUPPORT_MEMORY_CONFIG.get("bend_side_no_worsen", {})
        bend_rejected_by_side: dict[str, np.ndarray] = {side: np.zeros(len(corridor_df), dtype=bool) for side in memory_sides}
        if bool(bend_guard_cfg.get("enabled", True)):
            bend_ref_family = str(bend_guard_cfg.get("reference_family", RV_FIT_SOURCE_FAMILY if "RV_FIT_SOURCE_FAMILY" in globals() else "norm"))
            bend_plane_ids = [str(item) for item in bend_guard_cfg.get("planes", ["xz_recording_depth", "yz_height_depth", "xy_recording_plane"])]
            bend_epsilon = float(bend_guard_cfg.get("epsilon", 0.01))
            for chain in RV_FIT_CONFIG.get("chains", []):
                side = str(chain.get("id", "")).split("_", 1)[0]
                if side not in bend_rejected_by_side:
                    continue
                hip_name = str(chain["proximal"])
                knee_name = str(chain["joint"])
                ankle_name = str(chain["distal"])
                if not rv_family_has_coordinates(corridor_df, bend_ref_family, [hip_name, knee_name, ankle_name]):
                    continue
                if not rv_family_has_coordinates(corridor_df, SUPPORT_MEMORY_FAMILY, [hip_name, knee_name, ankle_name]):
                    continue
                ref_hip = point_array(corridor_df, hip_name, bend_ref_family)
                ref_knee = point_array(corridor_df, knee_name, bend_ref_family)
                ref_ankle = point_array(corridor_df, ankle_name, bend_ref_family)
                cand_hip = point_array(corridor_df, hip_name, SUPPORT_MEMORY_FAMILY)
                cand_knee = point_array(corridor_df, knee_name, SUPPORT_MEMORY_FAMILY)
                cand_ankle = point_array(corridor_df, ankle_name, SUPPORT_MEMORY_FAMILY).copy()
                if ankle_name in shift_by_landmark:
                    cand_ankle[:, 2] = cand_ankle[:, 2] + shift_by_landmark[ankle_name]
                for plane_id in bend_plane_ids:
                    axes = support_memory_plane_axes(plane_id)
                    _, ref_sign = support_memory_signed_bend_plane(ref_hip, ref_knee, ref_ankle, axes, bend_epsilon)
                    _, cand_sign = support_memory_signed_bend_plane(cand_hip, cand_knee, cand_ankle, axes, bend_epsilon)
                    flip = (
                        apply_mask
                        & np.isfinite(ref_sign)
                        & np.isfinite(cand_sign)
                        & (ref_sign != 0.0)
                        & (cand_sign != 0.0)
                        & (cand_sign != ref_sign)
                    )
                    bend_rejected_by_side[side] |= flip
            for side, reject_mask in bend_rejected_by_side.items():
                for suffix in memory_suffixes:
                    landmark = f"{side}_{suffix}"
                    if landmark in shift_by_landmark:
                        shift_by_landmark[landmark][reject_mask] = 0.0

        burden_rows = []
        for landmark in memory_support_landmarks:
            before_xyz = point_array(corridor_df, landmark, SUPPORT_MEMORY_FAMILY)
            target_z = target_z_by_landmark.get(landmark, np.nan)
            shift = shift_by_landmark.get(landmark, np.zeros(len(corridor_df), dtype=float))
            after_xyz = before_xyz.copy()
            after_xyz[:, 2] = after_xyz[:, 2] + shift
            set_point(corridor_df, landmark, SUPPORT_MEMORY_FAMILY, after_xyz)
            applied_mask = apply_mask & (np.abs(shift) > 1e-9)
            z_residual_before = np.abs(target_z - before_xyz[:, 2]) if np.isfinite(target_z) else np.full(len(corridor_df), np.nan)
            z_residual_after = np.abs(target_z - after_xyz[:, 2]) if np.isfinite(target_z) else np.full(len(corridor_df), np.nan)
            ref = reference_info.get(landmark, {})
            burden_rows.append({
                "fit_component": "whole_video_planted_support_temporal_memory",
                "type": "planted_support_temporal_memory",
                "status": "applied_score_excluded_candidate",
                "source_family": memory_source_family,
                "target_family": SUPPORT_MEMORY_FAMILY,
                "landmark": landmark,
                "target_z": target_z,
                "reference_status": ref.get("reference_status", "unknown"),
                "visibility_source": ref.get("visibility_source", "unknown"),
                "reference_frame_count": ref.get("reference_frame_count", 0),
                "reference_frames_preview": ref.get("reference_frames_preview", ""),
                "active_frame_count": int(apply_mask.sum()),
                "applied_frame_count": int(applied_mask.sum()),
                "applied_frame_ratio": float(np.nanmean(applied_mask[apply_mask])) if np.any(apply_mask) else np.nan,
                "rejected_frame_count": int(width_rejected.sum()) if landmark in memory_support_pair else np.nan,
                "bend_side_rejected_frame_count": int(bend_rejected_by_side.get(str(landmark).split("_", 1)[0], np.zeros(len(corridor_df), dtype=bool)).sum()),
                "bend_side_rejected_frame_ratio": float(np.nanmean(bend_rejected_by_side.get(str(landmark).split("_", 1)[0], np.zeros(len(corridor_df), dtype=bool))[apply_mask])) if np.any(apply_mask) else np.nan,
                "support_width_target": support_width_target if landmark in memory_support_pair else np.nan,
                "support_width_rejected_frame_ratio": float(np.nanmean(width_rejected[apply_mask])) if np.any(apply_mask) and landmark in memory_support_pair else np.nan,
                "median_abs_z_residual_before": rv_safe_median(z_residual_before[apply_mask]),
                "median_abs_z_residual_after": rv_safe_median(z_residual_after[apply_mask]),
                "median_abs_z_shift": rv_safe_median(np.abs(shift[apply_mask])),
                "p95_abs_z_shift": rv_safe_percentile(np.abs(shift[apply_mask]), 95),
                "max_abs_z_shift": float(np.nanmax(np.abs(shift[apply_mask]))) if np.any(apply_mask) else np.nan,
                "applied_frames_preview": support_memory_frame_preview(corridor_df, applied_mask),
                "used_for_features_or_scores": bool(reporting_cfg.get("used_for_features_or_scores", False)),
                "confidence_when_applied": reporting_cfg.get("confidence_when_applied", "very_low"),
            })
        support_memory_burden_df = pd.DataFrame(burden_rows)
        for col in ["target_z", "support_width_target", "support_width_rejected_frame_ratio", "bend_side_rejected_frame_ratio", "median_abs_z_residual_before", "median_abs_z_residual_after", "median_abs_z_shift", "p95_abs_z_shift", "max_abs_z_shift"]:
            if col in support_memory_burden_df.columns:
                support_memory_burden_df[col] = support_memory_burden_df[col].astype(float).round(6)

        support_memory_frame_df = pd.DataFrame({
            "frame": corridor_df["frame"].astype(int),
            "support_width_target": support_width_target,
            "support_width_before": width_before,
            "support_width_after": width_after,
            "support_width_residual_before": np.abs(width_before - support_width_target),
            "support_width_residual_after": np.abs(width_after - support_width_target),
            "support_width_rejected": width_rejected,
        })
        for landmark in memory_support_pair:
            if landmark in shift_by_landmark:
                support_memory_frame_df[f"{landmark}_z_shift"] = shift_by_landmark[landmark]

        FINAL_REVIEW_OUTPUT_FAMILY = SUPPORT_MEMORY_FAMILY
        FINAL_REVIEW_AUDIT_LABEL = SUPPORT_MEMORY_AUDIT_LABEL
        display(Markdown("### Whole-Video Planted Support Temporal Memory Burden"))
        support_memory_burden_display_columns = [
            "landmark",
            "reference_status",
            "visibility_source",
            "reference_frame_count",
            "active_frame_count",
            "applied_frame_count",
            "applied_frame_ratio",
            "rejected_frame_count",
            "support_width_rejected_frame_ratio",
            "bend_side_rejected_frame_count",
            "bend_side_rejected_frame_ratio",
            "median_abs_z_residual_before",
            "median_abs_z_residual_after",
            "median_abs_z_shift",
            "p95_abs_z_shift",
            "max_abs_z_shift",
            "used_for_features_or_scores",
            "confidence_when_applied",
        ]
        display(support_memory_burden_df[[col for col in support_memory_burden_display_columns if col in support_memory_burden_df.columns]])

        fig_support_memory = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("bilateral ankle support-width residual", "primary support-landmark z shift"), vertical_spacing=0.12)
        fig_support_memory.add_trace(go.Scatter(x=support_memory_frame_df["frame"], y=support_memory_frame_df["support_width_residual_before"], mode="lines", name="before width residual", line=dict(color="rgba(220,40,80,0.70)", dash="dash", width=2)), row=1, col=1)
        fig_support_memory.add_trace(go.Scatter(x=support_memory_frame_df["frame"], y=support_memory_frame_df["support_width_residual_after"], mode="lines", name="after width residual", line=dict(color="rgba(0,170,170,0.95)", width=3)), row=1, col=1)
        for landmark, color in [(memory_support_pair[0], "rgba(230,130,40,0.90)"), (memory_support_pair[1], "rgba(40,110,220,0.90)")]:
            col = f"{landmark}_z_shift"
            if col in support_memory_frame_df.columns:
                fig_support_memory.add_trace(go.Scatter(x=support_memory_frame_df["frame"], y=support_memory_frame_df[col], mode="lines", name=f"{landmark} z shift", line=dict(color=color, width=2)), row=2, col=1)
        fig_support_memory.update_layout(title="Whole-video planted support memory burden (report-only)", height=620, width=1100, legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0.0))
        fig_support_memory.update_yaxes(title_text="torso-length ratio", row=1, col=1)
        fig_support_memory.update_yaxes(title_text="torso-length ratio", row=2, col=1)
        fig_support_memory.update_xaxes(title_text="source frame", row=2, col=1)
        if review_show_visual("support_memory_burden") or review_show_visual("final_playback"):
            fig_support_memory.show()


endpoint_play_norm_family = str(SUPPORT_MEMORY_CONFIG.get("playback_norm_family", ENDPOINT_BLEND_CONFIG.get("playback_norm_family", BOUNDED_RV_SOURCE_FAMILY if "BOUNDED_RV_SOURCE_FAMILY" in globals() else "norm")))
endpoint_play_output_family = FINAL_REVIEW_OUTPUT_FAMILY
endpoint_play_families = [endpoint_play_norm_family, endpoint_play_output_family]
endpoint_play_required = list(LANDMARKS)
endpoint_play_start = int(corridor_df["frame"].astype(int).min())
endpoint_play_end = int(corridor_df["frame"].astype(int).max())
endpoint_play_mask = endpoint_frame_mask(corridor_df, endpoint_play_start, endpoint_play_end)
endpoint_play_df = corridor_df.loc[endpoint_play_mask].copy()
play_support_pair = [str(item) for item in SUPPORT_MEMORY_CONFIG.get("support_pair", ENDPOINT_BLEND_CONFIG.get("support_pair", ["left_ankle", "right_ankle"]))]

if endpoint_play_df.empty or not all(rv_family_has_coordinates(endpoint_play_df, family, endpoint_play_required) for family in endpoint_play_families):
    display(Markdown("Whole-video final playback is unavailable because required coordinate columns are missing."))
else:
    def endpoint_play_has_point(row: pd.Series, landmark: str, family: str) -> bool:
        columns = [fcol(landmark, family, axis) for axis in ["x", "y", "z"]]
        return all(column in row.index and np.isfinite(float(row[column])) for column in columns)


    def endpoint_play_anchor(row: pd.Series) -> np.ndarray:
        points = []
        for landmark in play_support_pair:
            if endpoint_play_has_point(row, landmark, endpoint_play_output_family):
                points.append(np.asarray(plot_xyz(row, landmark, endpoint_play_output_family), dtype=float))
        if not points:
            return np.zeros(3, dtype=float)
        anchor = np.nanmedian(np.vstack(points), axis=0)
        anchor[~np.isfinite(anchor)] = 0.0
        return anchor


    def endpoint_play_xyz(row: pd.Series, landmark: str, family: str) -> tuple[float, float, float]:
        xyz = np.asarray(plot_xyz(row, landmark, family), dtype=float)
        return tuple((xyz - endpoint_play_anchor(row)).tolist())


    def endpoint_play_skeleton_traces(row: pd.Series, family: str, label: str, color: str, width: int) -> list[go.Scatter3d]:
        px, py, pz = [], [], []
        for landmark in LANDMARKS:
            x, y, z = endpoint_play_xyz(row, landmark, family)
            px.append(x); py.append(y); pz.append(z)
        lx, ly, lz = [], [], []
        for a, b in CONNECTIONS:
            ax, ay, az = endpoint_play_xyz(row, a, family)
            bx, by, bz = endpoint_play_xyz(row, b, family)
            lx.extend([ax, bx, None]); ly.extend([ay, by, None]); lz.extend([az, bz, None])
        return [
            go.Scatter3d(x=lx, y=ly, z=lz, mode="lines", line=dict(color=color, width=width), name=f"{label} skeleton"),
            go.Scatter3d(x=px, y=py, z=pz, mode="markers", marker=dict(color=color, size=3), name=f"{label} joints"),
        ]


    def endpoint_play_traces(row: pd.Series) -> list[go.Scatter3d]:
        return (
            endpoint_play_skeleton_traces(row, endpoint_play_norm_family, "norm input", "rgba(30,30,30,0.72)", 4)
            + endpoint_play_skeleton_traces(row, endpoint_play_output_family, "final score-excluded candidate", "rgba(0,170,170,0.96)", 6)
            + [go.Scatter3d(x=[0.0], y=[0.0], z=[0.0], mode="markers", marker=dict(color="rgba(30,30,30,0.85)", size=5), name="display anchor")]
        )


    def endpoint_play_range_from_values(values: list[float], cfg: dict) -> list[float]:
        arr = np.asarray(values, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            return [-1.0, 1.0]
        low_q = float(cfg.get("robust_percentile_low", 0.0))
        high_q = float(cfg.get("robust_percentile_high", 100.0))
        if 0.0 <= low_q < high_q <= 100.0 and (low_q > 0.0 or high_q < 100.0):
            low, high = np.nanpercentile(arr, [low_q, high_q])
        else:
            low, high = float(np.nanmin(arr)), float(np.nanmax(arr))
        if not np.isfinite(low) or not np.isfinite(high) or abs(high - low) < 1e-9:
            center = float(np.nanmedian(arr)) if arr.size else 0.0
            low, high = center - 1.0, center + 1.0
        pad = max(abs(high - low) * float(cfg.get("padding_ratio", 0.15)), 0.15)
        return [float(low - pad), float(high + pad)]


    def endpoint_play_ranges(df: pd.DataFrame, families: list[str], cfg: dict) -> tuple[list[float], list[float], list[float]]:
        xs, ys, zs = [], [], []
        for _, row in df.iterrows():
            for family in families:
                for landmark in LANDMARKS:
                    x, y, z = endpoint_play_xyz(row, landmark, family)
                    xs.append(x); ys.append(y); zs.append(z)
        return endpoint_play_range_from_values(xs, cfg), endpoint_play_range_from_values(ys, cfg), endpoint_play_range_from_values(zs, cfg)


    def endpoint_play_axis_layout(title: str, axis_range: list[float]) -> dict:
        return {"title": title, "range": [float(axis_range[0]), float(axis_range[1])], "autorange": False, "showspikes": False}


    def endpoint_play_aspect_ratio(ranges: tuple[list[float], list[float], list[float]]) -> dict[str, float]:
        spans = np.array([max(float(axis[1]) - float(axis[0]), 1e-6) for axis in ranges], dtype=float)
        max_span = float(np.nanmax(spans)) if np.isfinite(spans).any() else 1.0
        if not np.isfinite(max_span) or max_span <= 0.0:
            max_span = 1.0
        ratio = spans / max_span
        return {"x": float(ratio[0]), "y": float(ratio[1]), "z": float(ratio[2])}


    endpoint_play_view_cfg = SUPPORT_MEMORY_CONFIG.get("playback_view", {})
    endpoint_play_view_df = endpoint_play_df
    if str(endpoint_play_view_cfg.get("axis_range_source", "whole_video")) == "annotated_reps":
        view_rep_start = int(endpoint_play_view_cfg.get("rep_id_start", 1))
        view_rep_end = int(endpoint_play_view_cfg.get("rep_id_end", view_rep_start))
        view_rep_rows = ann_df[(ann_df["segment_type"].eq("rep")) & (ann_df["rep_id"].astype("Int64") >= view_rep_start) & (ann_df["rep_id"].astype("Int64") <= view_rep_end)]
        if not view_rep_rows.empty:
            view_start = int(view_rep_rows["start_frame"].min())
            view_end = int(view_rep_rows["end_frame"].max())
            view_mask = endpoint_play_df["frame"].astype(int).between(view_start, view_end).to_numpy()
            if np.any(view_mask):
                endpoint_play_view_df = endpoint_play_df.loc[view_mask].copy()
                display(Markdown(f"Final playback axis range source: reps {view_rep_start}-{view_rep_end} ({view_start}-{view_end}); playback sequence remains {endpoint_play_start}-{endpoint_play_end}."))
    endpoint_play_stride = max(1, int(PLAYBACK_PROFILE.get("stride", 1)))
    endpoint_play_anim_df = endpoint_play_df.iloc[::endpoint_play_stride].copy()
    if int(endpoint_play_anim_df.iloc[-1]["frame"]) != int(endpoint_play_df.iloc[-1]["frame"]):
        endpoint_play_anim_df = pd.concat([endpoint_play_anim_df, endpoint_play_df.tail(1)], ignore_index=True)
    endpoint_play_ranges_value = endpoint_play_ranges(endpoint_play_view_df, endpoint_play_families, endpoint_play_view_cfg)
    endpoint_play_fps = float(globals().get("fps_estimate", PLAYBACK_PROFILE.get("fps", 30.0)))
    endpoint_play_speed = float(PLAYBACK_PROFILE.get("speed", 1.0))
    endpoint_play_frame_duration_ms = max(10, int((1000.0 * endpoint_play_stride) / max(endpoint_play_fps * endpoint_play_speed, 1e-6)))
    fig_endpoint_blend_play = go.Figure(data=endpoint_play_traces(endpoint_play_anim_df.iloc[0]))
    fig_endpoint_blend_play.frames = [go.Frame(data=endpoint_play_traces(row), name=str(int(row["frame"]))) for _, row in endpoint_play_anim_df.iterrows()]
    fig_endpoint_blend_play.update_layout(
        title="Whole video: norm input vs final score-excluded candidate",
        scene=dict(
            xaxis=endpoint_play_axis_layout("recording x relative to final support anchor", endpoint_play_ranges_value[0]),
            yaxis=endpoint_play_axis_layout("model depth z relative to final support anchor", endpoint_play_ranges_value[1]),
            zaxis=endpoint_play_axis_layout("height relative to final support anchor", endpoint_play_ranges_value[2]),
            aspectmode="manual",
            aspectratio=endpoint_play_aspect_ratio(endpoint_play_ranges_value),
            camera=COMPACT_SCENE_CAMERA,
        ),
        height=720,
        width=1050,
        showlegend=True,
        sliders=[{"active": 0, "currentvalue": {"prefix": "Source frame: "}, "pad": {"t": 50}, "steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}}], "label": frame.name, "method": "animate"} for frame in fig_endpoint_blend_play.frames]}],
        updatemenus=[
            {"type": "buttons", "showactive": False, "x": 0.0, "y": 1.0, "xanchor": "left", "yanchor": "top", "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": endpoint_play_frame_duration_ms, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True, "mode": "immediate"}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
            ]},
            {"type": "buttons", "showactive": True, "x": 0.0, "y": 0.88, "xanchor": "left", "yanchor": "top", "buttons": [
                {"label": "3D", "method": "relayout", "args": [{"scene.camera": COMPACT_SCENE_CAMERA}]},
                {"label": "Top-down x-z", "method": "relayout", "args": [{"scene.camera": COMPACT_TOP_DOWN_CAMERA}]},
            ]},
        ],
    )
    if "add_floor_axis_guide" in globals():
        add_floor_axis_guide(fig_endpoint_blend_play, endpoint_play_ranges_value)
    if review_show_visual("endpoint_blend_playback_inline"):
        fig_endpoint_blend_play.show()


## 12 Frame-Window Chain Feasibility Audit

Review-only audit for frames 115-126. It checks whether the knee reversal around frames 119-123 is created by support placement, hip-ankle feasibility, or the bounded candidate solver.


In [ ]:

AUDIT_FRAME_START = 115
AUDIT_FRAME_END = 126
AUDIT_SUPPORT_FAMILY = "rv_support_context_only"
AUDIT_FOCUS_FRAMES = list(range(119, 124))


def audit_soft_bend_epsilon(default: float = 0.01) -> float:
    values = [float(default)]
    for cfg in [
        RV_FIT_CONFIG.get("solver", {}).get("bend_side_guard", {}),
        BOUNDED_RV_CONFIG.get("solver", {}).get("bend_side_guard", {}) if "BOUNDED_RV_CONFIG" in globals() else {},
    ]:
        if bool(cfg.get("enabled", False)):
            values.append(float(cfg.get("soft_flip_tolerance_ratio", cfg.get("near_neutral_tolerance_ratio", default))))
    return max(values)


AUDIT_BEND_EPS = audit_soft_bend_epsilon()


def audit_trimmed_median(values: np.ndarray, trim_quantile: float = 0.10) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return float("nan")
    trim_quantile = float(np.clip(trim_quantile, 0.0, 0.45))
    if trim_quantile > 0 and finite.size >= 3:
        lo, hi = np.nanquantile(finite, [trim_quantile, 1.0 - trim_quantile])
        trimmed = finite[(finite >= lo) & (finite <= hi)]
        if trimmed.size:
            finite = trimmed
    return float(np.nanmedian(finite))


def audit_family_available(family: str) -> bool:
    required = ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"]
    return rv_family_has_coordinates(corridor_df, family, required)


def audit_signed_bend_xz(hip: np.ndarray, knee: np.ndarray, ankle: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    base = ankle[:, [0, 2]] - hip[:, [0, 2]]
    knee_vec = knee[:, [0, 2]] - hip[:, [0, 2]]
    base_len = np.linalg.norm(base, axis=1)
    signed_area = base[:, 0] * knee_vec[:, 1] - base[:, 1] * knee_vec[:, 0]
    signed_distance = np.full(len(base), np.nan)
    valid = base_len > 1e-9
    signed_distance[valid] = signed_area[valid] / base_len[valid]
    sign = np.sign(signed_distance)
    sign[np.abs(signed_distance) <= AUDIT_BEND_EPS] = 0.0
    return signed_distance, sign


def audit_link_metrics(hip: np.ndarray, knee: np.ndarray, ankle: np.ndarray, thigh_target: float, shank_target: float) -> dict[str, np.ndarray]:
    thigh_length = np.linalg.norm(knee - hip, axis=1)
    shank_length = np.linalg.norm(ankle - knee, axis=1)
    hip_ankle_distance = np.linalg.norm(ankle - hip, axis=1)
    hip_knee_xy = np.linalg.norm((knee - hip)[:, :2], axis=1)
    knee_ankle_xy = np.linalg.norm((ankle - knee)[:, :2], axis=1)
    finite_targets = np.isfinite(thigh_target) and np.isfinite(shank_target) and thigh_target > 0 and shank_target > 0
    triangle_feasible = (
        finite_targets
        & np.isfinite(hip_ankle_distance)
        & (hip_ankle_distance <= thigh_target + shank_target)
        & (hip_ankle_distance >= abs(thigh_target - shank_target))
    )
    z_only_feasible = (
        finite_targets
        & np.isfinite(hip_knee_xy)
        & np.isfinite(knee_ankle_xy)
        & (hip_knee_xy <= thigh_target)
        & (knee_ankle_xy <= shank_target)
    )
    return {
        "thigh_length": thigh_length,
        "shank_length": shank_length,
        "hip_ankle_distance": hip_ankle_distance,
        "thigh_residual": np.abs(thigh_length - thigh_target),
        "shank_residual": np.abs(shank_length - shank_target),
        "hip_knee_recording_xy_distance": hip_knee_xy,
        "knee_ankle_recording_xy_distance": knee_ankle_xy,
        "triangle_feasible": triangle_feasible,
        "z_only_link_feasible": z_only_feasible,
    }


def audit_frame_values(values: np.ndarray, frame_mask: np.ndarray) -> np.ndarray:
    return np.asarray(values)[frame_mask]


def audit_support_metrics(family: str, frame_mask: np.ndarray, support_width_target: float) -> dict[str, np.ndarray]:
    left_ankle = point_array(corridor_df, "left_ankle", family)
    right_ankle = point_array(corridor_df, "right_ankle", family)
    support_width = np.linalg.norm((right_ankle - left_ankle)[:, [0, 2]], axis=1)
    ankle_height_gap = np.abs(right_ankle[:, 1] - left_ankle[:, 1])
    bilateral_height_gaps = {"ankle": ankle_height_gap}
    for suffix in ["heel", "foot_index"]:
        left_landmark = f"left_{suffix}"
        right_landmark = f"right_{suffix}"
        if rv_family_has_coordinates(corridor_df, family, [left_landmark, right_landmark]):
            left_xyz = point_array(corridor_df, left_landmark, family)
            right_xyz = point_array(corridor_df, right_landmark, family)
            bilateral_height_gaps[suffix] = np.abs(right_xyz[:, 1] - left_xyz[:, 1])
        else:
            bilateral_height_gaps[suffix] = np.full(len(corridor_df), np.nan)
    bilateral_gap_stack = np.vstack([values for values in bilateral_height_gaps.values()])
    max_bilateral_support_height_gap = np.nanmax(bilateral_gap_stack, axis=0)
    support_landmarks = ["left_ankle", "right_ankle", "left_heel", "right_heel", "left_foot_index", "right_foot_index"]
    y_values = []
    for landmark in support_landmarks:
        if rv_family_has_coordinates(corridor_df, family, [landmark]):
            y_values.append(point_array(corridor_df, landmark, family)[:, 1])
    if y_values:
        y_stack = np.vstack(y_values)
        support_height_range = np.nanmax(y_stack, axis=0) - np.nanmin(y_stack, axis=0)
    else:
        support_height_range = np.full(len(corridor_df), np.nan)
    norm_fam = BOUNDED_RV_SOURCE_FAMILY if "BOUNDED_RV_SOURCE_FAMILY" in globals() else RV_FIT_SOURCE_FAMILY
    key_landmarks = ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"]
    if family == norm_fam:
        recording_xy_delta = np.zeros(len(corridor_df))
        depth_delta = np.zeros(len(corridor_df))
    else:
        xy_deltas = []
        z_deltas = []
        for landmark in key_landmarks:
            if rv_family_has_coordinates(corridor_df, family, [landmark]) and rv_family_has_coordinates(corridor_df, norm_fam, [landmark]):
                source = point_array(corridor_df, landmark, norm_fam)
                target = point_array(corridor_df, landmark, family)
                xy_deltas.append(np.linalg.norm(target[:, :2] - source[:, :2], axis=1))
                z_deltas.append(np.abs(target[:, 2] - source[:, 2]))
        recording_xy_delta = np.nanmax(np.vstack(xy_deltas), axis=0) if xy_deltas else np.full(len(corridor_df), np.nan)
        depth_delta = np.nanmax(np.vstack(z_deltas), axis=0) if z_deltas else np.full(len(corridor_df), np.nan)
    return {
        "support_width": audit_frame_values(support_width, frame_mask),
        "support_width_residual": audit_frame_values(np.abs(support_width - support_width_target), frame_mask),
        "ankle_height_gap": audit_frame_values(ankle_height_gap, frame_mask),
        "heel_height_gap": audit_frame_values(bilateral_height_gaps["heel"], frame_mask),
        "foot_index_height_gap": audit_frame_values(bilateral_height_gaps["foot_index"], frame_mask),
        "max_bilateral_support_height_gap": audit_frame_values(max_bilateral_support_height_gap, frame_mask),
        "support_height_range": audit_frame_values(support_height_range, frame_mask),
        "max_recording_xy_delta_vs_norm": audit_frame_values(recording_xy_delta, frame_mask),
        "max_depth_delta_vs_norm": audit_frame_values(depth_delta, frame_mask),
    }


if RV_FIT_ENABLED and RV_FIT_CONFIG.get("closed_chain_support_context", {}).get("enabled", False):
    if AUDIT_SUPPORT_FAMILY not in {RV_FIT_SOURCE_FAMILY, RV_FIT_FAMILY, globals().get("BOUNDED_RV_FAMILY", "")}:
        corridor_df = rv_copy_coordinate_family(corridor_df, RV_FIT_SOURCE_FAMILY, AUDIT_SUPPORT_FAMILY)
        audit_fit_mask = globals().get("rv_fit_mask", compact_rep_mask.copy())
        rv_apply_closed_chain_support_context(
            corridor_df,
            RV_FIT_SOURCE_FAMILY,
            AUDIT_SUPPORT_FAMILY,
            RV_FIT_CONFIG.get("closed_chain_support_context", {}),
            audit_fit_mask,
            mask_ref,
        )

frame_mask = corridor_df["frame"].astype(int).between(AUDIT_FRAME_START, AUDIT_FRAME_END).to_numpy()
audit_frames = corridor_df.loc[frame_mask, "frame"].astype(int).to_numpy()

support_cfg = RV_FIT_CONFIG.get("support_residual", {})
support_pair = list(support_cfg.get("pair", ["left_ankle", "right_ankle"]))
support_trim_q = float(np.clip(float(support_cfg.get("trim_quantile", 0.10)), 0.0, 0.45))
support_width_series = np.linalg.norm(
    point_array(corridor_df, support_pair[1], RV_FIT_SOURCE_FAMILY)[:, [0, 2]]
    - point_array(corridor_df, support_pair[0], RV_FIT_SOURCE_FAMILY)[:, [0, 2]],
    axis=1,
)
support_width_target = audit_trimmed_median(support_width_series[mask_ref], support_trim_q) if np.any(mask_ref) else audit_trimmed_median(support_width_series[frame_mask], support_trim_q)

families = []
for label, family in [
    ("norm", RV_FIT_SOURCE_FAMILY),
    ("support_context_only", AUDIT_SUPPORT_FAMILY),
    ("rv_skeleton_fit", RV_FIT_FAMILY),
    ("rv_skeleton_fit_bounded_xy", globals().get("BOUNDED_RV_FAMILY", "")),
    ("endpoint_blend", globals().get("ENDPOINT_BLEND_FAMILY", "")),
    ("support_memory_final", globals().get("SUPPORT_MEMORY_FAMILY", "")),
]:
    if family and audit_family_available(family):
        families.append((label, family))

norm_bend_sign: dict[tuple[int, str], float] = {}
for chain in RV_FIT_CONFIG.get("chains", []):
    side = str(chain.get("id", "")).split("_", 1)[0]
    hip = point_array(corridor_df, str(chain["proximal"]), RV_FIT_SOURCE_FAMILY)
    knee = point_array(corridor_df, str(chain["joint"]), RV_FIT_SOURCE_FAMILY)
    ankle = point_array(corridor_df, str(chain["distal"]), RV_FIT_SOURCE_FAMILY)
    _, sign = audit_signed_bend_xz(hip, knee, ankle)
    for frame, value in zip(corridor_df["frame"].astype(int), sign):
        norm_bend_sign[(int(frame), side)] = float(value)

audit_rows = []
for label, family in families:
    support_values = audit_support_metrics(family, frame_mask, support_width_target)
    for chain in RV_FIT_CONFIG.get("chains", []):
        chain_id = str(chain.get("id", "chain"))
        side = chain_id.split("_", 1)[0]
        proximal = str(chain["proximal"])
        joint = str(chain["joint"])
        distal = str(chain["distal"])
        proximal_segment = str(chain["proximal_segment"])
        distal_segment = str(chain["distal_segment"])
        proximal_memory_id = str(chain.get("proximal_memory_id", rv_default_memory_id(chain_id, proximal_segment)))
        distal_memory_id = str(chain.get("distal_memory_id", rv_default_memory_id(chain_id, distal_segment)))
        thigh_target = rv_target_length(proximal_segment, proximal_memory_id)
        shank_target = rv_target_length(distal_segment, distal_memory_id)
        hip = point_array(corridor_df, proximal, family)
        knee = point_array(corridor_df, joint, family)
        ankle = point_array(corridor_df, distal, family)
        link = audit_link_metrics(hip, knee, ankle, thigh_target, shank_target)
        bend_distance, bend_sign = audit_signed_bend_xz(hip, knee, ankle)
        for row_index, frame in enumerate(audit_frames):
            ref_sign = norm_bend_sign.get((int(frame), side), np.nan)
            cur_sign = float(audit_frame_values(bend_sign, frame_mask)[row_index])
            ref_near_zero = (not np.isfinite(ref_sign)) or ref_sign == 0.0
            preserved = bool(ref_near_zero or cur_sign == 0.0 or cur_sign == ref_sign)
            flags = []
            if not preserved:
                flags.append("bend_flip_vs_norm")
            if not bool(audit_frame_values(link["triangle_feasible"], frame_mask)[row_index]):
                flags.append("triangle_infeasible")
            if not bool(audit_frame_values(link["z_only_link_feasible"], frame_mask)[row_index]):
                flags.append("z_only_link_infeasible")
            audit_rows.append({
                "frame": int(frame),
                "family_label": label,
                "family": family,
                "side": side,
                "thigh_target": thigh_target,
                "shank_target": shank_target,
                "thigh_length": audit_frame_values(link["thigh_length"], frame_mask)[row_index],
                "shank_length": audit_frame_values(link["shank_length"], frame_mask)[row_index],
                "thigh_residual": audit_frame_values(link["thigh_residual"], frame_mask)[row_index],
                "shank_residual": audit_frame_values(link["shank_residual"], frame_mask)[row_index],
                "hip_ankle_distance": audit_frame_values(link["hip_ankle_distance"], frame_mask)[row_index],
                "hip_knee_recording_xy_distance": audit_frame_values(link["hip_knee_recording_xy_distance"], frame_mask)[row_index],
                "knee_ankle_recording_xy_distance": audit_frame_values(link["knee_ankle_recording_xy_distance"], frame_mask)[row_index],
                "triangle_feasible": bool(audit_frame_values(link["triangle_feasible"], frame_mask)[row_index]),
                "z_only_link_feasible": bool(audit_frame_values(link["z_only_link_feasible"], frame_mask)[row_index]),
                "bend_signed_distance_xz": audit_frame_values(bend_distance, frame_mask)[row_index],
                "bend_sign_xz": cur_sign,
                "norm_bend_sign_xz": ref_sign,
                "bend_direction_preserved_vs_norm": preserved,
                "support_width_target": support_width_target,
                "support_width": support_values["support_width"][row_index],
                "support_width_residual": support_values["support_width_residual"][row_index],
                "ankle_height_gap": support_values["ankle_height_gap"][row_index],
                "heel_height_gap": support_values["heel_height_gap"][row_index],
                "foot_index_height_gap": support_values["foot_index_height_gap"][row_index],
                "max_bilateral_support_height_gap": support_values["max_bilateral_support_height_gap"][row_index],
                "support_height_range": support_values["support_height_range"][row_index],
                "max_recording_xy_delta_vs_norm": support_values["max_recording_xy_delta_vs_norm"][row_index],
                "max_depth_delta_vs_norm": support_values["max_depth_delta_vs_norm"][row_index],
                "review_flags": ",".join(flags),
            })

audit_df = pd.DataFrame(audit_rows)
metric_cols = [
    "thigh_length", "shank_length", "thigh_residual", "shank_residual", "hip_ankle_distance",
    "hip_knee_recording_xy_distance", "knee_ankle_recording_xy_distance", "bend_signed_distance_xz",
    "support_width", "support_width_residual", "ankle_height_gap", "heel_height_gap", "foot_index_height_gap",
    "max_bilateral_support_height_gap", "support_height_range",
    "max_recording_xy_delta_vs_norm", "max_depth_delta_vs_norm",
]
for col in metric_cols:
    if col in audit_df.columns:
        audit_df[col] = audit_df[col].astype(float).round(6)

focus_audit_df = audit_df[audit_df["frame"].isin(AUDIT_FOCUS_FRAMES)].copy()
flagged_audit_df = audit_df[audit_df["review_flags"].astype(str).ne("")].copy()
summary_audit_df = (
    audit_df.groupby(["family_label", "side"], dropna=False)
    .agg(
        median_thigh_residual=("thigh_residual", "median"),
        median_shank_residual=("shank_residual", "median"),
        max_support_width_residual=("support_width_residual", "max"),
        max_ankle_height_gap=("ankle_height_gap", "max"),
        max_bilateral_support_height_gap=("max_bilateral_support_height_gap", "max"),
        max_support_height_range=("support_height_range", "max"),
        bend_flip_frames=("bend_direction_preserved_vs_norm", lambda s: int((~s.astype(bool)).sum())),
        triangle_infeasible_frames=("triangle_feasible", lambda s: int((~s.astype(bool)).sum())),
        z_only_infeasible_frames=("z_only_link_feasible", lambda s: int((~s.astype(bool)).sum())),
    )
    .reset_index()
)

for col in ["median_thigh_residual", "median_shank_residual", "max_support_width_residual", "max_ankle_height_gap", "max_support_height_range"]:
    summary_audit_df[col] = summary_audit_df[col].astype(float).round(6)

display(Markdown("### Frame 115-126 chain feasibility audit summary"))
display(summary_audit_df)
display(Markdown("### Focus frames 119-123"))
display(focus_audit_df)
if not flagged_audit_df.empty:
    display(Markdown("### Review-flagged rows"))
    display(flagged_audit_df)

fig_audit = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("knee bend signed distance in recording x/model-depth z", "support-width residual", "ankle/support height gap"),
)
for (family_label, side), plot_df in audit_df.groupby(["family_label", "side"]):
    line_dash = "solid" if side == "left" else "dot"
    fig_audit.add_trace(
        go.Scatter(x=plot_df["frame"], y=plot_df["bend_signed_distance_xz"], mode="lines+markers", name=f"{family_label} {side} bend", line=dict(dash=line_dash)),
        row=1,
        col=1,
    )
for family_label, plot_df in audit_df.drop_duplicates(["frame", "family_label"]).groupby("family_label"):
    fig_audit.add_trace(
        go.Scatter(x=plot_df["frame"], y=plot_df["support_width_residual"], mode="lines+markers", name=f"{family_label} support width"),
        row=2,
        col=1,
    )
    fig_audit.add_trace(
        go.Scatter(x=plot_df["frame"], y=plot_df["ankle_height_gap"], mode="lines+markers", name=f"{family_label} ankle gap"),
        row=3,
        col=1,
    )
    fig_audit.add_trace(
        go.Scatter(x=plot_df["frame"], y=plot_df["support_height_range"], mode="lines", name=f"{family_label} support y range", opacity=0.45),
        row=3,
        col=1,
    )
fig_audit.update_layout(title="Frame 115-126 correction artifact audit", height=900, width=1100)
fig_audit.update_yaxes(title_text="torso-length ratio", row=1, col=1)
fig_audit.update_yaxes(title_text="torso-length ratio", row=2, col=1)
fig_audit.update_yaxes(title_text="torso-length ratio", row=3, col=1)
fig_audit.update_xaxes(title_text="source frame", row=3, col=1)
if review_show_visual("frame_window_audit"):
    fig_audit.show()


## 13 Multi-Plane Knee Bend Direction Audit

This companion audit checks knee bend-direction preservation in three projected planes: recording x/model-depth z, recording y/model-depth z, and recording x/recording y. The y/z plane is especially useful when the playback shows a sagittal-looking knee reversal.

In [ ]:

AUDIT_PLANES = {
    "xz_recording_depth": (0, 2),
    "yz_height_depth": (1, 2),
    "xy_recording_plane": (0, 1),
}
AUDIT_PLANE_EPS = globals().get("AUDIT_BEND_EPS", audit_soft_bend_epsilon())


def audit_signed_bend_plane(hip: np.ndarray, knee: np.ndarray, ankle: np.ndarray, axes: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    base = ankle[:, axes] - hip[:, axes]
    knee_vec = knee[:, axes] - hip[:, axes]
    base_len = np.linalg.norm(base, axis=1)
    signed_area = base[:, 0] * knee_vec[:, 1] - base[:, 1] * knee_vec[:, 0]
    signed_distance = np.full(len(base), np.nan)
    valid = base_len > 1e-9
    signed_distance[valid] = signed_area[valid] / base_len[valid]
    sign = np.sign(signed_distance)
    sign[np.abs(signed_distance) <= AUDIT_PLANE_EPS] = 0.0
    return signed_distance, sign


def audit_available_families_for_bend() -> list[tuple[str, str]]:
    out = []
    for label, family in [
        ("norm", RV_FIT_SOURCE_FAMILY),
        ("support_context_only", globals().get("AUDIT_SUPPORT_FAMILY", "rv_support_context_only")),
        ("rv_skeleton_fit", RV_FIT_FAMILY),
        ("rv_skeleton_fit_bounded_xy", globals().get("BOUNDED_RV_FAMILY", "")),
        ("endpoint_blend", globals().get("ENDPOINT_BLEND_FAMILY", "")),
        ("support_memory_final", globals().get("SUPPORT_MEMORY_FAMILY", "")),
    ]:
        if family and audit_family_available(family):
            out.append((label, family))
    return out


frame_mask = corridor_df["frame"].astype(int).between(AUDIT_FRAME_START, AUDIT_FRAME_END).to_numpy()
audit_frames = corridor_df.loc[frame_mask, "frame"].astype(int).to_numpy()

norm_plane_signs: dict[tuple[int, str, str], float] = {}
for chain in RV_FIT_CONFIG.get("chains", []):
    side = str(chain.get("id", "")).split("_", 1)[0]
    hip = point_array(corridor_df, str(chain["proximal"]), RV_FIT_SOURCE_FAMILY)
    knee = point_array(corridor_df, str(chain["joint"]), RV_FIT_SOURCE_FAMILY)
    ankle = point_array(corridor_df, str(chain["distal"]), RV_FIT_SOURCE_FAMILY)
    for plane_id, axes in AUDIT_PLANES.items():
        _, sign = audit_signed_bend_plane(hip, knee, ankle, axes)
        for frame, value in zip(corridor_df["frame"].astype(int), sign):
            norm_plane_signs[(int(frame), side, plane_id)] = float(value)

plane_rows = []
for family_label, family in audit_available_families_for_bend():
    for chain in RV_FIT_CONFIG.get("chains", []):
        chain_id = str(chain.get("id", "chain"))
        side = chain_id.split("_", 1)[0]
        hip = point_array(corridor_df, str(chain["proximal"]), family)
        knee = point_array(corridor_df, str(chain["joint"]), family)
        ankle = point_array(corridor_df, str(chain["distal"]), family)
        for plane_id, axes in AUDIT_PLANES.items():
            signed_distance, sign = audit_signed_bend_plane(hip, knee, ankle, axes)
            for row_index, frame in enumerate(audit_frames):
                ref_sign = norm_plane_signs.get((int(frame), side, plane_id), np.nan)
                current_sign = float(sign[frame_mask][row_index])
                ref_near_zero = (not np.isfinite(ref_sign)) or ref_sign == 0.0
                current_near_zero = current_sign == 0.0
                preserved = bool(ref_near_zero or current_near_zero or current_sign == ref_sign)
                plane_rows.append({
                    "frame": int(frame),
                    "family_label": family_label,
                    "family": family,
                    "side": side,
                    "plane": plane_id,
                    "bend_signed_distance": float(np.round(signed_distance[frame_mask][row_index], 6)),
                    "bend_sign": current_sign,
                    "norm_bend_sign": ref_sign,
                    "reference_near_zero": bool(ref_near_zero),
                    "bend_direction_preserved_vs_norm": preserved,
                    "review_flag": "bend_flip_vs_norm" if not preserved else "",
                })

bend_plane_audit_df = pd.DataFrame(plane_rows)
bend_plane_summary_df = (
    bend_plane_audit_df.groupby(["family_label", "side", "plane"], dropna=False)
    .agg(
        flip_frames=("bend_direction_preserved_vs_norm", lambda s: int((~s.astype(bool)).sum())),
        min_signed_distance=("bend_signed_distance", "min"),
        max_signed_distance=("bend_signed_distance", "max"),
    )
    .reset_index()
)
for col in ["min_signed_distance", "max_signed_distance"]:
    bend_plane_summary_df[col] = bend_plane_summary_df[col].astype(float).round(6)

focus_bend_plane_df = bend_plane_audit_df[bend_plane_audit_df["frame"].isin(AUDIT_FOCUS_FRAMES)].copy()
flagged_bend_plane_df = bend_plane_audit_df[bend_plane_audit_df["review_flag"].ne("")].copy()

display(Markdown("### Multi-plane knee bend-direction summary"))
display(bend_plane_summary_df)
display(Markdown("### Multi-plane focus frames 119-123"))
display(focus_bend_plane_df)
if not flagged_bend_plane_df.empty:
    display(Markdown("### Multi-plane bend-direction flagged rows"))
    display(flagged_bend_plane_df)

fig_bend_planes = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("xz recording/depth", "yz height/depth", "xy recording plane"),
)
plane_to_row = {"xz_recording_depth": 1, "yz_height_depth": 2, "xy_recording_plane": 3}
for (family_label, side, plane), plot_df in bend_plane_audit_df.groupby(["family_label", "side", "plane"]):
    line_dash = "solid" if side == "left" else "dot"
    fig_bend_planes.add_trace(
        go.Scatter(
            x=plot_df["frame"],
            y=plot_df["bend_signed_distance"],
            mode="lines+markers",
            name=f"{family_label} {side} {plane}",
            line=dict(dash=line_dash),
        ),
        row=plane_to_row[plane],
        col=1,
    )
fig_bend_planes.update_layout(title="Frame 115-126 multi-plane knee bend-direction audit", height=900, width=1100)
fig_bend_planes.update_yaxes(title_text="signed distance")
fig_bend_planes.update_xaxes(title_text="source frame", row=3, col=1)
if review_show_visual("multi_plane_bend_audit"):
    fig_bend_planes.show()

final_review_norm_family = str(globals().get("endpoint_play_norm_family", "norm"))
final_review_output_family = str(globals().get("FINAL_REVIEW_OUTPUT_FAMILY", globals().get("ENDPOINT_BLEND_FAMILY", "rv_skeleton_fit_bounded_xy_endpoint_blend")))
final_review_audit_label = str(globals().get("FINAL_REVIEW_AUDIT_LABEL", "endpoint_blend"))
final_review_rep_id = int(globals().get("endpoint_rep_id", BOUNDED_RV_REP_ID if "BOUNDED_RV_REP_ID" in globals() else COMPACT_REP_ID))
final_review_rep_rows = ann_df[(ann_df["segment_type"].eq("rep")) & (ann_df["rep_id"].astype("Int64").eq(final_review_rep_id))]
final_output_support_review_summary_df = pd.DataFrame()
final_output_support_review_sensitivity_df = pd.DataFrame()
final_output_support_review_frame_df = pd.DataFrame()


def final_review_frame_mask(df: pd.DataFrame, start_frame: int, end_frame: int) -> np.ndarray:
    return df["frame"].astype(int).between(int(start_frame), int(end_frame)).to_numpy()


def final_review_safe_percentile(values: np.ndarray, percentile: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.nanpercentile(arr, percentile)) if arr.size else np.nan


def final_review_angle_deg(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    ba = a - b
    bc = c - b
    denom = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1)
    cos_value = np.full(len(b), np.nan, dtype=float)
    valid = denom > 1e-9
    cos_value[valid] = np.sum(ba[valid] * bc[valid], axis=1) / denom[valid]
    return np.degrees(np.arccos(np.clip(cos_value, -1.0, 1.0)))


def final_review_shank_support_angle_deg(knee: np.ndarray, ankle: np.ndarray) -> np.ndarray:
    vec = knee - ankle
    horizontal = np.linalg.norm(vec[:, [0, 2]], axis=1)
    vertical = np.abs(vec[:, 1])
    return np.degrees(np.arctan2(vertical, horizontal))


def final_review_support_width(df: pd.DataFrame, family: str, suffix: str) -> np.ndarray:
    left = point_array(df, f"left_{suffix}", family)
    right = point_array(df, f"right_{suffix}", family)
    return np.linalg.norm((right - left)[:, [0, 2]], axis=1)


def final_review_family_frame_metrics(df: pd.DataFrame, family: str) -> pd.DataFrame:
    rows = []
    frames = df["frame"].astype(int).to_numpy()
    for suffix in ["ankle", "heel", "foot_index"]:
        if rv_family_has_coordinates(df, family, [f"left_{suffix}", f"right_{suffix}"]):
            values = final_review_support_width(df, family, suffix)
            rows.extend({"frame": int(frame), "family": family, "metric_id": f"{suffix}_support_width_xz", "side": "bilateral", "value": float(value)} for frame, value in zip(frames, values))
    for side in ["left", "right"]:
        hip_name = f"{side}_hip"
        knee_name = f"{side}_knee"
        ankle_name = f"{side}_ankle"
        if not rv_family_has_coordinates(df, family, [hip_name, knee_name, ankle_name]):
            continue
        hip = point_array(df, hip_name, family)
        knee = point_array(df, knee_name, family)
        ankle = point_array(df, ankle_name, family)
        knee_angle = final_review_angle_deg(hip, knee, ankle)
        flexion_proxy = 180.0 - knee_angle
        shank_support_angle = final_review_shank_support_angle_deg(knee, ankle)
        for metric_id, values in [
            ("knee_angle_deg", knee_angle),
            ("knee_flexion_proxy_deg", flexion_proxy),
            ("shank_support_elevation_deg", shank_support_angle),
        ]:
            rows.extend({"frame": int(frame), "family": family, "metric_id": metric_id, "side": side, "value": float(value)} for frame, value in zip(frames, values))
    return pd.DataFrame(rows)


def final_review_summary_rows(frame_df: pd.DataFrame, key_frames: dict[str, int]) -> pd.DataFrame:
    rows = []
    for (family, metric_id, side), group in frame_df.groupby(["family", "metric_id", "side"]):
        values = group["value"].to_numpy(float)
        row = {
            "family": family,
            "metric_id": metric_id,
            "side": side,
            "median": final_review_safe_percentile(values, 50),
            "p05": final_review_safe_percentile(values, 5),
            "p95": final_review_safe_percentile(values, 95),
            "range": float(np.nanmax(values) - np.nanmin(values)) if np.isfinite(values).any() else np.nan,
            "used_for_features_or_scores": False,
            "evaluation_domain": "corrected_3d_hypothesis_review",
        }
        for label, frame in key_frames.items():
            match = group[group["frame"].astype(int).eq(int(frame))]
            row[f"{label}_frame"] = int(frame)
            row[f"{label}_value"] = float(match.iloc[0]["value"]) if not match.empty else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def final_review_sensitivity_rows(frame_df: pd.DataFrame, norm_family: str, output_family: str, key_frames: dict[str, int]) -> pd.DataFrame:
    rows = []
    norm_df = frame_df[frame_df["family"].eq(norm_family)]
    output_df = frame_df[frame_df["family"].eq(output_family)]
    for metric_id in sorted(set(norm_df["metric_id"]).intersection(set(output_df["metric_id"]))):
        for side in sorted(set(norm_df.loc[norm_df["metric_id"].eq(metric_id), "side"]).intersection(set(output_df.loc[output_df["metric_id"].eq(metric_id), "side"]))):
            left = norm_df[(norm_df["metric_id"].eq(metric_id)) & (norm_df["side"].eq(side))][["frame", "value"]].rename(columns={"value": "norm_value"})
            right = output_df[(output_df["metric_id"].eq(metric_id)) & (output_df["side"].eq(side))][["frame", "value"]].rename(columns={"value": "final_value"})
            merged = left.merge(right, on="frame", how="inner")
            if merged.empty:
                continue
            delta = merged["final_value"].to_numpy(float) - merged["norm_value"].to_numpy(float)
            row = {
                "metric_id": metric_id,
                "side": side,
                "median_delta": final_review_safe_percentile(delta, 50),
                "median_abs_delta": final_review_safe_percentile(np.abs(delta), 50),
                "p95_abs_delta": final_review_safe_percentile(np.abs(delta), 95),
                "max_abs_delta": float(np.nanmax(np.abs(delta))) if np.isfinite(delta).any() else np.nan,
                "used_for_features_or_scores": False,
            }
            for label, frame in key_frames.items():
                match = merged[merged["frame"].astype(int).eq(int(frame))]
                row[f"{label}_delta"] = float(match.iloc[0]["final_value"] - match.iloc[0]["norm_value"]) if not match.empty else np.nan
            rows.append(row)
    return pd.DataFrame(rows)


if not all(rv_family_has_coordinates(corridor_df, family, ["left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"]) for family in [final_review_norm_family, final_review_output_family]):
    display(Markdown("Final output support review is unavailable because required norm/final coordinate columns are missing."))
else:
    final_review_start_frame = int(corridor_df["frame"].astype(int).min())
    final_review_end_frame = int(corridor_df["frame"].astype(int).max())
    final_review_mask = final_review_frame_mask(corridor_df, final_review_start_frame, final_review_end_frame)
    final_review_df = corridor_df.loc[final_review_mask].copy()
    final_review_frames = final_review_df["frame"].astype(int).to_numpy()
    final_family_df = final_review_family_frame_metrics(final_review_df, final_review_output_family)
    final_flexion = final_family_df[(final_family_df["metric_id"].eq("knee_flexion_proxy_deg")) & final_family_df["side"].isin(["left", "right"])]
    if final_flexion.empty:
        bottom_frame = int(final_review_frames[len(final_review_frames) // 2])
    else:
        mean_flexion = final_flexion.groupby("frame")["value"].mean()
        bottom_frame = int(mean_flexion.idxmax())
    key_frames = {"start": final_review_start_frame, "bottom_proxy": bottom_frame, "end": final_review_end_frame}
    final_output_support_review_frame_df = pd.concat([
        final_review_family_frame_metrics(final_review_df, final_review_norm_family),
        final_family_df,
    ], ignore_index=True)
    final_output_support_review_summary_df = final_review_summary_rows(final_output_support_review_frame_df, key_frames)
    final_output_support_review_sensitivity_df = final_review_sensitivity_rows(final_output_support_review_frame_df, final_review_norm_family, final_review_output_family, key_frames)

    display(Markdown("### Final Output Support-Referenced Review"))
    display(final_output_support_review_summary_df)
    display(Markdown("### Norm-vs-Final Sensitivity"))
    display(final_output_support_review_sensitivity_df)

    fig_final_support_review = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("support width in x-z support plane", "knee flexion proxy", "ankle-to-knee shank elevation relative to support plane"),
        vertical_spacing=0.10,
    )
    family_styles = [
        (final_review_norm_family, "norm", "rgba(30,30,30,0.78)", "dot"),
        (final_review_output_family, "final output", "rgba(0,170,170,0.95)", "solid"),
    ]
    for family, label, color, dash in family_styles:
        plot_df = final_output_support_review_frame_df[final_output_support_review_frame_df["family"].eq(family)]
        for metric_id, short_label in [("ankle_support_width_xz", "ankle"), ("heel_support_width_xz", "heel"), ("foot_index_support_width_xz", "foot index")]:
            metric_df = plot_df[(plot_df["metric_id"].eq(metric_id)) & (plot_df["side"].eq("bilateral"))]
            if not metric_df.empty:
                fig_final_support_review.add_trace(go.Scatter(x=metric_df["frame"], y=metric_df["value"], mode="lines", name=f"{label} {short_label} width", line=dict(color=color, dash=dash, width=3 if short_label == "ankle" else 2)), row=1, col=1)
        for side, side_dash in [("left", dash), ("right", "dash" if dash == "solid" else "dashdot")]:
            flex_df = plot_df[(plot_df["metric_id"].eq("knee_flexion_proxy_deg")) & (plot_df["side"].eq(side))]
            shank_df = plot_df[(plot_df["metric_id"].eq("shank_support_elevation_deg")) & (plot_df["side"].eq(side))]
            if not flex_df.empty:
                fig_final_support_review.add_trace(go.Scatter(x=flex_df["frame"], y=flex_df["value"], mode="lines", name=f"{label} {side} knee flexion", line=dict(color=color, dash=side_dash, width=3)), row=2, col=1)
            if not shank_df.empty:
                fig_final_support_review.add_trace(go.Scatter(x=shank_df["frame"], y=shank_df["value"], mode="lines", name=f"{label} {side} shank support angle", line=dict(color=color, dash=side_dash, width=3)), row=3, col=1)
    for marker_frame in key_frames.values():
        fig_final_support_review.add_vline(x=int(marker_frame), line=dict(color="rgba(90,90,90,0.45)", dash="dot"))
    fig_final_support_review.update_layout(title="Whole video: support-referenced final output review", height=900, width=1120, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0))
    fig_final_support_review.update_yaxes(title_text="torso-length ratio", row=1, col=1)
    fig_final_support_review.update_yaxes(title_text="degrees", row=2, col=1)
    fig_final_support_review.update_yaxes(title_text="degrees", row=3, col=1)
    fig_final_support_review.update_xaxes(title_text="source frame", row=3, col=1)
    if review_show_visual("final_output_support_review") or review_show_visual("final_playback"):
        fig_final_support_review.show()

SCORING_READINESS_CONFIG = CORRIDOR_CONFIG.get("score_readiness_review", {})
scoring_readiness_review_df = pd.DataFrame()
scoring_readiness_summary_df = pd.DataFrame()


def scoring_readiness_visibility(rep_df: pd.DataFrame, landmarks: list[str]) -> tuple[float, float]:
    values = []
    for landmark in landmarks:
        col = f"{landmark}_visibility"
        if col in rep_df.columns:
            arr = rep_df[col].astype(float).to_numpy()
            values.extend(arr[np.isfinite(arr)].tolist())
    if not values:
        return np.nan, np.nan
    arr = np.asarray(values, dtype=float)
    return float(np.nanpercentile(arr, 10)), float(np.nanmedian(arr))


def scoring_readiness_metric_delta(metric_df: pd.DataFrame, frames: np.ndarray, metric_id: str, side: str, reference_family: str, candidate_family: str) -> tuple[float, float, float]:
    subset = metric_df[(metric_df["frame"].astype(int).isin(frames)) & metric_df["metric_id"].eq(metric_id) & metric_df["side"].eq(side)]
    reference = subset[subset["family"].eq(reference_family)][["frame", "value"]].rename(columns={"value": "reference_value"})
    candidate = subset[subset["family"].eq(candidate_family)][["frame", "value"]].rename(columns={"value": "candidate_value"})
    merged = pd.merge(reference, candidate, on="frame", how="inner")
    if merged.empty:
        return np.nan, np.nan, np.nan
    delta = (merged["candidate_value"].astype(float) - merged["reference_value"].astype(float)).abs().to_numpy()
    delta = delta[np.isfinite(delta)]
    if delta.size == 0:
        return np.nan, np.nan, np.nan
    return float(np.nanmedian(delta)), float(np.nanpercentile(delta, 95)), float(np.nanmax(delta))


def scoring_segment_cv(rep_df: pd.DataFrame, family: str, proximal: str, distal: str) -> float:
    if not rv_family_has_coordinates(rep_df, family, [proximal, distal]):
        return np.nan
    length = np.linalg.norm(point_array(rep_df, distal, family) - point_array(rep_df, proximal, family), axis=1)
    length = length[np.isfinite(length)]
    mean_length = float(np.nanmean(length)) if length.size else np.nan
    if not np.isfinite(mean_length) or abs(mean_length) < 1e-9:
        return np.nan
    return float(np.nanstd(length) / mean_length)


def scoring_joint_step_p95(rep_df: pd.DataFrame, family: str, landmarks: list[str]) -> float:
    steps = []
    for landmark in landmarks:
        if not rv_family_has_coordinates(rep_df, family, [landmark]):
            continue
        xyz = point_array(rep_df, landmark, family)
        if len(xyz) > 1:
            step = np.linalg.norm(np.diff(xyz, axis=0), axis=1)
            steps.extend(step[np.isfinite(step)].tolist())
    return float(np.nanpercentile(np.asarray(steps, dtype=float), 95)) if steps else np.nan


def scoring_signed_bend_plane(hip: np.ndarray, knee: np.ndarray, ankle: np.ndarray, axes: tuple[int, int], epsilon: float) -> tuple[np.ndarray, np.ndarray]:
    base = ankle[:, axes] - hip[:, axes]
    knee_vec = knee[:, axes] - hip[:, axes]
    base_len = np.linalg.norm(base, axis=1)
    signed_area = base[:, 0] * knee_vec[:, 1] - base[:, 1] * knee_vec[:, 0]
    signed_distance = np.full(len(base), np.nan)
    valid = base_len > 1e-9
    signed_distance[valid] = signed_area[valid] / base_len[valid]
    sign = np.sign(signed_distance)
    sign[np.abs(signed_distance) <= float(epsilon)] = 0.0
    return signed_distance, sign


def scoring_bend_flip_frames(rep_df: pd.DataFrame, side: str, reference_family: str, candidate_family: str) -> int:
    chain_map = {str(chain.get("id", "")).split("_", 1)[0]: chain for chain in RV_FIT_CONFIG.get("chains", [])}
    chain = chain_map.get(side)
    if chain is None:
        return 0
    hip_name = str(chain["proximal"])
    knee_name = str(chain["joint"])
    ankle_name = str(chain["distal"])
    required = [hip_name, knee_name, ankle_name]
    if not rv_family_has_coordinates(rep_df, reference_family, required) or not rv_family_has_coordinates(rep_df, candidate_family, required):
        return 0
    ref_hip = point_array(rep_df, hip_name, reference_family)
    ref_knee = point_array(rep_df, knee_name, reference_family)
    ref_ankle = point_array(rep_df, ankle_name, reference_family)
    cand_hip = point_array(rep_df, hip_name, candidate_family)
    cand_knee = point_array(rep_df, knee_name, candidate_family)
    cand_ankle = point_array(rep_df, ankle_name, candidate_family)
    flip_mask = np.zeros(len(rep_df), dtype=bool)
    epsilon = float(SCORING_READINESS_CONFIG.get("bend_epsilon", AUDIT_PLANE_EPS))
    for axes in AUDIT_PLANES.values():
        _, ref_sign = scoring_signed_bend_plane(ref_hip, ref_knee, ref_ankle, axes, epsilon)
        _, cand_sign = scoring_signed_bend_plane(cand_hip, cand_knee, cand_ankle, axes, epsilon)
        flip_mask |= np.isfinite(ref_sign) & np.isfinite(cand_sign) & (ref_sign != 0.0) & (cand_sign != 0.0) & (cand_sign != ref_sign)
    return int(flip_mask.sum())


BEND_FLIP_PROVENANCE_CONFIG = CORRIDOR_CONFIG.get("bend_flip_provenance_review", {})
bend_flip_provenance_df = pd.DataFrame()
bend_flip_first_family_df = pd.DataFrame()


def bend_provenance_resolve_family(family_name: str) -> str:
    family = str(family_name)
    if family == "final_review_output":
        return final_review_output_family
    return family


def bend_provenance_frame_preview(frames: list[int], limit: int = 12) -> str:
    if len(frames) <= limit:
        return ",".join(str(frame) for frame in frames)
    head = ",".join(str(frame) for frame in frames[: limit // 2])
    tail = ",".join(str(frame) for frame in frames[-(limit // 2) :])
    return f"{head},...,{tail}"


if bool(BEND_FLIP_PROVENANCE_CONFIG.get("enabled", False)):
    provenance_rep_start = int(BEND_FLIP_PROVENANCE_CONFIG.get("rep_id_start", 1))
    provenance_rep_end = int(BEND_FLIP_PROVENANCE_CONFIG.get("rep_id_end", provenance_rep_start))
    provenance_reference_family = bend_provenance_resolve_family(BEND_FLIP_PROVENANCE_CONFIG.get("reference_family", final_review_norm_family))
    provenance_plane_ids = [str(item) for item in BEND_FLIP_PROVENANCE_CONFIG.get("planes", list(AUDIT_PLANES.keys()))]
    provenance_epsilon = float(BEND_FLIP_PROVENANCE_CONFIG.get("epsilon", AUDIT_PLANE_EPS))
    provenance_sequence = BEND_FLIP_PROVENANCE_CONFIG.get("family_sequence", [])
    if not provenance_sequence:
        provenance_sequence = [
            {"label": "norm", "family": provenance_reference_family},
            {"label": "rv_skeleton_fit", "family": RV_FIT_FAMILY},
            {"label": "rv_skeleton_fit_bounded_xy", "family": globals().get("BOUNDED_RV_FAMILY", "")},
            {"label": "endpoint_blend", "family": globals().get("ENDPOINT_BLEND_FAMILY", "")},
            {"label": "support_memory_final", "family": final_review_output_family},
        ]
    rep_ids = ann_df["rep_id"].astype("Int64")
    provenance_rep_rows = ann_df[ann_df["segment_type"].eq("rep") & rep_ids.ge(provenance_rep_start).fillna(False) & rep_ids.le(provenance_rep_end).fillna(False)].copy()
    provenance_rows = []
    for _, rep_row in provenance_rep_rows.iterrows():
        rep_id = int(rep_row["rep_id"])
        rep_start = int(rep_row["start_frame"])
        rep_end = int(rep_row["end_frame"])
        rep_mask = final_review_frame_mask(corridor_df, rep_start, rep_end)
        rep_df = corridor_df.loc[rep_mask].copy()
        rep_frames = rep_df["frame"].astype(int).to_numpy()
        for chain in RV_FIT_CONFIG.get("chains", []):
            side = str(chain.get("id", "")).split("_", 1)[0]
            hip_name = str(chain["proximal"])
            knee_name = str(chain["joint"])
            ankle_name = str(chain["distal"])
            required = [hip_name, knee_name, ankle_name]
            if not rv_family_has_coordinates(rep_df, provenance_reference_family, required):
                continue
            ref_hip = point_array(rep_df, hip_name, provenance_reference_family)
            ref_knee = point_array(rep_df, knee_name, provenance_reference_family)
            ref_ankle = point_array(rep_df, ankle_name, provenance_reference_family)
            for plane_id in provenance_plane_ids:
                axes = AUDIT_PLANES.get(plane_id, (0, 2))
                ref_signed, ref_sign = scoring_signed_bend_plane(ref_hip, ref_knee, ref_ankle, axes, provenance_epsilon)
                for order_index, item in enumerate(provenance_sequence):
                    label = str(item.get("label", item.get("family", f"family_{order_index}")))
                    family = bend_provenance_resolve_family(item.get("family", ""))
                    if not family or not rv_family_has_coordinates(rep_df, family, required):
                        provenance_rows.append({
                            "rep_id": rep_id,
                            "side": side,
                            "plane": plane_id,
                            "family_order": order_index,
                            "family_label": label,
                            "family": family,
                            "status": "missing_family_coordinates",
                            "flip_frames": np.nan,
                            "first_flip_frame": np.nan,
                            "flip_frame_preview": "",
                            "median_abs_signed_distance_on_flip": np.nan,
                            "max_abs_signed_distance_on_flip": np.nan,
                        })
                        continue
                    hip = point_array(rep_df, hip_name, family)
                    knee = point_array(rep_df, knee_name, family)
                    ankle = point_array(rep_df, ankle_name, family)
                    signed, sign = scoring_signed_bend_plane(hip, knee, ankle, axes, provenance_epsilon)
                    flip_mask = np.isfinite(ref_sign) & np.isfinite(sign) & (ref_sign != 0.0) & (sign != 0.0) & (sign != ref_sign)
                    flip_frames = rep_frames[flip_mask].astype(int).tolist()
                    abs_signed = np.abs(signed[flip_mask]) if np.any(flip_mask) else np.asarray([], dtype=float)
                    provenance_rows.append({
                        "rep_id": rep_id,
                        "side": side,
                        "plane": plane_id,
                        "family_order": order_index,
                        "family_label": label,
                        "family": family,
                        "status": "ok",
                        "flip_frames": int(np.sum(flip_mask)),
                        "first_flip_frame": int(flip_frames[0]) if flip_frames else np.nan,
                        "flip_frame_preview": bend_provenance_frame_preview(flip_frames),
                        "median_abs_signed_distance_on_flip": float(np.nanmedian(abs_signed)) if abs_signed.size else np.nan,
                        "max_abs_signed_distance_on_flip": float(np.nanmax(abs_signed)) if abs_signed.size else np.nan,
                    })
    bend_flip_provenance_df = pd.DataFrame(provenance_rows)
    for col in bend_flip_provenance_df.select_dtypes(include=["float"]).columns:
        bend_flip_provenance_df[col] = bend_flip_provenance_df[col].astype(float).round(6)
    first_rows = []
    if not bend_flip_provenance_df.empty:
        ok_rows = bend_flip_provenance_df[bend_flip_provenance_df["status"].eq("ok") & (bend_flip_provenance_df["flip_frames"].fillna(0).astype(float) > 0)].copy()
        for keys, group in ok_rows.groupby(["rep_id", "side", "plane"], dropna=False):
            first = group.sort_values(["family_order", "first_flip_frame"]).iloc[0]
            first_rows.append({
                "rep_id": int(keys[0]),
                "side": str(keys[1]),
                "plane": str(keys[2]),
                "first_flip_family_label": first["family_label"],
                "first_flip_family": first["family"],
                "first_flip_frame": first["first_flip_frame"],
                "flip_frames_in_first_family": first["flip_frames"],
                "median_abs_signed_distance_on_flip": first["median_abs_signed_distance_on_flip"],
                "max_abs_signed_distance_on_flip": first["max_abs_signed_distance_on_flip"],
            })
    bend_flip_first_family_df = pd.DataFrame(first_rows)
    display(Markdown("### Bend-Flip Provenance Review (Report-Only)"))
    if bend_flip_first_family_df.empty:
        display(Markdown("No bend-side flips were found across the configured family sequence and annotated rep windows."))
    else:
        display(bend_flip_first_family_df)
    flip_detail_df = bend_flip_provenance_df[bend_flip_provenance_df["flip_frames"].fillna(0).astype(float) > 0].copy() if not bend_flip_provenance_df.empty else pd.DataFrame()
    if not flip_detail_df.empty:
        display(flip_detail_df)


if bool(SCORING_READINESS_CONFIG.get("enabled", False)) and not final_output_support_review_frame_df.empty:
    readiness_thresholds = SCORING_READINESS_CONFIG.get("thresholds", {})
    readiness_rep_start = int(SCORING_READINESS_CONFIG.get("rep_id_start", 1))
    readiness_rep_end = int(SCORING_READINESS_CONFIG.get("rep_id_end", readiness_rep_start))
    readiness_reference_family = str(SCORING_READINESS_CONFIG.get("reference_family", final_review_norm_family))
    readiness_candidate_family = final_review_output_family if str(SCORING_READINESS_CONFIG.get("candidate_family", "final_review_output")) == "final_review_output" else str(SCORING_READINESS_CONFIG.get("candidate_family"))
    critical_by_side = SCORING_READINESS_CONFIG.get("critical_landmarks_by_side", {"left": ["left_hip", "left_knee", "left_ankle"], "right": ["right_hip", "right_knee", "right_ankle"]})
    rep_ids = ann_df["rep_id"].astype("Int64")
    readiness_rep_rows = ann_df[ann_df["segment_type"].eq("rep") & rep_ids.ge(readiness_rep_start).fillna(False) & rep_ids.le(readiness_rep_end).fillna(False)].copy()
    readiness_rows = []
    for _, rep_row in readiness_rep_rows.iterrows():
        rep_id = int(rep_row["rep_id"])
        rep_start = int(rep_row["start_frame"])
        rep_end = int(rep_row["end_frame"])
        rep_mask = final_review_frame_mask(corridor_df, rep_start, rep_end)
        rep_df = corridor_df.loc[rep_mask].copy()
        rep_frames = rep_df["frame"].astype(int).to_numpy()
        for side in ["left", "right"]:
            landmarks = [str(item) for item in critical_by_side.get(side, [f"{side}_hip", f"{side}_knee", f"{side}_ankle"])]
            visibility_p10, visibility_median = scoring_readiness_visibility(rep_df, landmarks)
            knee_median, knee_p95, knee_max = scoring_readiness_metric_delta(final_output_support_review_frame_df, rep_frames, "knee_flexion_proxy_deg", side, readiness_reference_family, readiness_candidate_family)
            shank_median, shank_p95, shank_max = scoring_readiness_metric_delta(final_output_support_review_frame_df, rep_frames, "shank_support_elevation_deg", side, readiness_reference_family, readiness_candidate_family)
            thigh_cv = scoring_segment_cv(rep_df, readiness_candidate_family, f"{side}_hip", f"{side}_knee")
            shank_cv = scoring_segment_cv(rep_df, readiness_candidate_family, f"{side}_knee", f"{side}_ankle")
            joint_step_p95 = scoring_joint_step_p95(rep_df, readiness_candidate_family, landmarks)
            bend_flip_frames = scoring_bend_flip_frames(rep_df, side, readiness_reference_family, readiness_candidate_family)
            flags = []
            if np.isfinite(visibility_p10) and visibility_p10 < float(readiness_thresholds.get("min_visibility_p10", 0.50)):
                flags.append("low_visibility_p10")
            if np.isfinite(visibility_median) and visibility_median < float(readiness_thresholds.get("min_visibility_median", 0.65)):
                flags.append("low_visibility_median")
            if bend_flip_frames > int(readiness_thresholds.get("max_bend_flip_frames", 0)):
                flags.append("bend_flip")
            if np.isfinite(knee_median) and knee_median > float(readiness_thresholds.get("max_knee_flexion_median_abs_delta_deg", 3.0)):
                flags.append("knee_median_sensitivity")
            if np.isfinite(knee_p95) and knee_p95 > float(readiness_thresholds.get("max_knee_flexion_p95_abs_delta_deg", 8.0)):
                flags.append("knee_p95_sensitivity")
            if np.isfinite(shank_median) and shank_median > float(readiness_thresholds.get("max_shank_angle_median_abs_delta_deg", 3.0)):
                flags.append("shank_median_sensitivity")
            if np.isfinite(shank_p95) and shank_p95 > float(readiness_thresholds.get("max_shank_angle_p95_abs_delta_deg", 8.0)):
                flags.append("shank_p95_sensitivity")
            max_segment_cv = np.nanmax([thigh_cv, shank_cv]) if np.isfinite([thigh_cv, shank_cv]).any() else np.nan
            if np.isfinite(max_segment_cv) and max_segment_cv > float(readiness_thresholds.get("max_segment_length_cv", 0.08)):
                flags.append("segment_length_cv")
            if np.isfinite(joint_step_p95) and joint_step_p95 > float(readiness_thresholds.get("max_joint_step_p95_ratio", 0.12)):
                flags.append("joint_step_p95")
            hard_not_ready = ("bend_flip" in flags) or (np.isfinite(visibility_p10) and visibility_p10 < 0.30)
            status = "not_ready" if hard_not_ready else ("caution" if flags else "review_ready")
            readiness_rows.append({
                "rep_id": rep_id,
                "side": side,
                "start_frame": rep_start,
                "end_frame": rep_end,
                "visibility_p10": visibility_p10,
                "visibility_median": visibility_median,
                "bend_flip_frames": bend_flip_frames,
                "knee_flexion_median_abs_delta_deg": knee_median,
                "knee_flexion_p95_abs_delta_deg": knee_p95,
                "knee_flexion_max_abs_delta_deg": knee_max,
                "shank_angle_median_abs_delta_deg": shank_median,
                "shank_angle_p95_abs_delta_deg": shank_p95,
                "shank_angle_max_abs_delta_deg": shank_max,
                "thigh_length_cv": thigh_cv,
                "shank_length_cv": shank_cv,
                "joint_step_p95_ratio": joint_step_p95,
                "status": status,
                "flags": ",".join(flags),
                "used_for_features_or_scores": bool(SCORING_READINESS_CONFIG.get("reporting", {}).get("used_for_features_or_scores", False)),
            })
    scoring_readiness_review_df = pd.DataFrame(readiness_rows)
    for col in scoring_readiness_review_df.select_dtypes(include=["float"]).columns:
        scoring_readiness_review_df[col] = scoring_readiness_review_df[col].astype(float).round(6)
    scoring_readiness_summary_df = scoring_readiness_review_df.groupby("status", dropna=False).size().reset_index(name="rep_side_count") if not scoring_readiness_review_df.empty else pd.DataFrame()
    scoring_readiness_display_columns = [
        "rep_id",
        "side",
        "start_frame",
        "end_frame",
        "visibility_p10",
        "visibility_median",
        "bend_flip_frames",
        "knee_flexion_median_abs_delta_deg",
        "knee_flexion_p95_abs_delta_deg",
        "shank_angle_median_abs_delta_deg",
        "shank_angle_p95_abs_delta_deg",
        "thigh_length_cv",
        "shank_length_cv",
        "joint_step_p95_ratio",
        "status",
        "flags",
        "used_for_features_or_scores",
    ]
    display(Markdown("### Rep 1-10 Scoring-Readiness Stability Review (Report-Only)"))
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 240, "display.width", 220):
        display(scoring_readiness_summary_df)
        display(scoring_readiness_review_df[[col for col in scoring_readiness_display_columns if col in scoring_readiness_review_df.columns]])

final_playback_ready = "fig_endpoint_blend_play" in globals()
final_playback_checks = []
if "summary_audit_df" in globals() and not summary_audit_df.empty:
    endpoint_chain_rows = summary_audit_df[summary_audit_df["family_label"].astype(str).eq(final_review_audit_label)]
    chain_ok = not endpoint_chain_rows.empty
    for col in ["bend_flip_frames", "triangle_infeasible_frames", "z_only_infeasible_frames"]:
        if col in endpoint_chain_rows.columns:
            chain_ok = chain_ok and int(endpoint_chain_rows[col].fillna(0).sum()) == 0
    final_playback_checks.append(("frame_window_chain_audit", chain_ok))
if "bend_plane_summary_df" in globals() and not bend_plane_summary_df.empty:
    endpoint_plane_rows = bend_plane_summary_df[bend_plane_summary_df["family_label"].astype(str).eq(final_review_audit_label)]
    plane_ok = (not endpoint_plane_rows.empty) and int(endpoint_plane_rows["flip_frames"].fillna(0).sum()) == 0
    final_playback_checks.append(("multi_plane_bend_audit", plane_ok))
final_playback_passed = final_playback_ready and bool(final_playback_checks) and all(ok for _, ok in final_playback_checks)
final_playback_status_df = pd.DataFrame(final_playback_checks, columns=["gate", "passed"])
if final_playback_ready and review_show_visual("final_playback"):
    display(Markdown("### Final Candidate Playback Gate"))
    display(final_playback_status_df)
    if final_playback_passed:
        fig_endpoint_blend_play.show()
    else:
        display(Markdown("Final 3D Play/Pause view is withheld because at least one audit gate did not pass."))
